## Problem 1: Double Pendulum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from IPython.display import display

# The system_dynamics function is now a wrapper that calls hamilton_equations
# to ensure a single, consistent definition of the system's dynamics.
def system_dynamics(t, state, m1, m2, l1, l2, g):
    return hamilton_equations(t, state, m1, m2, l1, l2, g)

def interactive_poincare_map(m1, m2, l1, l2, g, phi1_0_deg, phi2_0_deg, p1_0, p2_0):
    # Convert initial angles from degrees to radians
    phi1_0 = np.deg2rad(phi1_0_deg)
    phi2_0 = np.deg2rad(phi2_0_deg)

    # --- Simulación ---
    t_span = (0, 200)
    t_eval = np.linspace(0, 200, 20000)
    y0 = [phi1_0, phi2_0, p1_0, p2_0] # Condiciones iniciales

    sol = solve_ivp(hamilton_equations, t_span, y0, method='DOP853', t_eval=t_eval, atol=1e-12, rtol=1e-12, args=(m1, m2, l1, l2, g))

    # --- Gráfica Mapa de Poincaré ---
    # Sección de cruce por phi1 = 0
    phi1_sol = sol.y[0]
    phi2_sol = sol.y[1]
    p2_sol = sol.y[3]

    p_phi2, p_p2 = [], []
    for i in range(len(phi1_sol)-1):
        # Check for positive crossing of phi1=0 (or a small epsilon region around it)
        # Using a small tolerance to account for numerical precision
        if phi1_sol[i] < 0 and phi1_sol[i+1] > 0:
            # Interpolate to find the exact crossing point (linear interpolation)
            # This improves the accuracy of the Poincaré map points
            t_interp = (0 - phi1_sol[i]) / (phi1_sol[i+1] - phi1_sol[i])

            phi2_interp = phi2_sol[i] + t_interp * (phi2_sol[i+1] - phi2_sol[i])
            p2_interp = p2_sol[i] + t_interp * (p2_sol[i+1] - p2_sol[i])

            p_phi2.append(phi2_interp)
            p_p2.append(p2_interp)

    plt.figure(figsize=(8,6))
    plt.scatter(p_phi2, p_p2, s=1, color='blue')
    plt.title(r"Poincaré Map ($\phi_1=0$)")
    plt.xlabel(r"$\phi_2$")
    plt.ylabel(r"$p_{\phi_2}$ ")
    plt.grid(True)
    plt.show()

# --- Sliders interactivos ---
widgets.interact(
    interactive_poincare_map,
    m1=widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='m1'),
    m2=widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='m2'),
    l1=widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='l1'),
    l2=widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='l2'),
    g=widgets.FloatSlider(min=0.0, max=20.0, step=0.1, value=9.81, description='g'),
    phi1_0_deg=widgets.FloatSlider(min=-360.0, max=360.0, step=5.0, value=np.rad2deg(2.0), description='phi1_0 (deg)'),
    phi2_0_deg=widgets.FloatSlider(min=-360.0, max=360.0, step=5.0, value=np.rad2deg(0.0), description='phi2_0 (deg)'),
    p1_0=widgets.FloatSlider(min=-5.0, max=5.0, step=0.1, value=0.0, description='p1_0'),
    p2_0=widgets.FloatSlider(min=-5.0, max=5.0, step=0.1, value=0.0, description='p2_0')
);

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp

def hamilton_equations(t, state, m1, m2, l1, l2, g):
    # This function implements the Hamiltonian equations of motion
    # It is consistent with the system_dynamics function used elsewhere in the notebook
    # and the Hamiltonian presented in section 4.1.
    phi1, phi2, p1, p2 = state
    delta = phi1 - phi2
    sin_d = np.sin(delta)
    cos_d = np.cos(delta)

    # Denominator D from user's optimized code (previously den_base)
    D = (m1 + m2 * sin_d**2)
    if np.abs(D) < 1e-12: # Check for near-zero denominator
        return np.array([np.nan, np.nan, np.nan, np.nan]) # Indicate integration failure

    # Angular velocities (dphi1/dt, dphi2/dt) - These expressions were correct
    dphi1_den = (l1**2 * l2 * D)
    if np.abs(dphi1_den) < 1e-12:
        return np.array([np.nan, np.nan, np.nan, np.nan])
    dphi1 = (l2 * p1 - l1 * p2 * cos_d) / dphi1_den

    dphi2_den = (l1 * l2**2 * m2 * D)
    if np.abs(dphi2_den) < 1e-12:
        return np.array([np.nan, np.nan, np.nan, np.nan])
    dphi2 = (l1 * (m1 + m2) * p2 - l2 * m2 * p1 * cos_d) / dphi2_den

    # --- Corrected Derivatives of Momenta (dp1/dt, dp2/dt) ---
    # These terms are derived from -dK/d(phi1) and -dK/d(phi2) based on the Hamiltonian H=K+V
    # where K is the kinetic energy term involving momenta and angles.
    # dp1 = -dK/d(phi1) - dV/d(phi1) and dp2 = -dK/d(phi2) - dV/d(phi2)
    # Since K depends on phi1 and phi2 only through delta = phi1 - phi2:
    # -dK/d(phi1) = -dK/d(delta) and -dK/d(phi2) = +dK/d(delta)

    # Numerator of the kinetic energy term in the Hamiltonian
    K_num = (m2 * l2**2 * p1**2 + (m1 + m2) * l1**2 * p2**2 - 2 * m2 * l1 * l2 * p1 * p2 * cos_d)

    # Derivative of K_num with respect to delta
    dK_num_d_delta = 2 * m2 * l1 * l2 * p1 * p2 * sin_d

    # Derivative of D (denominator part of K) with respect to delta
    dD_d_delta = 2 * m2 * sin_d * cos_d

    # Calculate the term d(K_num/D)/d(delta)
    if np.abs(D**2) < 1e-12:
         return np.array([np.nan, np.nan, np.nan, np.nan])
    term_K_num_div_D_derivative = (D * dK_num_d_delta - K_num * dD_d_delta) / (D**2)

    # Now compute dK/d(delta)
    # K = 1 / (2 * l1**2 * l2**2 * m2) * (K_num / D)
    # So, dK/d(delta) = (1 / (2 * l1**2 * l2**2 * m2)) * d(K_num/D)/d(delta)
    K_den_factor = (2 * l1**2 * l2**2 * m2)
    if np.abs(K_den_factor) < 1e-12:
         return np.array([np.nan, np.nan, np.nan, np.nan])
    kinetic_derivative_d_delta = (1 / K_den_factor) * term_K_num_div_D_derivative

    # Final expressions for dp1 and dp2
    dp1 = -kinetic_derivative_d_delta - (m1 + m2) * l1 * g * np.sin(phi1)
    dp2 = +kinetic_derivative_d_delta - m2 * l2 * g * np.sin(phi2)

    # Check for intermediate overflow/NaN before returning
    if np.any(np.isnan([dphi1, dphi2, dp1, dp2])) or np.any(np.isinf([dphi1, dphi2, dp1, dp2])):
        return np.array([np.nan, np.nan, np.nan, np.nan])

    return np.array([dphi1, dphi2, dp1, dp2])

In [ ]:
import matplotlib.pyplot as plt

def plot_lyapunov_results(times, separation, lambdas):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Gráfica A: Divergencia de trayectorias
    ax1.plot(times, np.log(separation), color='tab:red', label=r'$\ln(||\delta x||)$')
    ax1.set_title('Sensitive Dependence on Initial Conditions')
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Log-Separation')
    ax1.grid(True, linestyle='--')
    ax1.legend()

    # Gráfica B: Convergencia del exponente
    ax2.plot(times, lambdas, color='tab:blue', label='Max Lyapunov Exponent')
    ax2.axhline(y=np.mean(lambdas[-100:]), color='k', linestyle=':', label='Steady State')
    ax2.set_title('Lyapunov Exponent Convergence')
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel(r'$\lambda$ (bits/s)')
    ax2.grid(True, linestyle='--')
    ax2.legend()

    plt.tight_layout()
    plt.show()

## Interactive Time Evolution Plot for the Generalized Double Pendulum

This section provides an interactive plot showing the time evolution of the angular positions ($\phi_1, \phi_2$) and generalized momenta ($p_1, p_2$) of the non-linear double pendulum. You can adjust the physical parameters and initial conditions using the sliders to observe their effect on the system's dynamics.

In [ ]:
def interactive_time_evolution(m1, m2, l1, l2, g, phi1_0_deg, phi2_0_deg, p1_0, p2_0):
    # Convert initial angles from degrees to radians
    phi1_0 = np.deg2rad(phi1_0_deg)
    phi2_0 = np.deg2rad(phi2_0_deg)

    # Simulation time
    t_span = (0, 100) # Simulate for 100 seconds
    t_eval = np.linspace(t_span[0], t_span[1], 10000) # 10000 points for smooth plot
    y0 = [phi1_0, phi2_0, p1_0, p2_0] # Initial conditions

    # Solve the equations of motion using the hamilton_equations function
    sol = solve_ivp(hamilton_equations, t_span, y0, method='DOP853', t_eval=t_eval, atol=1e-12, rtol=1e-12, args=(m1, m2, l1, l2, g))

    # Extract solutions
    phi1_sol = sol.y[0]
    phi2_sol = sol.y[1]
    p1_sol = sol.y[2]
    p2_sol = sol.y[3]

    # Plotting the time evolution
    plt.figure(figsize=(12, 8))

    plt.subplot(2, 1, 1) # Two rows, one column, first plot
    plt.plot(sol.t, phi1_sol, label=r'$\phi_1$')
    plt.plot(sol.t, phi2_sol, label=r'$\phi_2$')
    plt.title('Angular Positions vs. Time')
    plt.xlabel('Time (s)')
    plt.ylabel('Angle (rad)')
    plt.grid(True)
    plt.legend()

    plt.subplot(2, 1, 2) # Two rows, one column, second plot
    plt.plot(sol.t, p1_sol, label=r'$p_1$')
    plt.plot(sol.t, p2_sol, label=r'$p_2$')
    plt.title('Generalized Momenta vs. Time')
    plt.xlabel('Time (s)')
    plt.ylabel('Momentum')
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()

# --- Interactive Sliders for Time Evolution Plot ---
widgets.interact(
    interactive_time_evolution,
    m1=widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='m1'),
    m2=widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='m2'),
    l1=widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='l1'),
    l2=widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='l2'),
    g=widgets.FloatSlider(min=0.0, max=20.0, step=0.1, value=9.81, description='g'),
    phi1_0_deg=widgets.FloatSlider(min=-360.0, max=360.0, step=5.0, value=150.0, description='phi1_0 (deg)'),
    phi2_0_deg=widgets.FloatSlider(min=-360.0, max=360.0, step=5.0, value=0.0, description='phi2_0 (deg)'),
    p1_0=widgets.FloatSlider(min=-5.0, max=5.0, step=0.1, value=0.0, description='p1_0'),
    p2_0=widgets.FloatSlider(min=-5.0, max=5.0, step=0.1, value=-3.4, description='p2_0')
);

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from IPython.display import display

# Reutiliza tu función system_dynamics (ahora global)
def calculate_lyapunov(y0, m1, m2, l1, l2, g, T_max=500, dt=0.5):
    t_steps = np.arange(0, T_max, dt)
    x = np.array(y0)
    # Start with a small perturbation vector in a random direction
    delta_x = np.random.rand(4) # Initialize with random direction
    delta_x /= np.linalg.norm(delta_x) # Normalize to unit length
    initial_perturbation_magnitude = 1e-8 # Initial magnitude for renormalization

    lyap_history = []
    running_sum = 0.0
    num_renormalizations = 0

    for i, t in enumerate(t_steps):
        # 1. Integrate the main trajectory and the perturbation vector together
        # The lyapunov_dynamics function (from rZvlgX469Qmp) handles the combined state
        combined_state = np.concatenate([x, delta_x * initial_perturbation_magnitude]) # Perturbation has initial magnitude

        sol = solve_ivp(lyapunov_dynamics, [t, t + dt], combined_state,
                        method='DOP853', atol=1e-12, rtol=1e-12, args=(m1, m2, l1, l2, g), max_step=dt)

        if sol.status != 0: # Check for integration failure
            lyap_history.extend([np.nan] * (len(t_steps) - i))
            break

        res = sol.y[:, -1]
        x_next = res[:4]
        new_delta_x_scaled = res[4:] # This is delta_x scaled by current divergence

        # Check for NaN/Inf in the integrated state or perturbation
        if np.any(np.isnan(x_next)) or np.any(np.isinf(x_next)) or \
           np.any(np.isnan(new_delta_x_scaled)) or np.any(np.isinf(new_delta_x_scaled)):
            lyap_history.extend([np.nan] * (len(t_steps) - i))
            break

        # Calculate the actual growth of the perturbation magnitude
        current_perturbation_magnitude = np.linalg.norm(new_delta_x_scaled)

        if current_perturbation_magnitude < 1e-20: # Avoid log of zero or extremely small numbers
            running_sum += np.log(1e-20 / initial_perturbation_magnitude) # Treat as minimal growth relative to initial
        elif current_perturbation_magnitude > 1e20: # Avoid log of too large numbers
            running_sum += np.log(1e20 / initial_perturbation_magnitude) # Treat as maximal growth relative to initial
        else:
            running_sum += np.log(current_perturbation_magnitude / initial_perturbation_magnitude)

        num_renormalizations += 1

        # Renormalize the perturbation vector for the next step
        delta_x = new_delta_x_scaled / current_perturbation_magnitude # Normalize new_delta_x_scaled to unit vector
        x = x_next

        # Calculate and store the instantaneous Lyapunov exponent
        if num_renormalizations > 0: # Ensure we don't divide by zero if loop runs for very short time
            lyap_history.append(running_sum / (num_renormalizations * dt))
        else:
            lyap_history.append(np.nan) # Should ideally not be reached if loop runs

    # If the loop broke early, ensure lyap_history has the correct length with NaNs
    while len(lyap_history) < len(t_steps):
        lyap_history.append(np.nan)

    return t_steps, lyap_history

def interactive_lyapunov_plot(m1, m2, l1, l2, g, phi1_0_deg, phi2_0_deg, p1_0, p2_0):
    # Convert initial angles from degrees to radians
    phi1_0 = np.deg2rad(phi1_0_deg)
    phi2_0 = np.deg2rad(phi2_0_deg)

    y0_actual = [phi1_0, phi2_0, p1_0, p2_0]
    # Using T_max=1000 and dt=0.5 for reasonable computation time and resolution
    t_mle, mle_val = calculate_lyapunov(y0_actual, m1, m2, l1, l2, g, T_max=1000, dt=0.5)

    plt.figure(figsize=(8,5))
    plt.plot(t_mle, mle_val, color='red', label='Maximal Lyapunov Exponent')
    plt.axhline(y=0, color='black', linestyle='--')
    plt.title("Lyapunov Exponent Convergence")
    plt.xlabel("Time")
    plt.ylabel(r"$\lambda$")
    plt.grid(True)
    plt.show()

    # Print final MLE, handling cases where it might be NaN
    if mle_val and not np.isnan(mle_val[-1]): # Check if mle_val is not empty
        print(f"Final MLE: {mle_val[-1]:.5f}")
    else:
        print("Final MLE: Diverged (NaN) or not computed.")

# --- Sliders interactivos para el exponente de Lyapunov ---
widgets.interact(
    interactive_lyapunov_plot,
    m1=widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='m1'),
    m2=widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='m2'),
    l1=widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='l1'),
    l2=widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='l2'),
    g=widgets.FloatSlider(min=0.0, max=20.0, step=0.1, value=9.81, description='g'),
    phi1_0_deg=widgets.FloatSlider(min=-360.0, max=360.0, step=5.0, value=np.rad2deg(0.5), description='phi1_0 (deg)'),
    phi2_0_deg=widgets.FloatSlider(min=-360.0, max=360.0, step=5.0, value=np.rad2deg(0.0), description='phi2_0 (deg)'),
    p1_0=widgets.FloatSlider(min=-5.0, max=5.0, step=0.1, value=0.0, description='p1_0'),
    p2_0=widgets.FloatSlider(min=-5.0, max=5.0, step=0.1, value=-3.4, description='p2_0')
);

## 2. Physical Model Implementation, Animation, and Analysis

Here, we will implement the provided Hamiltonian Equations of Motion, along with animating the divergence of two pendulums, verifying energy conservation, and calculating the high-resolution Poincaré map.

In [ ]:
# --- 2.1 Definición de Constantes y Función Hamiltoniana ---

# Constantes del sistema (mismas que las de los sliders, pero fijas para esta sección)
m1_const = 1
m2_const = 10.0
l1_const = 1.0
l2_const = 1.0
g_const = 9.81

def hamiltonian(y, m1, m2, l1, l2, g):
    phi1, phi2, p1, p2 = y

    delta = phi1 - phi2
    sin_d = np.sin(delta)
    cos_d = np.cos(delta)

    # Denominator from the kinetic energy part of the Hamiltonian (provided in the new formula)
    denominator_H_kinetic = (2 * l1**2 * l2**2 * m2 * (m1 + m2 * sin_d**2))
    if np.abs(denominator_H_kinetic) < 1e-12:
        return np.nan # Indicate an unstable state for Hamiltonian

    # Kinetic Energy part (first big fraction in the new Hamiltonian formula)
    kinetic_term_num = (m2 * l2**2 * p1**2 + (m1 + m2) * l1**2 * p2**2 - 2 * m2 * l1 * l2 * p1 * p2 * cos_d)
    kinetic_energy_H_part = kinetic_term_num / denominator_H_kinetic

    # Potential Energy part (the last two terms in the new Hamiltonian formula)
    potential_energy_H_part = -(m1 + m2) * l1 * g * np.cos(phi1) - m2 * l2 * g * np.cos(phi2)

    return kinetic_energy_H_part + potential_energy_H_part

print("Constants and Hamiltonian function defined.")

In [ ]:
from IPython.display import HTML
import matplotlib.animation as animation
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np # Ensure numpy is imported

# --- 2.2 Animation of Divergence, Energy Conservation, and Lyapunov Plot ---
# This section sets up and runs a simulation to demonstrate the chaotic nature
# of the double pendulum by showing the divergence of two initially close trajectories,
# verifying energy conservation, and plotting the Lyapunov exponent over time.

# Explicitly define constants here to avoid NameError issues
m1_const = 1.0
m2_const = 10.0
l1_const = 1.0
l2_const = 1.0
g_const = 9.81

# Also define the hamiltonian function here to ensure it's in scope
def hamiltonian(y, m1, m2, l1, l2, g):
    phi1, phi2, p1, p2 = y

    delta = phi1 - phi2
    sin_d = np.sin(delta)
    cos_d = np.cos(delta)

    # Denominator from the kinetic energy part of the Hamiltonian (provided in the new formula)
    denominator_H_kinetic = (2 * l1**2 * l2**2 * m2 * (m1 + m2 * sin_d**2))
    if np.abs(denominator_H_kinetic) < 1e-12:
        return np.nan # Indicate an unstable state for Hamiltonian

    # Kinetic Energy part (first big fraction in the new Hamiltonian formula)
    kinetic_term_num = (m2 * l2**2 * p1**2 + (m1 + m2) * l1**2 * p2**2 - 2 * m2 * l1 * l2 * p1 * p2 * cos_d)
    kinetic_energy_H_part = kinetic_term_num / denominator_H_kinetic

    # Potential Energy part (the last two terms in the new Hamiltonian formula)
    potential_energy_H_part = -(m1 + m2) * l1 * g * np.cos(phi1) - m2 * l2 * g * np.cos(phi2)

    return kinetic_energy_H_part + potential_energy_H_part


# --- Simulation Parameters ---
# T_max_anim: Maximum simulation time for the animation and Lyapunov analysis.
#             A longer time allows more observation of chaotic behavior.
# dt_anim: Time step for evaluating the solution. Smaller steps yield smoother animations
#          and more data points, but increase computation time.
T_max_anim = 30 # Reduced simulation time to generate a shorter animation
dt_anim = 0.05
t_eval_anim = np.linspace(0, T_max_anim, int(T_max_anim / dt_anim))

# --- Initial Conditions for Pendulum A (Reference Trajectory) ---
# These are fixed initial conditions, similar to the default slider values in previous sections.
# phi1_0_A, phi2_0_A: Initial angular positions (converted from degrees to radians).
# p1_0_A, p2_0_A: Initial generalized momenta.
phi1_0_A = np.deg2rad(150.0)
phi2_0_A = np.deg2rad(0.0)
p1_0_A = 0.0
p2_0_A = -3.4
y0_A = [phi1_0_A, phi2_0_A, p1_0_A, p2_0_A]

# --- Initial Conditions for Pendulum B (Perturbed Trajectory) ---
# A small perturbation is added to the initial momentum p2 of pendulum A.
# This perturbation is crucial for observing divergence and calculating the Lyapunov exponent.
perturbation = 1e-8
y0_B = list(y0_A) # Copy initial conditions from A
y0_B[3] += perturbation # Perturb p2_0

print("Simulating the trajectories of the two pendulums...")

# --- Solve Equations of Motion (EOMs) ---
# Using `solve_ivp` with 'DOP853' method for high accuracy.
# `atol` and `rtol` are set to very strict values (1e-12) to ensure numerical precision
# and better energy conservation, especially important for chaotic systems.
sol_A = solve_ivp(hamilton_equations, (0, T_max_anim), y0_A, method='DOP853', t_eval=t_eval_anim, atol=1e-12, rtol=1e-12, args=(m1_const, m2_const, l1_const, l2_const, g_const))
sol_B = solve_ivp(hamilton_equations, (0, T_max_anim), y0_B, method='DOP853', t_eval=t_eval_anim, atol=1e-12, rtol=1e-12, args=(m1_const, m2_const, l1_const, l2_const, g_const))

# --- Calculate Energy and Lyapunov Divergence ---
# Hamiltonian function is used to calculate total energy for each time step.
# This helps verify energy conservation over the simulation.
energy_A = np.array([hamiltonian(sol_A.y[:, i], m1_const, m2_const, l1_const, l2_const, g_const) for i in range(sol_A.y.shape[1])])
energy_B = np.array([hamiltonian(sol_B.y[:, i], m1_const, m2_const, l1_const, l2_const, g_const) for i in range(sol_B.y.shape[1])])

# Calculate the Euclidean distance between the two trajectories in phase space.
# The natural logarithm of the ratio of this distance to the initial perturbation
# is used to approximate the Lyapunov exponent.
distances = np.linalg.norm(sol_A.y - sol_B.y, axis=0)
# Replace very small distances to avoid issues with log(0) or log of negative numbers.
distances[distances < 1e-300] = 1e-300
log_distances = np.log(distances / perturbation)

# --- Animation Setup ---
# A figure with a grid of subplots is created to display the pendulum motion,
# the Lyapunov exponent's evolution, and energy conservation simultaneously.
fig = plt.figure(figsize=(15, 8))
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1], height_ratios=[2, 1])

ax_pendulum = fig.add_subplot(gs[0, 0])
ax_lyapunov = fig.add_subplot(gs[0, 1])
ax_energy = fig.add_subplot(gs[1, :])

# Initialize plot lines for pendulums, Lyapunov, and energy.
line_A, = ax_pendulum.plot([], [], 'o-', lw=2, color='blue', label='Pendulum A')
line_B, = ax_pendulum.plot([], [], 'o-', lw=2, color='red', label='Pendulum B')

# Configure pendulum plot limits and labels.
max_l = l1_const + l2_const
ax_pendulum.set_xlim(-(max_l + 0.1), (max_l + 0.1))
ax_pendulum.set_ylim(-(max_l + 0.1), (max_l + 0.1))
ax_pendulum.set_aspect('equal', 'box')
ax_pendulum.set_title('Double Pendulum Divergence')
ax_pendulum.grid(True)
ax_pendulum.legend()

# Configure Lyapunov plot.
lyap_line, = ax_lyapunov.plot([], [], lw=2, color='green')
ax_lyapunov.set_title('Lyapunov Exponent (log(distance))')
ax_lyapunov.set_xlabel('Time')
ax_lyapunov.set_ylabel(r'log($\Delta$x / $\Delta$x$_0$)')
ax_lyapunov.grid(True)

# Dynamically determine y-axis limits for the Lyapunov plot based on initial simulation results.
lyap_y_min = np.min(log_distances[~np.isnan(log_distances)]) if np.any(~np.isnan(log_distances)) else -10
lyap_y_max = np.max(log_distances[~np.isnan(log_distances)]) if np.any(~np.isnan(log_distances)) else 10
ax_lyapunov.set_xlim(0, T_max_anim)
ax_lyapunov.set_ylim(lyap_y_min - 1, lyap_y_max + 1)

# Configure energy plot with both pendulum's energy profiles.
ax_energy.plot(sol_A.t, energy_A, label='Energy Pendulum A', color='blue', linestyle='--')
ax_energy.plot(sol_B.t, energy_B, label='Energy Pendulum B', color='red', linestyle=':')
ax_energy.set_title('Energy Conservation (Hamiltonian)')
ax_energy.set_xlabel('Time')
ax_energy.set_ylabel('Total Energy')
ax_energy.grid(True)
ax_energy.legend()

# --- Animation Function ---
# This function updates the position of the pendulums and the Lyapunov plot
# for each frame of the animation.
def animate(i):
    # Calculate positions for Pendulum A
    x1_A = l1_const * np.sin(sol_A.y[0, i])
    y1_A = -l1_const * np.cos(sol_A.y[0, i])
    x2_A = x1_A + l2_const * np.sin(sol_A.y[1, i])
    y2_A = y1_A - l2_const * np.cos(sol_A.y[1, i])
    line_A.set_data([0, x1_A, x2_A], [0, y1_A, y2_A])

    # Calculate positions for Pendulum B
    x1_B = l1_const * np.sin(sol_B.y[0, i])
    y1_B = -l1_const * np.cos(sol_B.y[0, i])
    x2_B = x1_B + l2_const * np.sin(sol_B.y[1, i])
    y2_B = y1_B - l2_const * np.cos(sol_B.y[1, i])
    line_B.set_data([0, x1_B, x2_B], [0, y1_B, y2_B])

    # Update Lyapunov plot up to the current animation time.
    lyap_line.set_data(sol_A.t[:i+1], log_distances[:i+1])

    return line_A, line_B, lyap_line,

print("Generating animation...")
# Create the animation object. `blit=True` optimizes rendering by only redrawing changed elements.
an = animation.FuncAnimation(fig, animate, frames=len(sol_A.t), interval=dt_anim*1000, blit=True)
plt.tight_layout() # Adjust layout to prevent overlaps
plt.show() # Display the final frame of the animation

# --- Save Animation to File ---
# This saves the generated animation as an MP4 video file.
# `writer='ffmpeg'` requires ffmpeg to be installed (usually available in Colab).
# `fps` (frames per second) controls the playback speed of the video.
print("Saving animation as 'double_pendulum_divergence.mp4'...")
an.save('double_pendulum_divergence.mp4', writer='ffmpeg', fps=10)
print("Animation saved.")

# --- Energy Stability Check and Final Lyapunov Plot ---
# Print mean and standard deviation of energy to quantify conservation.
# A perfect system would have a standard deviation close to zero.
print("Animation generated. Checking energy stability...")
print(f"Mean Energy A: {np.nanmean(energy_A):.5f}, Std Dev A: {np.nanstd(energy_A):.5f}")
print(f"Mean Energy B: {np.nanmean(energy_B):.5f}, Std Dev B: {np.nanstd(energy_B):.5f}")

# Display the full Lyapunov exponent plot (non-animated) for reference.
plt.figure(figsize=(10, 6))
plt.plot(sol_A.t, log_distances, color='green')
plt.title('Lyapunov Exponent (log(distance) vs. Time)')
plt.xlabel('Time')
plt.ylabel(r'log($\Delta$x / $\Delta$x$_0$)')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# --- Poincaré Map Generation ---
# The Poincaré map is a powerful tool to visualize the phase space of a dynamical system.
# For a double pendulum, it helps in distinguishing between regular (periodic/quasi-periodic)
# and chaotic motion by plotting points only when a specific condition is met (a 'Poincaré section').

# Define the event function: when phi1 crosses zero (Poincaré section)
# This function returns the value of phi1. The integrator will find times when this value is zero.
def crossing_event(t, y, m1, m2, l1, l2, g):
    return y[0] # Returns phi1 (the first state variable)

# `terminal = False`: This ensures the simulation continues after the event is detected,
# allowing us to capture multiple crossings.
crossing_event.terminal = False

# `direction = 0`: This specifies that we want to detect crossings when phi1 is changing
# in any direction (from positive to negative or negative to positive).
# This captures more points for a richer Poincaré map.
crossing_event.direction = 0

# --- Long Simulation for the Poincaré Map ---
# T_map: Maximum simulation time for generating Poincaré points. A longer time is crucial
# for chaotic systems to adequately fill the phase space and reveal intricate patterns.
T_map = 5000 # Increased simulation time to capture more Poincaré points

# Run the simulation using solve_ivp with the defined event.
# `y0_A` are the initial conditions from the animation section.
# `method='DOP853'` provides high accuracy.
# `events=crossing_event` tells the integrator to look for the phi1=0 crossings.
# `atol` and `rtol` are set to very strict values (1e-12) for numerical precision.
# `args` pass the constant parameters to the hamilton_equations and crossing_event functions.
sol_map = solve_ivp(hamilton_equations, (0, T_map), y0_A, method='DOP853',
                    events=crossing_event, atol=1e-8, rtol=1e-8,
                    args=(m1_const, m2_const, l1_const, l2_const, g_const),
                    max_step=0.05) # Added max_step for potentially better stability and resolution

# --- Plotting the Poincaré Map ---
# The points where the event (phi1=0 crossing) occurred are stored in `sol_map.y_events[0]`.
# Each row in `y_events[0]` corresponds to a state vector (phi1, phi2, p1, p2) at a crossing.
if len(sol_map.y_events[0]) > 0:
    # Extract phi2 and p2 at each crossing point.
    # These are the variables typically used for a 2D Poincaré map of a double pendulum
    # when the section is taken at phi1=0.
    phi2_points = sol_map.y_events[0][:, 1]
    p2_points = sol_map.y_events[0][:, 3]

    # Create and display the scatter plot of the Poincaré map.
    plt.figure(figsize=(8, 6))
    plt.scatter(phi2_points, p2_points, s=1, color='blue', alpha=0.5)
    plt.title(f"Poincaré Map (Found {len(phi2_points)} points)")
    plt.xlabel(r"$\phi_2$")
    plt.ylabel(r"$p_2$")
    plt.grid(True)
    plt.show()
else:
    # If no crossings were found, inform the user.
    # This often happens with very short simulation times or if the initial conditions
    # do not lead to motion that crosses the chosen section.
    print("No crossings found. Try increasing simulation time (T_map) or adjusting initial energy (e.g., p2_0).")

## 3. Linearized Double Pendulum Analysis

Here, we will implement and analyze the linearized double pendulum system for small oscillations, compare its Poincaré map with the chaotic one, and visualize its phase space trajectories. We'll use the simplified case where $m_1=m_2=m$ and $l_1=l_2=l$ as suggested.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# --- 3.1 Define Linearized System Dynamics ---
# For small oscillations and m1=m2=m, l1=l2=l
def system_dynamics_linearized(t, y, m_val, l_val, g_val):
    phi1, phi2, dphi1, dphi2 = y

    # Linearized equations of motion for m1=m2=m and l1=l2=l
    # From: 2l * ddot(phi1) + l * ddot(phi2) + 2g * phi1 = 0
    #       l * ddot(phi1) + l * ddot(phi2) + g * phi2 = 0

    # Solving for ddot(phi1) and ddot(phi2):
    ddot_phi1 = (g_val / l_val) * (phi2 - 2 * phi1)
    ddot_phi2 = (g_val / l_val) * (2 * phi1 - 2 * phi2)

    return [dphi1, dphi2, ddot_phi1, ddot_phi2]

# --- 3.2 Simulation Parameters for Linearized System ---
# Using constant values from the main simulation for consistency where applicable
m_linearized = m1_const # Assume m1=m2=m_const
l_linearized = l1_const # Assume l1=l2=l_const
g_linearized = g_const

# Small initial conditions for linearized system
phi1_0_linearized = np.deg2rad(5)  # 5 degrees
phi2_0_linearized = np.deg2rad(0)
dphi1_0_linearized = 0.0
dphi2_0_linearized = 0.0
y0_linearized = [phi1_0_linearized, phi2_0_linearized, dphi1_0_linearized, dphi2_0_linearized]

T_linearized = 2000 # Longer time to observe oscillations clearly and generate more points for Poincare map
t_eval_linearized = np.linspace(0, T_linearized, 20000)

print("Simulating linearized double pendulum...")
sol_linearized = solve_ivp(system_dynamics_linearized, (0, T_linearized), y0_linearized, method='DOP853', t_eval=t_eval_linearized, atol=1e-12, rtol=1e-12, args=(m_linearized, l_linearized, g_linearized))

phi1_lin_sol = sol_linearized.y[0]
phi2_lin_sol = sol_linearized.y[1]
dphi1_lin_sol = sol_linearized.y[2]
dphi2_lin_sol = sol_linearized.y[3]

print("Linearized simulation complete.")

### 3.3 Interactive Time Evolution Plot for Linearized System

This interactive plot visualizes the time evolution of the angular positions ($\phi_1, \phi_2$) and angular velocities ($\dot{\phi}_1, \dot{\phi}_2$) for the *linearized* double pendulum. You can adjust the initial conditions using the sliders below to observe the characteristic regular (non-chaotic) motion.

In [ ]:
def interactive_linearized_time_evolution(m_val, l_val, g_val, phi1_0_deg, phi2_0_deg, dphi1_0, dphi2_0):
    # Convert initial angles from degrees to radians
    phi1_0 = np.deg2rad(phi1_0_deg)
    phi2_0 = np.deg2rad(phi2_0_deg)

    # Simulation time
    t_span = (0, 100) # Simulate for 100 seconds
    t_eval = np.linspace(t_span[0], t_span[1], 10000) # 10000 points for smooth plot
    y0 = [phi1_0, phi2_0, dphi1_0, dphi2_0] # Initial conditions

    # Solve the equations of motion for the linearized system
    sol = solve_ivp(system_dynamics_linearized, t_span, y0, method='DOP853', t_eval=t_eval, atol=1e-12, rtol=1e-12, args=(m_val, l_val, g_val))

    # Extract solutions
    phi1_sol = sol.y[0]
    phi2_sol = sol.y[1]
    dphi1_sol = sol.y[2]
    dphi2_sol = sol.y[3]

    # Plotting the time evolution
    plt.figure(figsize=(12, 8))

    plt.subplot(2, 1, 1) # Two rows, one column, first plot
    plt.plot(sol.t, phi1_sol, label=r'$\phi_1$')
    plt.plot(sol.t, phi2_sol, label=r'$\phi_2$')
    plt.title('Linearized: Angular Positions vs. Time')
    plt.xlabel('Time (s)')
    plt.ylabel('Angle (rad)')
    plt.grid(True)
    plt.legend()

    plt.subplot(2, 1, 2) # Two rows, one column, second plot
    plt.plot(sol.t, dphi1_sol, label=r'$\dot{\phi}_1$')
    plt.plot(sol.t, dphi2_sol, label=r'$\dot{\phi}_2$')
    plt.title('Linearized: Angular Velocities vs. Time')
    plt.xlabel('Time (s)')
    plt.ylabel('Angular Velocity (rad/s)')
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()

# --- Interactive Sliders for Linearized Time Evolution Plot ---
widgets.interact(
    interactive_linearized_time_evolution,
    m_val=widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=m_linearized, description='m'),
    l_val=widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=l_linearized, description='l'),
    g_val=widgets.FloatSlider(min=0.0, max=20.0, step=0.1, value=g_linearized, description='g'),
    phi1_0_deg=widgets.FloatSlider(min=-30.0, max=30.0, step=1.0, value=np.rad2deg(phi1_0_linearized), description='phi1_0 (deg)'),
    phi2_0_deg=widgets.FloatSlider(min=-30.0, max=30.0, step=1.0, value=np.rad2deg(phi2_0_linearized), description='phi2_0 (deg)'),
    dphi1_0=widgets.FloatSlider(min=-5.0, max=5.0, step=0.1, value=dphi1_0_linearized, description='dphi1_0'),
    dphi2_0=widgets.FloatSlider(min=-5.0, max=5.0, step=0.1, value=dphi2_0_linearized, description='dphi2_0')
);

### 3.3 Phase Space Plots for Linearized System

These plots show the trajectories of the linearized system in different phase planes. For a stable, non-chaotic system, we expect to see regular, bounded orbits (e.g., ellipses).

In [ ]:
# Plot phi1 vs phi2
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(phi1_lin_sol, phi2_lin_sol, color='purple', linewidth=0.5)
plt.title(r'Linearized: $\phi_1$ vs $\phi_2$')
plt.xlabel(r'$\phi_1$ (radians)')
plt.ylabel(r'$\phi_2$ (radians)')
plt.grid(True)

# Plot dphi1 vs dphi2
plt.subplot(1, 2, 2)
plt.plot(dphi1_lin_sol, dphi2_lin_sol, color='darkgreen', linewidth=0.5)
plt.title(r'Linearized: $\dot{\phi}_1$ vs $\dot{\phi}_2$')
plt.xlabel(r'$\dot{\phi}_1$ (rad/s)')
plt.ylabel(r'$\dot{\phi}_2$ (rad/s)')
plt.grid(True)

plt.tight_layout()
plt.show()

### 3.4 Analytical Time Evolution for Simplified Linearized System

This section plots the analytical solutions for the angular positions ($\phi_1, \phi_2$) and angular velocities ($\dot{\phi}_1, \dot{\phi}_2$) of the linearized double pendulum, specifically for the simplified case where $m_1=m_2=m$ and $l_1=l_2=l$. The general analytical solutions are based on the superposition of two normal modes.

For the normal modes, with $m_1=m_2=m$ and $l_1=l_2=l$, the angular frequencies are given by:

$$\omega_1^2 = \frac{g}{l}(2+\sqrt{2}) \quad \text{and} \quad \omega_2^2 = \frac{g}{l}(2-\sqrt{2})$$

And the corresponding eigenvectors (ratios of $\phi_1/\phi_2$) are:

$$r_1 = \frac{1+\sqrt{2}}{2+\sqrt{2}} = \frac{\sqrt{2}}{2} \quad \text{and} \quad r_2 = \frac{1-\sqrt{2}}{2-\sqrt{2}} = -\frac{\sqrt{2}}{2}$$

The general solution for small oscillations, with zero initial angular velocities, is:

$$\phi_1(t) = A_1 r_1 \cos(\omega_1 t) + A_2 r_2 \cos(\omega_2 t)$$
$$\phi_2(t) = A_1 \cos(\omega_1 t) + A_2 \cos(\omega_2 t)$$

Where $A_1$ and $A_2$ are amplitudes determined by the initial angular positions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters for the simplified linearized system (from previous definitions)
m = m_linearized
l = l_linearized
g = g_linearized

# Initial conditions (from previous definitions for linearized system)
phi1_0_rad = phi1_0_linearized # np.deg2rad(5)
phi2_0_rad = phi2_0_linearized # np.deg2rad(0)
# dphi1_0 and dphi2_0 are assumed to be 0 for this analytical solution

# Calculate constants for the analytical solution
sqrt2 = np.sqrt(2)

# Eigenvector ratios (r = phi1/phi2)
r1 = sqrt2 / 2
r2 = -sqrt2 / 2

# Angular frequencies (omegas)
omega1 = np.sqrt(g / l * (2 + sqrt2))
omega2 = np.sqrt(g / l * (2 - sqrt2))

# Solve for amplitudes A1 and A2 using initial conditions at t=0
# phi1_0_rad = A1 * r1 + A2 * r2
# phi2_0_rad = A1 + A2

# Given initial conditions: phi1_0_rad = np.deg2rad(5), phi2_0_rad = 0
# Since phi2_0_rad = 0, we have A1 = -A2
# Substitute into the first equation:
# phi1_0_rad = -A2 * r1 + A2 * r2 = A2 * (r2 - r1)

if (r2 - r1) != 0:
    A2 = phi1_0_rad / (r2 - r1)
    A1 = -A2
else:
    print("Warning: (r2 - r1) is zero. Cannot determine unique amplitudes. Setting A1=A2=0.")
    A1 = 0.0
    A2 = 0.0

# Time array for plotting
t_analytic = np.linspace(0, T_linearized, 1000) # Using T_linearized from earlier

# Analytical solutions for phi1, phi2
phi1_analytic = A1 * r1 * np.cos(omega1 * t_analytic) + A2 * r2 * np.cos(omega2 * t_analytic)
phi2_analytic = A1 * np.cos(omega1 * t_analytic) + A2 * np.cos(omega2 * t_analytic)

# Analytical solutions for dphi1, dphi2
dphi1_analytic = -A1 * r1 * omega1 * np.sin(omega1 * t_analytic) - A2 * r2 * omega2 * np.sin(omega2 * t_analytic)
dphi2_analytic = -A1 * omega1 * np.sin(omega1 * t_analytic) - A2 * omega2 * np.sin(omega2 * t_analytic)

# Plotting the analytical time evolution
plt.figure(figsize=(12, 8))

plt.subplot(2, 1, 1) # Two rows, one column, first plot
plt.plot(t_analytic, phi1_analytic, label=r'$\phi_1$ (Analytical)')
plt.plot(t_analytic, phi2_analytic, label=r'$\phi_2$ (Analytical)')
plt.title('Linearized: Analytical Angular Positions vs. Time')
plt.xlabel('Time (s)')
plt.ylabel('Angle (rad)')
plt.grid(True)
plt.legend()

plt.subplot(2, 1, 2) # Two rows, one column, second plot
plt.plot(t_analytic, dphi1_analytic, label=r'$\dot{\phi}_1$ (Analytical)')
plt.plot(t_analytic, dphi2_analytic, label=r'$\dot{\phi}_2$ (Analytical)')
plt.title('Linearized: Analytical Angular Velocities vs. Time')
plt.xlabel('Time (s)')
plt.ylabel('Angular Velocity (rad/s)')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

### 3.4 Poincaré Map for Linearized System

We will now generate a Poincaré map for the linearized system using the same section ($\phi_1=0$, $\dot{\phi}_1 > 0$) as the chaotic system. For comparison, we will approximate $p_2$ using the small angle approximation for generalized momenta:

$p_2 \approx m l^2 \dot{\phi}_2 + m l^2 \dot{\phi}_1$ (for $m_1=m_2=m$, $l_1=l_2=l$, and $\cos(\phi_1-\phi_2) \approx 1$).

In [ ]:
# Define event function for linearized system
def crossing_event_linearized(t, y, m_val, l_val, g_val):
    return y[0] # phi1
crossing_event_linearized.terminal = False
crossing_event_linearized.direction = 1 # Positive crossing

print("Generating Poincaré map for linearized system...")
sol_map_linearized = solve_ivp(system_dynamics_linearized, (0, T_linearized), y0_linearized, method='DOP853',
                               events=crossing_event_linearized, atol=1e-8, rtol=1e-8,
                               args=(m_linearized, l_linearized, g_linearized))

if len(sol_map_linearized.y_events[0]) > 0:
    phi1_lin_points = sol_map_linearized.y_events[0][:, 0]
    phi2_lin_points = sol_map_linearized.y_events[0][:, 1]
    dphi1_lin_points = sol_map_linearized.y_events[0][:, 2]
    dphi2_lin_points = sol_map_linearized.y_events[0][:, 3]

    # Approximate p2 for comparison with the chaotic case
    # Using p2 = m_val * l_val**2 * dphi2 + m_val * l_val**2 * dphi1 (small angle approx)
    p2_lin_points = m_linearized * l_linearized**2 * dphi2_lin_points + m_linearized * l_linearized**2 * dphi1_lin_points

    plt.figure(figsize=(8, 6))
    plt.scatter(phi2_lin_points, p2_lin_points, s=5, color='orange', alpha=0.7)
    plt.title(f"Linearized Poincaré Map (Found {len(phi2_lin_points)} points)")
    plt.xlabel(r'$\phi_2$ (radians)')
    plt.ylabel(r'$p_2$ (approx.)')
    plt.grid(True)
    plt.show()
else:
    print("No crossings found for linearized Poincaré map. This is unexpected for a well-behaved linear system.")

print("Linearized Poincaré map generated.")

### 3.5 Comparison of Poincaré Maps (Chaotic vs. Linearized)

Finally, let's compare the Poincaré map from the full, non-linear (potentially chaotic) double pendulum with the one generated from the linearized system. The chaotic map should show a more complex, scattered pattern, while the linearized map should exhibit regular, organized points (e.g., ellipses or a single point for periodic motion).

In [ ]:
plt.figure(figsize=(14, 6))

# Subplot 1: Chaotic Poincaré Map (re-plot from previous section for direct comparison)
plt.subplot(1, 2, 1)
if len(sol_map.y_events[0]) > 0:
    plt.scatter(phi2_points, p2_points, s=1, color='blue', alpha=0.5)
    plt.title(f"Chaotic Poincaré Map (Found {len(phi2_points)} points)")
    plt.xlabel(r'$\phi_2$ (radians)')
    plt.ylabel(r'$p_2$')
    plt.grid(True)
else:
    plt.text(0.5, 0.5, 'No chaotic crossings found.', horizontalalignment='center', verticalalignment='center', transform=plt.gca().transAxes)
    plt.title('Chaotic Poincaré Map')

# Subplot 2: Linearized Poincaré Map
plt.subplot(1, 2, 2)
if len(sol_map_linearized.y_events[0]) > 0:
    plt.scatter(phi2_lin_points, p2_lin_points, s=5, color='orange', alpha=0.7)
    plt.title(f"Linearized Poincaré Map (Found {len(phi2_lin_points)} points)")
    plt.xlabel(r'$\phi_2$ (radians)')
    plt.ylabel(r'$p_2$ (approx.)')
    plt.grid(True)
else:
    plt.text(0.5, 0.5, 'No linearized crossings found.', horizontalalignment='center', verticalalignment='center', transform=plt.gca().transAxes)
    plt.title('Linearized Poincaré Map')

plt.tight_layout()
plt.show()

## 3.6 Comparative Animation: Non-Linear (Chaotic) vs. Linearized Double Pendulum

This animation provides a direct visual comparison between the chaotic motion of the full non-linear double pendulum and the regular, oscillatory behavior of its linearized approximation (for small angles). It highlights the fundamental differences in their oscillation modes and the conservation of energy for both systems.

In [ ]:
def energy_linearized_double_pendulum(state, m, l, g):
    """Calculates the total energy (Kinetic + Potential) for a linearized double pendulum.
    Assumes m1=m2=m and l1=l2=l for the simplified case.
    State: [phi1, phi2, dphi1, dphi2]
    """
    phi1, phi2, dphi1, dphi2 = state

    # Kinetic Energy (T) based on simplified m1=m2=m, l1=l2=l
    T = 0.5 * (m + m) * l**2 * dphi1**2 + \
        0.5 * m * l**2 * dphi2**2 + \
        m * l * l * dphi1 * dphi2

    # Potential Energy (V) based on simplified m1=m2=m, l1=l2=l and small angle approximation
    V = 0.5 * (m + m) * g * l * phi1**2 + \
        0.5 * m * g * l * phi2**2

    return T + V

print("Linearized double pendulum energy function defined.")

## 4. Summary of Equations and Conclusions

### 4.1 Hamiltonian Equations of the Double Pendulum (Non-Linear)

The Hamiltonian equations of motion used in the simulation are as follows, where $\phi_1, \phi_2$ are the generalized angles and $p_1, p_2$ are the generalized momenta. We assume $m_1, m_2$ for the masses, $l_1, l_2$ for the lengths, and $g$ for gravity.

The system's Hamiltonian is:

$$H = \frac{m_2 l_2^2 p_1^2 + (m_1+m_2) l_1^2 p_2^2 - 2 m_2 l_1 l_2 p_1 p_2 \cos(\phi_1-\phi_2)}{2 l_1^2 l_2^2 m_2 [m_1+m_2 \sin^2(\phi_1-\phi_2)]} - (m_1+m_2) l_1 g \cos(\phi_1) - m_2 l_2 g \cos(\phi_2)$$

The equations of motion (derived from the Hamiltonian) are:

Let $\Delta = \phi_1 - \phi_2$ and $D = m_1 + m_2 \sin^2(\Delta)$.

The angular velocities are:

$$\dot{\phi_1} = \frac{l_2 p_1 - l_1 p_2 \cos(\Delta)}{l_1^2 l_2 D}$$

$$\dot{\phi_2} = \frac{l_1 (m_1+m_2) p_2 - l_2 m_2 p_1 \cos(\Delta)}{l_1 l_2^2 m_2 D}$$

The changes in momenta are:

To calculate $K_{\Delta} = \frac{\partial K}{\partial \Delta}$ (the derivative of the kinetic energy term of the Hamiltonian with respect to $\Delta = \phi_1 - \phi_2$):

First, define the numerator and denominator of the kinetic energy term:
$K_{num} = m_2 l_2^2 p_1^2 + (m_1+m_2) l_1^2 p_2^2 - 2 m_2 l_1 l_2 p_1 p_2 \cos(\Delta)$
$D = m_1 + m_2 \sin^2(\Delta)$

Then, the derivatives of these with respect to $\Delta$ are:
$\frac{\partial K_{num}}{\partial \Delta} = 2 m_2 l_1 l_2 p_1 p_2 \sin(\Delta)$
$\frac{\partial D}{\partial \Delta} = 2 m_2 \sin(\Delta) \cos(\Delta)$

Using the quotient rule, the derivative of $\left( \frac{K_{num}}{D} \right)$ with respect to $\Delta$ is:
$\frac{\partial}{\partial \Delta} \left( \frac{K_{num}}{D} \right) = \frac{D \frac{\partial K_{num}}{\partial \Delta} - K_{num} \frac{\partial D}{\partial \Delta}}{D^2}$

Finally, $K_{\Delta}$ is given by:
$K_{\Delta} = \frac{1}{2 l_1^2 l_2^2 m_2} \times \frac{\partial}{\partial \Delta} \left( \frac{K_{num}}{D} \right)$

$$\dot{p_1} = -K_{\Delta} - (m_1+m_2) l_1 g \sin(\phi_1)$$

$$\dot{p_2} = K_{\Delta} - m_2 l_2 g \sin(\phi_2)$$

### 4.2 Conclusions on the Comparison of Poincaré Maps

The comparison between the Poincaré Map of the non-linear (chaotic) system and the linearized system reveals fundamental differences in their dynamics:

*   **Linearized System:** The Poincaré map for the linearized double pendulum (valid for small oscillations and simplified conditions) shows **regular and closed trajectories**, often ellipses or discrete points. This is characteristic of integrable and predictable systems, where energy and other invariants are strictly conserved, and the motion is periodic or quasi-periodic. The points are organized in an orderly manner, reflecting an underlying harmonic motion.

*   **Non-Linear (Chaotic) System:** Once the correct equations of motion were implemented, the Poincaré map of the non-linear double pendulum exhibits a **much more complex and scattered structure**. Instead of closed curves, **many points form dense and often fractal patterns**. This "cloud" of points is a signature of chaotic behavior, where the system explores a vast region of phase space in an apparently random manner, even with very similar initial conditions. The lack of a discernible pattern and the dense filling of the phase space indicate an extreme sensitivity to initial conditions and the non-integrability of the system.

In summary, the Poincaré map is a powerful tool that clearly visualizes the transition from regular, predictable motion to chaotic, unpredictable motion, depending on the linear or non-linear nature of the underlying equations of motion.

## Comparative Table: Non-Linear (Chaotic) vs. Linearized Double Pendulum

This table summarizes the key characteristics and observed differences between the behavior of the double pendulum in its non-linear (potentially chaotic) formulation and its linearized approximation for small oscillations.

| Characteristic            | Non-Linear (Chaotic) System                         | Linearized System                                      |
| :------------------------ | :-------------------------------------------------- | :----------------------------------------------------- |
| **Equations of Motion**   | Full, non-linear Hamiltonian equations.             | Second-order, linear differential equations.           |
| **Validity**              | For any oscillation angle.                          | Only for small oscillations ($\phi \ll 1$ rad).      |
| **Behavior**              | Highly sensitive to initial conditions (chaotic).   | Predictable and stable (regular, periodic, or quasi-periodic). |
| **Lyapunov Exponent**     | Positive (indicates chaos and exponential divergence). | Zero or negative (indicates stability and no divergence). |
| **Energy Conservation**   | Strictly conserved (if the model is Hamiltonian).   | Approximately conserved (with small numerical deviations). |
| **Poincaré Map**          | Scattered points forming complex and fractal patterns. | Points organized in closed orbits (ellipses, discrete points). |
| **Phase Space**           | Trajectories densely filling complex regions.       | Trajectories in closed, regular orbits.                |
| **Integrability**         | Non-integrable (generally).                         | Integrable (two well-defined normal modes).            |
| **Example**               | Divergence of two identical pendulums with minimal initial differences. | Harmonic oscillations with fixed frequencies and amplitudes. |

## Visualization of the Complete Phase Space (Non-Linear System)

Here we plot the complete trajectories in the phase space for the non-linear double pendulum. These are the trajectories that, when intersecting the Poincaré plane ($\phi_1=0$), generate the points observed in the Poincaré map.

In [ ]:
import matplotlib.pyplot as plt

# Assuming sol_map contains the full solution for the non-linear system from the Poincaré map section
# sol_map was generated in cell `48d9bfc7` or `da8a9f17` with T_map = 5000

if 'sol_map' in locals() and sol_map is not None and sol_map.y.shape[1] > 0:
    phi1_full_sol = sol_map.y[0]
    phi2_full_sol = sol_map.y[1]
    p1_full_sol = sol_map.y[2]
    p2_full_sol = sol_map.y[3]

    plt.figure(figsize=(14, 6))

    # Plot phi1 vs p1
    plt.subplot(1, 2, 1)
    plt.plot(phi1_full_sol, p1_full_sol, color='blue', linewidth=0.5, alpha=0.7)
    plt.title(r'Phase Space: $\phi_1$ vs $p_1$ (Non-Linear)')
    plt.xlabel(r'$\phi_1$ (rad)')
    plt.ylabel(r'$p_1$')
    plt.grid(True)

    # Plot phi2 vs p2
    plt.subplot(1, 2, 2)
    plt.plot(phi2_full_sol, p2_full_sol, color='red', linewidth=0.5, alpha=0.7)
    plt.title(r'Phase Space: $\phi_2$ vs $p_2$ (Non-Linear)')
    plt.xlabel(r'$\phi_2$ (rad)')
    plt.ylabel(r'$p_2$')
    plt.grid(True)

    plt.tight_layout()
    plt.show()
else:
    print("The `sol_map` solution is not available or is empty. Please ensure that the non-linear Poincaré Map generation cell has been executed correctly.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Default gravity constant
DEFAULT_G = 9.81

# Default length for standard case frequency calculation
DEFAULT_L_STANDARD = 1.0

In [ ]:
def calculate_general_params(l1, l2, m1, m2, g):
    """Calculates amplitudes (A) and angular frequencies (omega) for the general case.
    Returns two tuples: (A_plus, omega_plus), (A_minus, omega_minus).
    Handles potential complex results from square roots.
    """
    if l1 <= 0 or l2 <= 0 or m1 <= 0 or m2 <= 0:
        print("Error: Masses and lengths must be positive.")
        return (0j, 0j), (0j, 0j)

    alpha = (4 * m1) / (m1 + m2)

    # Term inside sqrt for A's numerator
    arg_sqrt_num_A = (l2 + l1)**2 - alpha * l2 * l1
    sqrt_num_A = np.sqrt(complex(arg_sqrt_num_A))

    A_num_plus = (1 - l2 + l1) + sqrt_num_A
    A_num_minus = (1 - l2 + l1) - sqrt_num_A

    # Term inside sqrt for A's denominator and omega
    arg_sqrt_denom_A_omega = (1 + l1/l2)**2 - alpha * l1/l2
    sqrt_denom_A_omega = np.sqrt(complex(arg_sqrt_denom_A_omega))

    A_denom_plus = (1 + l1/l2) + sqrt_denom_A_omega
    A_denom_minus = (1 + l1/l2) - sqrt_denom_A_omega

    # Frequencies
    omega_plus = g * (m1 + m2) / (2 * m1 * l1) * A_denom_plus if A_denom_plus != 0 else 0j
    omega_minus = g * (m1 + m2) / (2 * m1 * l1) * A_denom_minus if A_denom_minus != 0 else 0j

    # Amplitudes
    A_plus = (2 * m1 / (m1 + m2)) * A_num_plus / A_denom_plus if A_denom_plus != 0 else 0j
    A_minus = (2 * m1 / (m1 + m2)) * A_num_minus / A_denom_minus if A_denom_minus != 0 else 0j

    return (A_plus, omega_plus), (A_minus, omega_minus)

In [ ]:
def calculate_standard_params(g, l_val=DEFAULT_L_STANDARD):
    """Calculates amplitudes (A) and angular frequencies (omega) for the standard linearized case.
    """
    # Frequencies (interpreted as g/l * [2±sqrt(2)])
    omega_s_plus = (g / l_val) * (2 + np.sqrt(2))
    omega_s_minus = (g / l_val) * (2 - np.sqrt(2))

    # Amplitudes - Updated according to the provided standard formula
    A_s_plus = (1 + np.sqrt(2)) / (2 + np.sqrt(2))
    A_s_minus = (1 - np.sqrt(2)) / (2 - np.sqrt(2))

    return (A_s_plus, omega_s_plus), (A_s_minus, omega_s_minus)

In [ ]:
def plot_pendulum(l1, l2, m1, m2, g, C1_gen, C2_gen, C1_std, C2_std, save_plot=False):
    """Plots the angular displacement for both general and standard cases.
    """
    t = np.linspace(0, 10, 500) # Time from 0 to 10 seconds

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

    # --- General Case Plot ---
    (A_gen_plus, omega_gen_plus), (A_gen_minus, omega_gen_minus) = calculate_general_params(l1, l2, m1, m2, g)

    phi_gen1_t = C1_gen * A_gen_plus * np.exp(1j * omega_gen_plus * t)
    phi_gen2_t = C2_gen * A_gen_minus * np.exp(1j * omega_gen_minus * t)

    axes[0].plot(t, np.real(phi_gen1_t), label='General Mode 1 (Re($\\phi_1$))', color='blue')
    axes[0].plot(t, np.real(phi_gen2_t), label='General Mode 2 (Re($\\phi_2$))', color='red')
    axes[0].set_title(f'General Linearized Double Pendulum (l1={l1}, l2={l2}, m1={m1}, m2={m2}, g={g:.2f})')
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Angular Displacement (rad)')
    axes[0].legend()
    axes[0].grid(True)

    # --- Standard Case Plot ---
    (A_std_plus, omega_std_plus), (A_std_minus, omega_std_minus) = calculate_standard_params(g, l_val=DEFAULT_L_STANDARD)

    phi_std1_t = C1_std * A_std_plus * np.exp(1j * omega_std_plus * t)
    phi_std2_t = C2_std * A_std_minus * np.exp(1j * omega_std_minus * t)

    axes[1].plot(t, np.real(phi_std1_t), label='Standard Mode 1 (Re($\\phi_1$))', color='green')
    axes[1].plot(t, np.real(phi_std2_t), label='Standard Mode 2 (Re($\\phi_2$))', color='purple')
    axes[1].set_title(f'Standard Linearized Double Pendulum (l=1, m=1, g={g:.2f})')
    axes[1].set_xlabel('Time (s)')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    if save_plot:
        plt.savefig(f'angular_displacement_l1_{l1}_l2_{l2}_m1_{m1}_m2_{m2}_g_{g:.2f}.png')
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Default gravity constant
DEFAULT_G = 9.81

# Default length for standard case frequency calculation
DEFAULT_L_STANDARD = 1.0

# Define sliders
l1_slider = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='l1 (m):', continuous_update=True)
l2_slider = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='l2 (m):', continuous_update=True)
m1_slider = widgets.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='m1 (kg):', continuous_update=True)
m2_slider = widgets.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='m2 (kg):', continuous_update=True)
g_slider = widgets.FloatSlider(value=DEFAULT_G, min=0.1, max=20.0, step=0.1, description='g (m/s²):', continuous_update=True)
C1_gen_slider = widgets.FloatSlider(value=1.0, min=-5.0, max=5.0, step=0.1, description='C1 General:', continuous_update=True)
C2_gen_slider = widgets.FloatSlider(value=1.0, min=-5.0, max=5.0, step=0.1, description='C2 General:', continuous_update=True)
C1_std_slider = widgets.FloatSlider(value=1.0, min=-5.0, max=5.0, step=0.1, description='C1 Standard:', continuous_update=True)
C2_std_slider = widgets.FloatSlider(value=1.0, min=-5.0, max=5.0, step=0.1, description='C2 Standard:', continuous_update=True)
save_plot_checkbox = widgets.Checkbox(value=False, description='Save Plot (PNG)', indent=False)

# Create an interactive widget
interactive_plot = widgets.interactive(
    plot_pendulum,
    l1=l1_slider,
    l2=l2_slider,
    m1=m1_slider,
    m2=m2_slider,
    g=g_slider,
    C1_gen=C1_gen_slider,
    C2_gen=C2_gen_slider,
    C1_std=C1_std_slider,
    C2_std=C2_std_slider,
    save_plot=save_plot_checkbox
)

# Display the widget
display(interactive_plot)

In [ ]:
def plot_phi1_vs_phi2(l1, l2, m1, m2, g, C1_gen, C2_gen, C1_std, C2_std):
    """Plots Re(phi1) vs Re(phi2) for both general and standard cases."""
    t = np.linspace(0, 10, 500) # Time from 0 to 10 seconds

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # --- General Case Plot (Re(phi1) vs Re(phi2)) ---
    (A_gen_plus, omega_gen_plus), (A_gen_minus, omega_gen_minus) = calculate_general_params(l1, l2, m1, m2, g)
    phi_gen1_t = C1_gen * A_gen_plus * np.exp(1j * omega_gen_plus * t) + \
                 C2_gen * A_gen_minus * np.exp(1j * omega_gen_minus * t)
    # Assuming the second amplitude component is 1 for phi2 based on the formula structure
    phi_gen2_t = C1_gen * np.exp(1j * omega_gen_plus * t) + \
                 C2_gen * np.exp(1j * omega_gen_minus * t)

    axes[0].plot(np.real(phi_gen1_t), np.real(phi_gen2_t), label='General', color='blue')
    axes[0].set_title(f'General Linearized: Re($\\phi_1$) vs Re($\\phi_2$)\n(l1={l1}, l2={l2}, m1={m1}, m2={m2}, g={g:.2f})')
    axes[0].set_xlabel('Re($\\phi_1$)')
    axes[0].set_ylabel('Re($\\phi_2$)')
    axes[0].legend()
    axes[0].grid(True)
    axes[0].set_aspect('equal', adjustable='box')

    # --- Standard Case Plot (Re(phi1) vs Re(phi2)) ---
    (A_std_plus, omega_std_plus), (A_std_minus, omega_std_minus) = calculate_standard_params(g, l_val=DEFAULT_L_STANDARD)
    phi_std1_t = C1_std * A_std_plus * np.exp(1j * omega_std_plus * t) + \
                 C2_std * A_std_minus * np.exp(1j * omega_std_minus * t)
    # Assuming the second amplitude component is 1 for phi2 based on the formula structure
    phi_std2_t = C1_std * np.exp(1j * omega_std_plus * t) + \
                 C2_std * np.exp(1j * omega_std_minus * t)

    axes[1].plot(np.real(phi_std1_t), np.real(phi_std2_t), label='Standard', color='green')
    axes[1].set_title(f'Standard Linearized: Re($\\phi_1$) vs Re($\\phi_2$)\n(l=1, m=1, g={g:.2f})')
    axes[1].set_xlabel('Re($\\phi_1$)')
    axes[1].set_ylabel('Re($\\phi_2$)')
    axes[1].legend()
    axes[1].grid(True)
    axes[1].set_aspect('equal', adjustable='box')

    plt.tight_layout()
    plt.show()

# Create an interactive widget for the new plot, using the same sliders as before
interactive_phi1_phi2_plot = widgets.interactive(
    plot_phi1_vs_phi2,
    l1=l1_slider,
    l2=l2_slider,
    m1=m1_slider,
    m2=m2_slider,
    g=g_slider,
    C1_gen=C1_gen_slider,
    C2_gen=C2_gen_slider,
    C1_std=C1_std_slider,
    C2_std=C2_std_slider
)

# Display the new interactive plot
display(interactive_phi1_phi2_plot)

In [ ]:
def plot_angular_velocity(l1, l2, m1, m2, g, C1_gen, C2_gen, C1_std, C2_std, save_plot=False):
    """Plots the angular velocity for both general and standard cases.
    """
    t = np.linspace(0, 10, 500) # Time from 0 to 10 seconds

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

    # --- General Case Plot (Angular Velocity) ---
    (A_gen_plus, omega_gen_plus), (A_gen_minus, omega_gen_minus) = calculate_general_params(l1, l2, m1, m2, g)

    # Total angular displacements
    phi_gen1_t = C1_gen * A_gen_plus * np.exp(1j * omega_gen_plus * t) + \
                 C2_gen * A_gen_minus * np.exp(1j * omega_gen_minus * t)
    phi_gen2_t = C1_gen * np.exp(1j * omega_gen_plus * t) + \
                 C2_gen * np.exp(1j * omega_gen_minus * t)

    # Total angular velocities (derivative of total displacements)
    phi_dot_gen1_t = 1j * omega_gen_plus * C1_gen * A_gen_plus * np.exp(1j * omega_gen_plus * t) + \
                     1j * omega_gen_minus * C2_gen * A_gen_minus * np.exp(1j * omega_gen_minus * t)
    phi_dot_gen2_t = 1j * omega_gen_plus * C1_gen * np.exp(1j * omega_gen_plus * t) + \
                     1j * omega_gen_minus * C2_gen * np.exp(1j * omega_gen_minus * t)

    axes[0].plot(t, np.real(phi_dot_gen1_t), label=r'General Mode 1 ($\dot{\phi}_1$)', color='blue')
    axes[0].plot(t, np.real(phi_dot_gen2_t), label=r'General Mode 2 ($\dot{\phi}_2$)', color='red')
    axes[0].set_title(f'General Linearized Double Pendulum Angular Velocity (l1={l1}, l2={l2}, m1={m1}, m2={m2}, g={g:.2f})')
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Angular Velocity (rad/s)')
    axes[0].legend()
    axes[0].grid(True)

    # --- Standard Case Plot (Angular Velocity) ---
    (A_std_plus, omega_std_plus), (A_std_minus, omega_std_minus) = calculate_standard_params(g, l_val=DEFAULT_L_STANDARD)

    # Total angular displacements
    phi_std1_t = C1_std * A_std_plus * np.exp(1j * omega_std_plus * t) + \
                 C2_std * A_std_minus * np.exp(1j * omega_std_minus * t)
    phi_std2_t = C1_std * np.exp(1j * omega_std_plus * t) + \
                 C2_std * np.exp(1j * omega_std_minus * t)

    # Total angular velocities (derivative of total displacements)
    phi_dot_std1_t = 1j * omega_std_plus * C1_std * A_std_plus * np.exp(1j * omega_std_plus * t) + \
                     1j * omega_std_minus * C2_std * A_std_minus * np.exp(1j * omega_std_minus * t)
    phi_dot_std2_t = 1j * omega_std_plus * C1_std * np.exp(1j * omega_std_plus * t) + \
                     1j * omega_std_minus * C2_std * np.exp(1j * omega_std_minus * t)

    axes[1].plot(t, np.real(phi_dot_std1_t), label=r'Standard Mode 1 ($\dot{\phi}_1$)', color='green')
    axes[1].plot(t, np.real(phi_dot_std2_t), label=r'Standard Mode 2 ($\dot{\phi}_2$)', color='purple')
    axes[1].set_title(f'Standard Linearized Double Pendulum Angular Velocity (l=1, m=1, g={g:.2f})')
    axes[1].set_xlabel('Time (s)')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    if save_plot:
        plt.savefig(f'angular_velocity_l1_{l1}_l2_{l2}_m1_{m1}_m2_{m2}_g_{g:.2f}.png')
    plt.show()

In [ ]:
# Create an interactive widget for angular velocity
interactive_angular_velocity_plot = widgets.interactive(
    plot_angular_velocity,
    l1=l1_slider,
    l2=l2_slider,
    m1=m1_slider,
    m2=m2_slider,
    g=g_slider,
    C1_gen=C1_gen_slider,
    C2_gen=C2_gen_slider,
    C1_std=C1_std_slider,
    C2_std=C2_std_slider,
    save_plot=save_plot_checkbox
)

# Display the interactive plot
display(interactive_angular_velocity_plot)

In [ ]:
def calculate_total_energy(l1, l2, m1, m2, g, phi1, phi_dot1, phi2, phi_dot2):
    """Calculates the total energy (Kinetic + Potential) for a linearized double pendulum.
    phi1, phi_dot1, phi2, phi_dot2 should be arrays of real values over time.
    """
    # Kinetic Energy (T)
    T = 0.5 * (m1 + m2) * l1**2 * phi_dot1**2 + \
        0.5 * m2 * l2**2 * phi_dot2**2 + \
        m2 * l1 * l2 * phi_dot1 * phi_dot2

    # Potential Energy (V)
    V = 0.5 * (m1 + m2) * g * l1 * phi1**2 + \
        0.5 * m2 * g * l2 * phi2**2

    return T + V

In [ ]:
def plot_total_energy(l1, l2, m1, m2, g, C1_gen, C2_gen, C1_std, C2_std, save_plot=False):
    """Plots the total energy of the system over time for both general and standard cases."""
    t = np.linspace(0, 10, 500) # Time from 0 to 10 seconds

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

    # --- General Case Energy Calculation ---
    (A_gen_plus, omega_gen_plus), (A_gen_minus, omega_gen_minus) = calculate_general_params(l1, l2, m1, m2, g)

    # Total angular displacements
    phi_gen1_t = C1_gen * A_gen_plus * np.exp(1j * omega_gen_plus * t) + \
                 C2_gen * A_gen_minus * np.exp(1j * omega_gen_minus * t)
    phi_gen2_t = C1_gen * np.exp(1j * omega_gen_plus * t) + \
                 C2_gen * np.exp(1j * omega_gen_minus * t)

    # Total angular velocities
    phi_dot_gen1_t = 1j * omega_gen_plus * C1_gen * A_gen_plus * np.exp(1j * omega_gen_plus * t) + \
                     1j * omega_gen_minus * C2_gen * A_gen_minus * np.exp(1j * omega_gen_minus * t)
    phi_dot_gen2_t = 1j * omega_gen_plus * C1_gen * np.exp(1j * omega_gen_plus * t) + \
                     1j * omega_gen_minus * C2_gen * np.exp(1j * omega_gen_minus * t)

    # Calculate total energy using the real parts of phi and phi_dot
    total_energy_gen = calculate_total_energy(l1, l2, m1, m2, g,
                                              np.real(phi_gen1_t), np.real(phi_dot_gen1_t),
                                              np.real(phi_gen2_t), np.real(phi_dot_gen2_t))

    axes[0].plot(t, total_energy_gen, label='General Total Energy', color='blue')
    axes[0].set_title(f'General Linearized: Total Energy vs Time\n(l1={l1}, l2={l2}, m1={m1}, m2={m2}, g={g:.2f})')
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Total Energy (Joules)')
    axes[0].legend()
    axes[0].grid(True)

    # --- Standard Case Energy Calculation ---
    (A_std_plus, omega_std_plus), (A_std_minus, omega_std_minus) = calculate_standard_params(g, l_val=DEFAULT_L_STANDARD)

    # Total angular displacements
    phi_std1_t = C1_std * A_std_plus * np.exp(1j * omega_std_plus * t) + \
                 C2_std * A_std_minus * np.exp(1j * omega_std_minus * t)
    phi_std2_t = C1_std * np.exp(1j * omega_std_plus * t) + \
                 C2_std * np.exp(1j * omega_std_minus * t)

    # Total angular velocities
    phi_dot_std1_t = 1j * omega_std_plus * C1_std * A_std_plus * np.exp(1j * omega_std_plus * t) + \
                     1j * omega_std_minus * C2_std * A_std_minus * np.exp(1j * omega_std_minus * t)
    phi_dot_std2_t = 1j * omega_std_plus * C1_std * np.exp(1j * omega_std_plus * t) + \
                     1j * omega_std_minus * C2_std * np.exp(1j * omega_std_minus * t)

    # Calculate total energy using the real parts of phi and phi_dot
    total_energy_std = calculate_total_energy(DEFAULT_L_STANDARD, DEFAULT_L_STANDARD, 1.0, 1.0, g,
                                              np.real(phi_std1_t), np.real(phi_dot_std1_t),
                                              np.real(phi_std2_t), np.real(phi_dot_std2_t))

    axes[1].plot(t, total_energy_std, label='Standard Total Energy', color='green')
    axes[1].set_title(f'Standard Linearized: Total Energy vs Time\n(l=1, m=1, g={g:.2f})')
    axes[1].set_xlabel('Time (s)')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    if save_plot:
        plt.savefig(f'total_energy_l1_{l1}_l2_{l2}_m1_{m1}_m2_{m2}_g_{g:.2f}.png')
    plt.show()

In [ ]:
# Create an interactive widget for the total energy plot
interactive_total_energy_plot = widgets.interactive(
    plot_total_energy,
    l1=l1_slider,
    l2=l2_slider,
    m1=m1_slider,
    m2=m2_slider,
    g=g_slider,
    C1_gen=C1_gen_slider,
    C2_gen=C2_gen_slider,
    C1_std=C1_std_slider,
    C2_std=C2_std_slider,
    save_plot=save_plot_checkbox
)

# Display the interactive plot
display(interactive_total_energy_plot)

## Explanation of New Code Blocks: Analytical Solutions for Linearized Double Pendulum

This section introduces and visualizes the analytical solutions for the linearized double pendulum, extending the analysis beyond numerical simulations to provide deeper insight into its normal modes.

### Core Functions:

1.  **`calculate_general_params(l1, l2, m1, m2, g)`**:
    *   **Purpose**: This function computes the angular frequencies ($\omega_{\pm}$) and amplitude ratios ($A_{\pm}$, sometimes denoted as $r_{\pm}$) for the two normal modes of the *general* linearized double pendulum (where $m_1 \ne m_2$ and $l_1 \ne l_2$). These are derived from the characteristic equation of the system's equations of motion.
    *   **Formulas**: The calculations involve solving a quadratic equation (implicitly) for $\omega^2$, leading to two distinct frequencies ($\omega_{plus}, \omega_{minus}$) and their corresponding amplitude ratios ($A_{plus}, A_{minus}$), which describe the relationship between $\phi_1$ and $\phi_2$ in each mode.

2.  **`calculate_standard_params(g, l_val)`**:
    *   **Purpose**: A specialized version of the above, tailored for the *standard* linearized double pendulum where $m_1=m_2=m$ and $l_1=l_2=l$. The formulas for this case are simpler and often presented in textbooks.
    *   **Formulas**: As stated in a previous markdown cell, the frequencies are $\omega_1^2 = \frac{g}{l}(2+\sqrt{2})$ and $\omega_2^2 = \frac{g}{l}(2-\sqrt{2})$, and the amplitude ratios (eigenvector components for $\phi_1/\phi_2$) are $r_1 = \frac{\sqrt{2}}{2}$ and $r_2 = -\frac{\sqrt{2}}{2}$.

### Plotting and Interactive Functions:

These functions utilize the parameters calculated above to plot the analytical time evolution and phase space trajectories, allowing for interactive exploration of the normal modes.

1.  **`plot_pendulum(l1, l2, m1, m2, g, C1_gen, C2_gen, C1_std, C2_std, save_plot)`**:
    *   **Purpose**: Plots the angular displacement ($\phi_1(t)$ and $\phi_2(t)$) over time for both the general and standard linearized cases.
    *   **Analytical Solutions**: The general solution for $\phi_k(t)$ (where $k=1,2$) is a superposition of the two normal modes:
        $$\phi_1(t) = C_1 A_{plus} e^{i\omega_{plus} t} + C_2 A_{minus} e^{i\omega_{minus} t}$$
        $$\phi_2(t) = C_1 e^{i\omega_{plus} t} + C_2 e^{i\omega_{minus} t}$$
        (Note: $C_1, C_2$ are complex constants determined by initial conditions, and we plot their real parts.)

2.  **`plot_phi1_vs_phi2(...)`**:
    *   **Purpose**: Visualizes the phase space by plotting $\text{Re}(\phi_1(t))$ against $\text{Re}(\phi_2(t))$. For a linearized system, these typically show closed, regular orbits (e.g., Lissajous figures).

3.  **`plot_angular_velocity(...)`**:
    *   **Purpose**: Plots the angular velocities ($\dot{\phi}_1(t)$ and $\dot{\phi}_2(t)$) over time.
    *   **Analytical Solutions**: The angular velocities are obtained by differentiating the angular displacement solutions with respect to time:
        $$\dot{\phi}_1(t) = i\omega_{plus} C_1 A_{plus} e^{i\omega_{plus} t} + i\omega_{minus} C_2 A_{minus} e^{i\omega_{minus} t}$$
        $$\dot{\phi}_2(t) = i\omega_{plus} C_1 e^{i\omega_{plus} t} + i\omega_{minus} C_2 e^{i\omega_{minus} t}$$

4.  **`calculate_total_energy(...)`** and **`plot_total_energy(...)`**:
    *   **Purpose**: These functions calculate and plot the total mechanical energy (Kinetic + Potential) of the linearized system over time. For a correctly linearized and simulated system, the total energy should be conserved (i.e., remain constant over time).
    *   **Energy Formulas**: For the linearized double pendulum, the kinetic energy ($T$) and potential energy ($V$) are given by:
        $$T = \frac{1}{2}(m_1+m_2)l_1^2 \dot{\phi}_1^2 + \frac{1}{2}m_2 l_2^2 \dot{\phi}_2^2 + m_2 l_1 l_2 \dot{\phi}_1 \dot{\phi}_2$$
        $$V = \frac{1}{2}(m_1+m_2)gl_1 \phi_1^2 + \frac{1}{2}m_2 gl_2 \phi_2^2$$
        The total energy is $E = T + V$.

These interactive plots allow you to visually confirm the oscillatory nature of the linearized system and its energy conservation, in contrast to the chaotic behavior of the non-linear system previously studied.

## Essential/Interesting Addition: Connecting Analytical Constants to Initial Conditions

Currently, the interactive plots for the analytical solutions (`plot_pendulum`, `plot_phi1_vs_phi2`, `plot_angular_velocity`, `plot_total_energy`) use arbitrary complex constants `C1_gen`, `C2_gen`, `C1_std`, `C2_std` as slider inputs. While useful for exploring the influence of normal mode amplitudes, it makes it less intuitive to connect these plots to specific physical initial conditions (e.g., starting angles and velocities).

**Suggestion:** Implement a helper function that takes physical initial conditions ($\phi_1(0)$, $\phi_2(0)$, $\dot{\phi}_1(0)$, $\dot{\phi}_2(0)$) and automatically calculates the corresponding complex constants $C_1$ and $C_2$. This would make the interactive analytical plots directly comparable to the numerical simulations for the linearized system, and allow users to set physically meaningful starting points rather than abstract complex amplitudes. This would significantly enhance the educational value and practical applicability of these analytical visualizations.

## Problem 2: Pendulum with Movable Support

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from IPython.display import display, HTML
import matplotlib.animation as animation

# --- 2.1 Definition of Linearized System Dynamics --- Pendulum with movable support
# State: [x, theta, dot_x, dot_theta]
# Where:
#   x: horizontal position of the support
#   theta: pendulum angle with the vertical (small)
#   dot_x: support velocity
#   dot_theta: pendulum angular velocity
def system_dynamics_movable_support_linearized(t, state, m1, m2, l, g):
    x, theta, dot_x, dot_theta = state

    # Linearized equations of motion (derived in context):
    # (m1 + m2) * ddot_x + m2 * l * ddot_theta = 0
    # m2 * ddot_x + m2 * l * ddot_theta + m2 * g * theta = 0

    # Solving for ddot_x and ddot_theta:
    # ddot_theta = - (g / l) * ((m1 + m2) / m1) * theta
    # ddot_x = (m2 * g / m1) * theta

    ddot_theta = - (g / l) * ((m1 + m2) / m1) * theta
    ddot_x = (m2 * g / m1) * theta

    return [dot_x, dot_theta, ddot_x, ddot_theta]

print("System dynamics for pendulum with movable support (linearized) defined.")

### 2.2 Mass Sweep and Oscillation Frequency

We analyze how the oscillation frequency of the pendulum with a movable support changes as a function of the support's mass ($m_1$). For this linearized system, there is an oscillation mode with frequency $\omega_2 = \sqrt{\frac{g}{l} \frac{m_1+m_2}{m_1}}$, and a translational mode with zero frequency ($\omega_1 = 0$). We will plot $\omega_2$ for different values of $m_1$.

In [ ]:
# Base parameters
m2_const_mov = 1.0 # kg
l_const_mov = 1.0  # m
g_const_mov = 9.81 # m/s^2

# Range of m1 masses for the sweep
m1_values = np.logspace(-3, 3, 100) # m1 from 0.001 to 1000 kg

# Calculate omega_2 for each m1
omega2_values = np.sqrt((g_const_mov / l_const_mov) * ((m1_values + m2_const_mov) / m1_values))

plt.figure(figsize=(10, 6))
plt.plot(m1_values, omega2_values, color='purple')
plt.xscale('log')
plt.xlabel(r'$m_1$ (support mass, kg)')
plt.ylabel(r'$\%omega_2$ (angular oscillation frequency, rad/s)')
plt.title('Oscillation Mode Frequency vs. Support Mass ($m_1$)')
plt.grid(True, which="both", ls="--")
plt.show()

print("Mass sweep analysis completed. It shows how the frequency changes non-linearly with the support's mass, being high for small m1 and approaching the simple pendulum frequency (sqrt(g/l)) for very large m1.")

### 2.3 Phase Plot: Support Position vs. Pendulum Angle ($x(t)$ vs $\theta(t)$)

This phase plot is crucial for understanding the interaction between the support's movement and the pendulum's oscillation. Here, we can observe the conservation of the center of mass: when the pendulum moves to the right, the support shifts to the left to maintain the system's center of mass. This dynamic is fundamental in the design of shock absorbers and control systems.

In [ ]:
# Parameters for the simulation
m1_phase = 1000.0
m2_phase = 1.0
l_phase = 1.0
g_phase = 9.81

# Initial conditions to excite oscillation:
# [x_0, theta_0, dot_x_0, dot_theta_0]
# Start with the pendulum slightly displaced and the support at rest.
x0_phase = 0.0
theta0_phase = np.deg2rad(10) # 10 degrees
dot_x0_phase = 0.0
dot_theta0_phase = 0.0
y0_phase = [x0_phase, theta0_phase, dot_x0_phase, dot_theta0_phase]

T_sim_phase = 50 # Longer simulation time to observe the pattern
t_eval_phase = np.linspace(0, T_sim_phase, 1000)

# Solve the equations of motion
sol_phase = solve_ivp(system_dynamics_movable_support_linearized, (0, T_sim_phase), y0_phase,
                        method='DOP853', t_eval=t_eval_phase, atol=1e-12, rtol=1e-12,
                        args=(m1_phase, m2_phase, l_phase, g_phase))

# Extract results
x_sol_phase = sol_phase.y[0]
theta_sol_phase = sol_phase.y[1]

plt.figure(figsize=(8, 6))
plt.plot(theta_sol_phase, x_sol_phase, color='darkred', linewidth=0.8)
plt.xlabel(r'$\%theta$ (pendulum angle, rad)')
plt.ylabel(r'$x$ (support position, m)')
plt.title(r'Phase Plot: $x(t)$ vs $\%theta(t)$ for Pendulum with Movable Support')
plt.grid(True)
plt.axvline(0, color='gray', linestyle='--')
plt.axhline(0, color='gray', linestyle='--')
plt.show()

print("Phase plot x(t) vs theta(t) generated. Observe the closed orbits, typical of a conservative linear system.")

### 2.4 Normal Modes Animation

We will animate the two normal modes of this linearized system for better visualization:

1.  **Mode 1 (Pure Translation, $\omega_1 = 0$):** The pendulum remains vertical ($\theta=0$) while the support moves at a constant velocity. Physically, this means the system's center of mass moves without any relative oscillation between the pendulum and the support.
2.  **Mode 2 (Oscillation with Recoil, $\omega_2 = \sqrt{\frac{g}{l} \frac{m_1+m_2}{m_1}}$):** The pendulum oscillates, and the support moves in the opposite direction to keep the center of mass vertically above a fixed point. The amplitude ratio between $x$ and $\theta$ is $A/B = - \frac{m_2 l}{m_1+m_2}$. It is "very satisfying" to see the large displacement of the support when $m_1 \ll m_2$.

In [ ]:
# System parameters for animation
m1_anim = 0.1 # Support mass (small to emphasize Mode 2)
m2_anim = 1.0 # Pendulum mass
l_anim = 1.0  # Pendulum length
g_anim = 9.81 # Gravity

T_anim = 20 # Animation duration
dt_anim_frames = 0.05 # Time step for frames
t_eval_anim = np.linspace(0, T_anim, int(T_anim / dt_anim_frames))

# --- Normal Mode 1: Pure Translation (omega = 0) ---
# Initial conditions: pendulum is vertical and at rest relative to the support,
# but the support has an initial velocity.
y0_mode1 = [0.0, 0.0, 0.5, 0.0] # [x_0, theta_0, dot_x_0, dot_theta_0]
sol_mode1 = solve_ivp(system_dynamics_movable_support_linearized, (0, T_anim), y0_mode1,
                        method='DOP853', t_eval=t_eval_anim, atol=1e-12, rtol=1e-12,
                        args=(m1_anim, m2_anim, l_anim, g_anim))

# --- Normal Mode 2: Oscillation with Recoil ---
# Initial conditions to excite only Mode 2 (approximate):
# The x/theta amplitude ratio should be A/B = - (m2*l / (m1+m2))
# Start with the pendulum at a small angle and the support at a corresponding position
# so that dot_x and dot_theta are zero.
theta0_mode2 = np.deg2rad(10) # Initial angle
x0_mode2 = - (m2_anim * l_anim / (m1_anim + m2_anim)) * theta0_mode2
y0_mode2 = [x0_mode2, theta0_mode2, 0.0, 0.0] # [x_0, theta_0, dot_x_0, dot_theta_0]
sol_mode2 = solve_ivp(system_dynamics_movable_support_linearized, (0, T_anim), y0_mode2,
                        method='DOP853', t_eval=t_eval_anim, atol=1e-12, rtol=1e-12,
                        args=(m1_anim, m2_anim, l_anim, g_anim))

# --- Animation ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# Axis configuration
max_x = max(np.max(np.abs(sol_mode1.y[0])), np.max(np.abs(sol_mode2.y[0] + l_anim * np.sin(sol_mode2.y[1])))) + 0.5
max_y = l_anim + 0.5

for ax in [ax1, ax2]:
    ax.set_xlim(-max_x, max_x)
    ax.set_ylim(-(l_anim + 0.2), 0.2)
    ax.set_aspect('equal')
    ax.grid(True)

ax1.set_title('Normal Mode 1 (Pure Translation)')
ax2.set_title('Normal Mode 2 (Oscillation with Recoil)')

# Animation elements for Mode 1
line_mode1, = ax1.plot([], [], 'o-', lw=2, color='blue', label='Pendulum')
cart_mode1, = ax1.plot([], [], 's', markersize=15, color='gray', label='Support')

# Animation elements for Mode 2
line_mode2, = ax2.plot([], [], 'o-', lw=2, color='red', label='Pendulum')
cart_mode2, = ax2.plot([], [], 's', markersize=15, color='gray', label='Support')

# Initialization function for the animation
def init():
    line_mode1.set_data([], [])
    cart_mode1.set_data([], [])
    line_mode2.set_data([], [])
    cart_mode2.set_data([], [])
    return line_mode1, cart_mode1, line_mode2, cart_mode2

# Animation function
def animate(i):
    # Mode 1
    x_cart1 = sol_mode1.y[0, i]
    theta1 = sol_mode1.y[1, i]
    x_pend1 = x_cart1 + l_anim * np.sin(theta1)
    y_pend1 = -l_anim * np.cos(theta1)
    line_mode1.set_data([x_cart1, x_pend1], [0, y_pend1])
    cart_mode1.set_data([x_cart1], [0])

    # Mode 2
    x_cart2 = sol_mode2.y[0, i]
    theta2 = sol_mode2.y[1, i]
    x_pend2 = x_cart2 + l_anim * np.sin(theta2)
    y_pend2 = y_pend1 # Pivot is fixed at y=0, so y_pend2 = -l_anim * np.cos(theta2)
    line_mode2.set_data([x_cart2, x_pend2], [0, y_pend2])
    cart_mode2.set_data([x_cart2], [0])

    return line_mode1, cart_mode1, line_mode2, cart_mode2

print("Generating normal modes animation...")
an = animation.FuncAnimation(fig, animate, frames=len(t_eval_anim), interval=dt_anim_frames*1000, blit=True)
plt.tight_layout()
plt.close(fig) # Prevent matplotlib from showing the static figure
display(HTML(an.to_jshtml()))

print("Normal modes animation generated.")

# Save the animation to a file and provide a download link
animation_filename = 'movable_support_normal_modes.mp4'
print(f"Saving animation as '{animation_filename}'...")
an.save(animation_filename, writer='ffmpeg', fps=30)
print("Animation saved.")
display(FileLink(animation_filename))

### 2.5 Non-Linear Pendulum with Movable Support: Hamiltonian Dynamics

Now, we transition to the non-linear regime for the pendulum with a movable support. The system's dynamics are governed by Hamiltonian equations derived from the following Hamiltonian function. This non-linear treatment allows for larger oscillations and a more complete understanding of the system's behavior, where the approximations made for linearization no longer hold.

In [ ]:
def hamilton_equations_movable_support(t, state, m1, m2, l, g):
    x, theta, p_x, p_theta = state

    cos_t = np.cos(theta)
    sin_t = np.sin(theta)

    # 1. Determinante y Denominador
    det_M = m2 * l**2 * (m1 + m2 * sin_t**2)

    if np.abs(det_M) < 1e-12:
        return np.array([0, 0, 0, 0]) * np.nan

    # 2. Velocidades canónicas (q_dot = dH/dp)
    # Derivadas directas de H = 1/2 * p^T * M^-1 * p
    dot_x = (m2 * l**2 * p_x - m2 * l * cos_t * p_theta) / det_M
    dot_theta = ((m1 + m2) * p_theta - m2 * l * cos_t * p_x) / det_M

    # 3. Derivadas de los momentos (p_dot = -dH/dq)
    dot_p_x = 0.0  # x es cíclica

    # Derivada de T respecto a theta (dH/dtheta)
    # Usamos la forma: H_kin = (A + B + C) / (2 * det_M)
    term_A = m2 * l**2 * p_x**2
    term_B = (m1 + m2) * p_theta**2
    term_C = -2 * m2 * l * cos_t * p_x * p_theta

    numerator = term_A + term_B + term_C

    # d(numerator)/dtheta
    d_num_d_theta = 2 * m2 * l * sin_t * p_x * p_theta
    # d(det_M)/dtheta
    d_det_d_theta = 2 * m2**2 * l**2 * sin_t * cos_t

    # Regla del cociente para dT/dtheta
    dT_d_theta = (d_num_d_theta * (2 * det_M) - numerator * (2 * d_det_d_theta)) / (2 * det_M)**2

    # dV/dtheta (V = -m2 * g * l * cos_t)
    dV_d_theta = m2 * g * l * sin_t

    dot_p_theta = -(dT_d_theta + dV_d_theta)

    return np.array([dot_x, dot_theta, dot_p_x, dot_p_theta])


In [ ]:
def hamiltonian_movable_support(state, m1, m2, l, g):
    x, theta, p_x, p_theta = state

    cos_t = np.cos(theta)
    sin_t = np.sin(theta)

    # El determinante de la matriz de masa es clave
    det_M = m2 * l**2 * (m1 + m2 * sin_t**2)

    if np.abs(det_M) < 1e-12:
        return np.nan

    # La energía cinética es 1/2 * p^T * M^-1 * p
    # Expandiendo p^T * adj(M) * p:
    term_px = m2 * l**2 * p_x**2
    term_ptheta = (m1 + m2) * p_theta**2
    term_coupling = -2 * m2 * l * cos_t * p_x * p_theta

    kinetic_energy = (term_px + term_ptheta + term_coupling) / (2 * det_M)

    # Energía Potencial (referencia en el eje x)
    potential_energy = -m2 * g * l * cos_t

    return kinetic_energy + potential_energy


### 2.6 Kinetic Energy Transfer Analysis

This analysis visualizes the transfer of kinetic energy between the movable support and the pendulum bob in the non-linear regime. For a conservative system, the total energy (Hamiltonian) should remain constant. This plot specifically highlights how the kinetic energy components of the support ($KE_x$) and the pendulum ($KE_{\theta}$) fluctuate over time, while their sum (total kinetic energy) remains conserved (assuming total mechanical energy is conserved and potential energy changes are accounted for, this shows the dynamic exchange of kinetic energy).

In [ ]:
import matplotlib.pyplot as plt

# --- Simulation Parameters ---
m1_val = 1.0 # Mass of support
m2_val = 1.0 # Mass of pendulum
l_val = 1.0  # Length of pendulum
g_val = 9.81 # Gravity

# Initial conditions for the non-linear system:
# [x_0, theta_0, p_x_0, p_theta_0]
# Start with pendulum displaced at a larger angle to show non-linear effects
# and zero initial velocities, so momenta should be calculated carefully if needed.
# For now, let's set initial momenta to zero, and let the system evolve from angle.
x0 = 0.0
theta0 = np.deg2rad(90) # 60 degrees (large angle)
p_x0 = 0.0
p_theta0 = 0.0
y0_non_linear = [x0, theta0, p_x0, p_theta0]

T_sim = 100 # Total simulation time
t_eval = np.linspace(0, T_sim, 5000) # Time points for evaluation

print("Simulating non-linear pendulum with movable support...")

sol_non_linear = solve_ivp(hamilton_equations_movable_support, (0, T_sim), y0_non_linear,
                            method='DOP853', t_eval=t_eval, atol=1e-12, rtol=1e-12,
                            args=(m1_val, m2_val, l_val, g_val))

# Extract solutions
x_sol = sol_non_linear.y[0]
theta_sol = sol_non_linear.y[1]
p_x_sol = sol_non_linear.y[2]
p_theta_sol = sol_non_linear.y[3]

# Calculate velocities from momenta using the definitions:
# dot_x = dH/dp_x
# dot_theta = dH/dp_theta
dot_x_sol = []
dot_theta_sol = []

for i in range(len(t_eval)):
    current_state = sol_non_linear.y[:, i]
    _, current_theta, current_px, current_ptheta = current_state
    cos_theta = np.cos(current_theta)
    sin_theta = np.sin(current_theta)

    denominator_common = (m1_val + m2_val * sin_theta**2)
    if np.abs(denominator_common) < 1e-12:
        dot_x_sol.append(np.nan)
        dot_theta_sol.append(np.nan)
    else:
        dx = (m2_val * l_val**2 * current_px - m2_val * l_val * cos_theta * current_ptheta) / (m2_val * l_val**2 * denominator_common)
        dt = ((m1_val + m2_val) * current_ptheta - m2_val * l_val * cos_theta * current_px) / (m2_val * l_val**2 * denominator_common)
        dot_x_sol.append(dx)
        dot_theta_sol.append(dt)

dot_x_sol = np.array(dot_x_sol)
dot_theta_sol = np.array(dot_theta_sol)

# Calculate kinetic energies
KE_support = 0.5 * m1_val * dot_x_sol**2
KE_pendulum = 0.5 * m2_val * (dot_x_sol**2 + l_val**2 * dot_theta_sol**2 + 2 * l_val * dot_x_sol * dot_theta_sol * np.cos(theta_sol))
KE_total = KE_support + KE_pendulum
PE_total = -m2_val * g_val * l_val * np.cos(theta_sol)
Total_Energy = KE_total + PE_total



# Plot Kinetic Energy Transfer
plt.figure(figsize=(12, 6))
plt.plot(t_eval, Total_Energy, label='Hamiltonian (H = T + V)', color='black', lw=2)
plt.plot(t_eval, KE_support, label='Kinetic Energy of Support ($KE_x$)', color='blue')
plt.plot(t_eval, KE_pendulum, label='Kinetic Energy of Pendulum ($KE_\theta$)', color='red')
plt.plot(t_eval, KE_total, label='Total Kinetic Energy ($KE_{total}$)', color='green', linestyle='--')
plt.title('Kinetic Energy Transfer in Non-Linear Movable Support Pendulum')
plt.xlabel('Time (s)')
plt.xlim(0, 10)
plt.ylabel('Kinetic Energy')
plt.grid(True)
plt.legend()
plt.show()

print("Kinetic energy transfer plot generated.")

### 2.7 Phase Space Plot for the Non-Linear Pendulum

In contrast to chaotic systems, integrable or quasi-periodic systems often exhibit regular, well-defined structures in their phase space. For the non-linear pendulum with a movable support, a plot of the pendulum's angle ($\theta$) versus its conjugate momentum ($p_\theta$) can reveal such ordered trajectories, often forming 'tori' (donut-like shapes) if the system is quasi-periodic, indicating ordered motion within the phase space.

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(theta_sol, p_theta_sol, color='darkgreen', linewidth=0.5, alpha=0.7)
plt.title(r'Phase Space: $\theta$ vs $p_\theta$ (Non-Linear Movable Support Pendulum)')
plt.xlabel(r'$\%theta$ (rad)')
plt.ylabel(r'$p_\theta$')
plt.grid(True)
plt.show()

print("Phase space plot for non-linear pendulum generated.")

### 2.8 Phase Space Plot for the Non-Linear Pendulum: $x$ vs $p_x$

This plot visualizes the trajectory of the support in its phase space. For the system with a cyclic coordinate $x$, its conjugate momentum $p_x$ is conserved. Therefore, we expect to see a single point or a very narrow band if $p_x$ is constant, or a broader region if $p_x$ varies slightly due to numerical integration errors.

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(x_sol, p_x_sol, color='darkorange', linewidth=0.5, alpha=0.7)
plt.title(r'Phase Space: $x$ vs $p_x$ (Non-Linear Movable Support Pendulum)')
plt.xlabel(r'$x$ (m)')
plt.ylabel(r'$p_x$')
plt.grid(True)
plt.show()

print("Phase space plot for non-linear pendulum (x vs px) generated.")

### 2.9 Time Evolution Plot: $F_x(t)$ vs $t$ (Non-Linear System)

This plot displays the horizontal force ($F_x$) exerted on the movable support as a function of time. It directly shows the dynamic forces acting on the support, revealing their oscillatory nature over time as the pendulum swings.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters and solutions (m1_val, m2_val, l_val, g_val, x_sol, theta_sol, dot_theta_sol, t_eval)
# are assumed to be available from previous cells (e.g., f5ca4bf5).

# Calculate ddot_x_sol (acceleration of the support)
ddot_x_sol = np.zeros_like(theta_sol, dtype=float)
for i in range(len(theta_sol)):
    current_theta = theta_sol[i]
    current_dot_theta = dot_theta_sol[i]
    sin_theta = np.sin(current_theta)
    cos_theta = np.cos(current_theta)

    numerator = m2_val * sin_theta * (g_val * cos_theta + l_val * current_dot_theta**2)
    denominator = m1_val + m2_val * sin_theta**2

    if np.abs(denominator) < 1e-12: # Avoid division by zero
        ddot_x_sol[i] = np.nan
    else:
        ddot_x_sol[i] = numerator / denominator

# Calculate Fx_sol = m1 * ddot_x
Fx_sol = m1_val * ddot_x_sol

print("ddot_x_sol and Fx_sol calculated successfully.")

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(t_eval, Fx_sol, color='darkviolet', linewidth=0.7, alpha=0.8)
plt.title(r'Time Evolution: $F_x(t)$ vs $t$ (Non-Linear Movable Support Pendulum)')
plt.xlabel('Time (s)')
plt.ylabel(r'$F_x$ (N)')
plt.xlim(0, 10)
plt.grid(True)
plt.show()

print("Time evolution plot of Fx vs t generated.")

### 2.10 Phase Space Plot: $x$ vs $\theta$ (Non-Linear System)

This plot visualizes the trajectory of the support's position ($x$) against the pendulum's angle ($\theta$) in the non-linear system. This provides a direct view of the coupled motion between the support and the pendulum, showcasing the system's overall configuration changes during its evolution.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(x_sol, theta_sol, color='magenta', linewidth=0.7, alpha=0.8)
plt.title(r'Phase Space: $x$ vs $\theta$ (Non-Linear Movable Support Pendulum)')
plt.xlabel(r'$x$ (m)')
plt.ylabel(r'$\theta$ (rad)')
plt.grid(True)
plt.show()

print("Phase space plot of x vs theta generated.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import matplotlib.animation as animation
from IPython.display import HTML, display, FileLink

# --- Function Definitions (copied for self-contained animation) ---

def system_dynamics_movable_support_linearized(t, state, m1, m2, l, g):
    x, theta, dot_x, dot_theta = state
    ddot_theta = - (g / l) * ((m1 + m2) / m1) * theta
    ddot_x = (m2 * g / m1) * theta
    return [dot_x, dot_theta, ddot_x, ddot_theta]

def hamilton_equations_movable_support(t, state, m1, m2, l, g):
    # This function implements the Hamiltonian equations of motion for the non-linear
    # pendulum with a movable support.
    # For minimizing energy errors, ensuring the mathematical derivation is correct
    # and that numerical calculations are robust is crucial.
    x, theta, p_x, p_theta = state

    cos_t = np.cos(theta)
    sin_t = np.sin(theta)

    # 1. Determinant of the mass matrix in the denominator of velocities and kinetic energy
    # This is D_vel = m2 * l**2 * (m1 + m2 * sin_t**2) from derivation.
    det_M_actual = m2 * l**2 * (m1 + m2 * sin_t**2)

    # Handle cases where the denominator becomes very small to prevent numerical instability.
    if np.abs(det_M_actual) < 1e-12:
        return np.array([0, 0, 0, 0]) * np.nan

    # 2. Canonical velocities (q_dot = dH/dp)
    # These are derived from the partial derivatives of the Hamiltonian with respect to momenta.
    dot_x = (m2 * l**2 * p_x - m2 * l * cos_t * p_theta) / det_M_actual
    dot_theta = ((m1 + m2) * p_theta - m2 * l * cos_t * p_x) / det_M_actual

    # 3. Derivatives of momenta (p_dot = -dH/dq)
    # p_dot_x = -dH/dx. Since x is a cyclic coordinate (Hamiltonian does not explicitly depend on x),
    # its conjugate momentum p_x is conserved (dot_p_x = 0).
    dot_p_x = 0.0

    # p_dot_theta = -dH/dtheta. This requires calculating dH/dtheta = dK/dtheta + dV/dtheta.

    # Numerator of the kinetic energy term in the Hamiltonian
    K_num = (m2 * l**2 * p_x**2 + (m1 + m2) * p_theta**2 - 2 * m2 * l * cos_t * p_x * p_theta)

    # Denominator of the kinetic energy term in the Hamiltonian
    # This is K_den = 2 * m2 * l**2 * (m1 + m2 * sin_t**2) = 2 * det_M_actual
    K_den = 2 * det_M_actual

    # Calculate derivatives of K_num and K_den with respect to theta
    dK_num_d_theta = 2 * m2 * l * sin_t * p_x * p_theta
    dK_den_d_theta = 2 * (2 * m2**2 * l**2 * sin_t * cos_t) # Derivative of (2 * m2 * l**2 * (m1 + m2 * sin_t**2))

    # Apply the quotient rule to find dK/dtheta = d(K_num / K_den) / dtheta
    if np.abs(K_den**2) < 1e-12: # Check for near-zero denominator to avoid division by zero
        return np.array([np.nan, np.nan, np.nan, np.nan])

    dT_d_theta = (K_den * dK_num_d_theta - K_num * dK_den_d_theta) / (K_den**2)

    # Potential energy part (V = -m2 * g * l * cos_t)
    dV_d_theta = m2 * g * l * sin_t

    # Combine to get dot_p_theta
    dot_p_theta = -(dT_d_theta + dV_d_theta)

    # Return the derivatives of all state variables.
    # The overall energy conservation depends significantly on the chosen integrator
    # (e.g., 'DOP853' here) and the absolute/relative tolerances (atol, rtol).
    # For near-perfect energy conservation in Hamiltonian systems, a symplectic integrator
    # would generally be preferred, but 'DOP853' with strict tolerances provides high accuracy.
    return np.array([dot_x, dot_theta, dot_p_x, dot_p_theta])

def hamiltonian_movable_support(state, m1, m2, l, g):
    x, theta, p_x, p_theta = state

    cos_t = np.cos(theta)
    sin_t = np.sin(theta)

    # The determinant of the mass matrix is key
    det_M = m2 * l**2 * (m1 + m2 * sin_t**2)

    if np.abs(det_M) < 1e-12:
        return np.nan

    # Kinetic energy (K) is calculated as 1/2 * p^T * M^-1 * p
    term_px_sq = m2 * l**2 * p_x**2
    term_ptheta_sq = (m1 + m2) * p_theta**2
    term_coupling = -2 * m2 * l * cos_t * p_x * p_theta

    kinetic_energy_val = (term_px_sq + term_ptheta_sq + term_coupling) / (2 * det_M)

    # Potential Energy (V) (reference at x-axis for theta=0)
    potential_energy_val = -m2 * g * l * cos_t

    return kinetic_energy_val + potential_energy_val

def energy_linearized_movable_support(state, m1, m2, l, g):
    x, theta, dot_x, dot_theta = state

    # Kinetic Energy for linearized system
    KE = 0.5 * (m1 + m2) * dot_x**2 + \
         0.5 * m2 * l**2 * dot_theta**2 + \
         m2 * l * dot_x * dot_theta

    # Potential Energy for linearized system
    PE = 0.5 * m2 * g * l * theta**2

    return KE + PE

def calculate_total_mechanical_energy_pendulum(l, g, sigma, gamma, phi, phi_dot, t):
    """Calculates the total mechanical energy of the pendulum with horizontal oscillating support.
    This is NOT a conserved quantity in the presence of the oscillating support, as the support does work.
    E = 0.5 * m * l^2 * phi_dot^2 + 0.5 * m * (dx/dt)^2 + mgl(1-cos(phi))
    However, for the specific equation of motion given, this isn't a simple conserved energy.
    The Hamiltonian is time-dependent (due to the explicit 't' in the forcing term).
    A more useful quantity for understanding is the energy in the rotating frame, or just the Hamiltonian if it's constant.
    Here, we'll calculate the 'naive' mechanical energy considering only the pendulum's motion relative to an inertial frame
    and the potential energy relative to its lowest point, ignoring the work done by the support for this specific function.
    Since the equations are given, we will calculate the Kinetic and Potential energy of the pendulum mass only.
    This is for the non-linear horizontal rotating support pendulum.
    """
    # Pendulum kinetic energy: 0.5 * m * v_rel^2 where v_rel is velocity relative to support
    # Support velocity: dx_support/dt = -sigma * gamma * sin(gamma * t)
    # Pendulum x-velocity: dx_support/dt + l * phi_dot * cos(phi)
    # Pendulum y-velocity: l * phi_dot * sin(phi)
    # For the equation given, it's easier to consider the total kinetic energy of the bob from the given equation of motion if it were derived from a Lagrangian.
    # However, the given equation is a direct second-order ODE for phi.
    # Let's consider the total energy of the pendulum bob relative to an inertial frame, where the support x-position is X_s(t) = sigma*cos(gamma*t)

    # Position of pendulum bob
    x_s = sigma * np.cos(gamma * t)
    x_bob = x_s + l * np.sin(phi)
    y_bob = -l * np.cos(phi) # Measuring from support height

    # Velocity of pendulum bob
    vx_s = -sigma * gamma * np.sin(gamma * t)
    vx_bob = vx_s + l * phi_dot * np.cos(phi) # d(x_bob)/dt
    vy_bob = l * phi_dot * np.sin(phi) # d(y_bob)/dt

    # Kinetic Energy (assuming unit mass m=1 for simplicity as it's often absorbed or relative)
    # Since the equation of motion is given for a single pendulum bob 'm', we use a generic 'm_val' from context if available
    # If 'm' is not explicitly passed to this function, assume it's part of 'g' implicitly or a unit mass.
    # Let's use m=1 as the equation of motion already has 'm' factored out.
    m = 1.0 # This function expects l, g, sigma, gamma, phi, phi_dot, t. The mass 'm' was part of the original derivation but factored out.

    KE_bob = 0.5 * m * (vx_bob**2 + vy_bob**2)

    # Potential Energy
    PE_bob = m * g * (l - y_bob) # Relative to the lowest point of the bob if support is fixed at y=0, adjusted for variable support y.
                               # For the given equation where gravity is 'g*sin(phi)', potential is mgl(1-cos(phi))
    PE_bob = m * g * l * (1 - np.cos(phi))

    return KE_bob + PE_bob


print("Starting comparative animation for Pendulum with Movable Support...")

# --- Simulation Parameters ---
T_anim = 20 # Animation duration
dt_anim_frames = 0.05 # Time step for frames
t_eval_anim = np.linspace(0, T_anim, int(T_anim / dt_anim_frames))

# --- System Parameters ---
m1_comp = 1.0 # Mass of support
m2_comp = 1.0 # Mass of pendulum
l_comp = 1.0  # Length of pendulum
g_comp = 9.81 # Gravity

# --- Initial Conditions ---
x0_comp = 0.0
theta0_comp = np.deg2rad(90) # Updated to 90 degrees (user request: 90-120 degrees)
dot_x0_comp = 0.0
dot_theta0_comp = 0.0

# Initial conditions for Linearized System: [x, theta, dot_x, dot_theta]
y0_lin_comp = [x0_comp, theta0_comp, dot_x0_comp, dot_theta0_comp]

# Initial conditions for Non-Linear System: [x, theta, p_x, p_theta]
# For initial zero velocities, momenta are also zero.
p_x0_comp = 0.0
p_theta0_comp = 0.0
y0_nonlin_comp = [x0_comp, theta0_comp, p_x0_comp, p_theta0_comp]

# --- Solve Equations of Motion ---
print("Solving linearized system...")
sol_lin_comp = solve_ivp(system_dynamics_movable_support_linearized, (0, T_anim), y0_lin_comp,
                        method='DOP853', t_eval=t_eval_anim, atol=1e-12, rtol=1e-12,
                        args=(m1_comp, m2_comp, l_comp, g_comp))

print("Solving non-linear system...")
sol_nonlin_comp = solve_ivp(hamilton_equations_movable_support, (0, T_anim), y0_nonlin_comp,
                             method='DOP853', t_eval=t_eval_anim, atol=1e-12, rtol=1e-12,
                             args=(m1_comp, m2_comp, l_comp, g_comp))

# --- Extract and Process Non-Linear Solutions for Plotting ---
x_nonlin_sol = sol_nonlin_comp.y[0]
theta_nonlin_sol = sol_nonlin_comp.y[1]
p_x_nonlin_sol = sol_nonlin_comp.y[2]
p_theta_nonlin_sol = sol_nonlin_comp.y[3]

# Calculate velocities for non-linear system from momenta
dot_x_nonlin_sol = []
dot_theta_nonlin_sol = []
for i in range(len(t_eval_anim)):
    _, current_theta, current_px, current_ptheta = sol_nonlin_comp.y[:, i]
    cos_theta = np.cos(current_theta)
    sin_theta = np.sin(current_theta)
    # Use the actual determinant of the mass matrix for calculating velocities
    det_M_actual_for_velocities = m2_comp * l_comp**2 * (m1_comp + m2_comp * sin_theta**2)

    if np.abs(det_M_actual_for_velocities) < 1e-12:
        dot_x_nonlin_sol.append(np.nan)
        dot_theta_nonlin_sol.append(np.nan)
    else:
        dx = (m2_comp * l_comp**2 * current_px - m2_comp * l_comp * cos_theta * current_ptheta) / det_M_actual_for_velocities
        dt = ((m1_comp + m2_comp) * current_ptheta - m2_comp * l_comp * cos_theta * current_px) / det_M_actual_for_velocities
        dot_x_nonlin_sol.append(dx)
        dot_theta_nonlin_sol.append(dt)

dot_x_nonlin_sol = np.array(dot_x_nonlin_sol)
dot_theta_nonlin_sol = np.array(dot_theta_nonlin_sol)

# --- Calculate Energies ---
energy_lin_comp = np.array([energy_linearized_movable_support(sol_lin_comp.y[:, i], m1_comp, m2_comp, l_comp, g_comp) for i in range(sol_lin_comp.y.shape[1])])
energy_nonlin_comp = np.array([hamiltonian_movable_support(sol_nonlin_comp.y[:, i], m1_comp, m2_comp, l_comp, g_comp) for i in range(sol_nonlin_comp.y.shape[1])])

# --- Animation Setup ---
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Comparative Animation: Movable Support Pendulum (Non-Linear vs. Linearized)', fontsize=16)

# Set limits for pendulum plots
max_x_pos = max(np.max(np.abs(x_nonlin_sol + l_comp * np.sin(theta_nonlin_sol))), np.max(np.abs(sol_lin_comp.y[0] + l_comp * np.sin(sol_lin_comp.y[1])))) + 0.5
max_y_pos = l_comp + 0.5

for ax_p in [axs[0, 0], axs[0, 1]]:
    ax_p.set_xlim(-max_x_pos, max_x_pos)
    ax_p.set_ylim(-(max_y_pos), 0.5) # Allow for y-axis to extend slightly above 0 for cart
    ax_p.set_aspect('equal', 'box')
    ax_p.grid(True)

# Energy Plot Limits
for ax_e in [axs[1, 0], axs[1, 1]]:
    ax_e.set_xlim(0, T_anim)
    ax_e.set_xlabel('Time (s)')
    ax_e.set_ylabel('Total Energy')
    ax_e.grid(True)

# Non-Linear Pendulum Plot (Top-Left)
axs[0, 0].set_title('Non-Linear Model')
line_nonlin, = axs[0, 0].plot([], [], 'o-', lw=2, color='blue', label='Pendulum')
cart_nonlin, = axs[0, 0].plot([], [], 's', markersize=15, color='gray', label='Support')

# Linearized Pendulum Plot (Top-Right)
axs[0, 1].set_title('Linearized Model')
line_lin, = axs[0, 1].plot([], [], 'o-', lw=2, color='red', label='Pendulum')
cart_lin, = axs[0, 1].plot([], [], 's', markersize=15, color='gray', label='Support')

# Non-Linear Energy Plot (Bottom-Left)
axs[1, 0].set_title('Non-Linear Energy (Hamiltonian)')
energy_line_nonlin, = axs[1, 0].plot([], [], lw=2, color='blue')
axs[1, 0].set_ylim(np.nanmin(energy_nonlin_comp[~np.isnan(energy_nonlin_comp)]) * 0.95, np.nanmax(energy_nonlin_comp[~np.isnan(energy_nonlin_comp)]) * 1.05)

# Linearized Energy Plot (Bottom-Right)
axs[1, 1].set_title('Linearized Energy')
energy_line_lin, = axs[1, 1].plot([], [], lw=2, color='red')
axs[1, 1].set_ylim(np.nanmin(energy_lin_comp[~np.isnan(energy_lin_comp)]) * 0.95, np.nanmax(energy_lin_comp[~np.isnan(energy_lin_comp)]) * 1.05)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

# Initialization function
def init():
    line_nonlin.set_data([], [])
    cart_nonlin.set_data([], [])
    line_lin.set_data([], [])
    cart_lin.set_data([], [])
    energy_line_nonlin.set_data([], [])
    energy_line_lin.set_data([], [])
    return line_nonlin, cart_nonlin, line_lin, cart_lin, energy_line_nonlin, energy_line_lin

# Animation function
def animate_comp(i):
    # Non-Linear Pendulum
    x_cart_nonlin = x_nonlin_sol[i]
    theta_nonlin = theta_nonlin_sol[i]
    x_pend_nonlin = x_cart_nonlin + l_comp * np.sin(theta_nonlin)
    y_pend_nonlin = -l_comp * np.cos(theta_nonlin)
    line_nonlin.set_data([x_cart_nonlin, x_pend_nonlin], [0, y_pend_nonlin])
    cart_nonlin.set_data([x_cart_nonlin], [0])

    # Linearized Pendulum
    x_cart_lin = sol_lin_comp.y[0, i]
    theta_lin = sol_lin_comp.y[1, i]
    x_pend_lin = x_cart_lin + l_comp * np.sin(theta_lin) # Use sin/cos for plotting actual position
    y_pend_lin = -l_comp * np.cos(theta_lin)
    line_lin.set_data([x_cart_lin, x_pend_lin], [0, y_pend_lin])
    cart_lin.set_data([x_cart_lin], [0])

    # Energy Plots
    energy_line_nonlin.set_data(t_eval_anim[:i+1], energy_nonlin_comp[:i+1])
    energy_line_lin.set_data(t_eval_anim[:i+1], energy_lin_comp[:i+1])

    return line_nonlin, cart_nonlin, line_lin, cart_lin, energy_line_nonlin, energy_line_lin

print("Generating comparative animation...")
an_comp = animation.FuncAnimation(fig, animate_comp, frames=len(t_eval_anim), interval=dt_anim_frames*1000, blit=True)
plt.close(fig)
display(HTML(an_comp.to_jshtml()))

print("Comparative animation generated and displayed.")

# Save the animation to a file and provide a download link
animation_filename = 'movable_support_pendulum_comparison.mp4'
print(f"Saving animation as '{animation_filename}'...")
an_comp.save(animation_filename, writer='ffmpeg', fps=30)
print("Animation saved.")
display(FileLink(animation_filename))

### 2.11 Energy Conservation Analysis (Non-Linear vs. Linearized)

This section provides a detailed analysis of energy conservation for both the non-linear and linearized movable support pendulum. While the linearized system (when solved analytically or numerically with careful initialization) should show perfect conservation, the non-linear system solved numerically will always exhibit some very small deviations. The plot below highlights these deviations to assess the quality of the numerical integration.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming t_eval_anim, energy_nonlin_comp, and energy_lin_comp are available from cell urZ-RSVRJ0bx

plt.figure(figsize=(14, 6))

# Plot for Non-Linear System Energy
plt.subplot(1, 2, 1)
plt.plot(t_eval_anim, energy_nonlin_comp, color='blue', linewidth=0.7)
plt.title('Non-Linear System: Hamiltonian vs. Time')
plt.xlabel('Time (s)')
plt.ylabel('Total Energy (Hamiltonian)')
plt.grid(True)

# Determine max absolute deviation from initial energy for the non-linear system
initial_energy_nonlin = energy_nonlin_comp[0]
max_deviation_nonlin = np.nanmax(np.abs(energy_nonlin_comp - initial_energy_nonlin))
plt.text(0.05, 0.95, f'Initial H: {initial_energy_nonlin:.2e}\nMax Deviation: {max_deviation_nonlin:.2e}',
         transform=plt.gca().transAxes, fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.5))


# Plot for Linearized System Energy
plt.subplot(1, 2, 2)
plt.plot(t_eval_anim, energy_lin_comp, color='red', linewidth=0.7)
plt.title('Linearized System: Total Energy vs. Time')
plt.xlabel('Time (s)')
plt.ylabel('Total Energy')
plt.grid(True)

# Determine max absolute deviation from initial energy for the linearized system
initial_energy_lin = energy_lin_comp[0]
max_deviation_lin = np.nanmax(np.abs(energy_lin_comp - initial_energy_lin))
plt.text(0.05, 0.95, f'Initial E: {initial_energy_lin:.2f}\nMax Deviation: {max_deviation_lin:.2e}',
         transform=plt.gca().transAxes, fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.5))

plt.tight_layout()
plt.show()

print(f"For the Non-Linear System, the maximum absolute energy deviation is approximately {max_deviation_nonlin:.2e}.")
print(f"For the Linearized System, the maximum absolute energy deviation is approximately {max_deviation_lin:.2e}.")
print("These values indicate excellent energy conservation for both simulations, particularly for the non-linear system where non-symplectic integrators inherently introduce small numerical errors.")

## Problem 3: Pendulum with Rotating Support

3.a) circular rotating support

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from IPython.display import display

# --- 3.1 Non-Linear System Dynamics for Pendulum with Rotating Support ---
# The equation of motion provided is: lφ̈ + g sin(ϕ) - σγ^2 cos(ϕ-γt) = 0
# We convert this second-order ODE into a system of two first-order ODEs.
# Let state = [phi, phi_dot]

def system_dynamics_rotating_support_nonlinear(t, state, l, g, sigma, gamma):
    phi, phi_dot = state

    # phi_ddot = (-g/l) * sin(phi) + (sigma * gamma**2 / l) * cos(phi - gamma * t)
    phi_ddot = -(g / l) * np.sin(phi) + (sigma * gamma**2 / l) * np.cos(phi - gamma * t)

    return [phi_dot, phi_ddot]

# --- Effective Potential Definition ---
# U_eff = -mσlγ^2 sin(ϕ-γt)
# Assuming 'm' is a general mass factor, it can be absorbed into sigma for the potential shape,
# or we can keep it explicit if we want actual energy values.
# For now, let's include 'm' for a physical interpretation of energy.

def effective_potential(phi, t, m, l, sigma, gamma):
    return -m * sigma * l * gamma**2 * np.sin(phi - gamma * t)

print("Non-linear system dynamics and effective potential for rotating support pendulum defined.")

# --- 3.2 Simulation of the Non-Linear System ---

# Define System Parameters
l_val = 1.0     # Length of the pendulum (m)
g_val = 9.81    # Acceleration due to gravity (m/s^2)
sigma_val = 0.5 # Amplitude factor for the support's rotation (dimensionless)
gamma_val = 2.0 # Angular frequency of the support's rotation (rad/s)
m_val = 1.0     # Mass of the pendulum bob (kg), for potential energy calculation

# Initial Conditions: [phi_0, phi_dot_0]
# Start with pendulum slightly displaced and at rest relative to the rotating frame initially, or absolute rest.
phi0 = np.deg2rad(90) # Initial angle (10 degrees)
phi_dot0 = 0.0        # Initial angular velocity
y0 = [phi0, phi_dot0]

# Simulation Time
T_sim = 100       # Total simulation time (s)
t_eval = np.linspace(0, T_sim, 10000) # Time points for evaluation

print("Simulating non-linear pendulum with rotating support...")

sol_nonlinear_rotating = solve_ivp(system_dynamics_rotating_support_nonlinear, (0, T_sim), y0,
                                   method='DOP853', t_eval=t_eval, atol=1e-9, rtol=1e-9,
                                   args=(l_val, g_val, sigma_val, gamma_val))

phi_sol = sol_nonlinear_rotating.y[0]
phi_dot_sol = sol_nonlinear_rotating.y[1]

print("Simulation complete.")

### 3.3 Time Evolution of Angular Position and Angular Velocity

Let's visualize how the pendulum's angular position ($\phi$) and angular velocity ($\dot{\phi}$) change over time for the non-linear system with a rotating support. This plot helps us understand the basic dynamics, such as oscillations, period, and any signs of chaotic or synchronous behavior.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming phi_sol, phi_dot_sol, and t_eval are available from the previous simulation cell

plt.figure(figsize=(12, 6))

plt.subplot(2, 1, 1)
plt.plot(t_eval, phi_sol, label=r'$\phi(t)$', color='blue')
plt.title('Angular Position vs. Time (Non-Linear Rotating Support Pendulum)')
plt.xlabel('Time (s)')
plt.ylabel('Angle (rad)')
plt.grid(True)
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(t_eval, phi_dot_sol, label=r'$\dot{\phi}(t)$', color='red')
plt.title('Angular Velocity vs. Time (Non-Linear Rotating Support Pendulum)')
plt.xlabel('Time (s)')
plt.ylabel('Angular Velocity (rad/s)')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

print("Time evolution plots for angular position and velocity generated.")

### 3.4 Effective Potential Analysis

This plot visualizes the effective potential energy ($U_{eff}$) of the pendulum with a rotating support. Analyzing the shape of this potential can provide insights into the system's stable equilibrium points, oscillations, and how these change with the system parameters, especially due to the time-dependent driving force from the rotating support. It's particularly useful for understanding the conditions for synchronization and stability.

For a system with a time-dependent forcing, the effective potential can give insight into the dynamics in the rotating frame of reference. The minima of this potential correspond to stable equilibrium points, while maxima correspond to unstable ones. The pendulum tends to oscillate around these minima.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Parameters are already defined in cell M8_MHHAhYF3F:
# l_val, g_val, sigma_val, gamma_val, m_val

# Define a range for phi (angular position)
phi_range = np.linspace(-2 * np.pi, 2 * np.pi, 500) # From -360 to 360 degrees

# Choose a specific time instance for the plot (e.g., t=0 or any t_eval point)
# We'll use t=0 for simplicity to see the initial potential shape.
t_plot = 0.0

# Calculate the effective potential over the phi range
# U_eff = -m * sigma * l * gamma**2 * np.sin(phi - gamma * t)
# Note: The effective_potential function as defined in M8_MHHAhYF3F is:
# def effective_potential(phi, t, m, l, sigma, gamma):
#    return -m * sigma * l * gamma**2 * np.sin(phi - gamma * t)

# Make sure all parameters are accessible. They are global from previous cells.
effective_potential_values = effective_potential(phi_range, t_plot, m_val, l_val, sigma_val, gamma_val)

plt.figure(figsize=(10, 6))
plt.plot(phi_range, effective_potential_values, label=r'$U_{eff}(\phi, t=0)$', color='purple')
plt.title('Effective Potential Energy of Pendulum with Rotating Support')
plt.xlabel(r'$\phi$ (pendulum angle, rad)')
plt.ylabel(r'$U_{eff}$ (Effective Potential Energy)')
plt.grid(True)
plt.axhline(0, color='gray', linestyle='--')
plt.axvline(0, color='gray', linestyle='--')
plt.legend()
plt.show()

print("Effective potential plot generated.")

### 3.5 Solución Analítica Linealizada y Comparación con la Solución Numérica

Para ángulos pequeños $\phi$, podemos linealizar la ecuación de movimiento del péndulo con soporte rotatorio:

Ecuación no lineal: $l\ddot{\phi} + g \sin(\phi) - \sigma\gamma^2 \cos(\phi - \gamma t) = 0$

Para $\phi \ll 1$ (pequeñas oscilaciones):
$\sin(\phi) \approx \phi$
$\cos(\phi - \gamma t) \approx 1$ (asumiendo que $\phi - \gamma t$ es pequeño, lo cual es una aproximación más fuerte para esta linealización, o simplemente consideramos la fuerza impulsora como constante para la parte particular si el término $\gamma t$ domina)

La linealización más común para sistemas impulsados de esta forma se centra en la parte oscilatoria y la constante de impulso. Si consideramos la fuerza impulsora $\sigma\gamma^2 \cos(\phi - \gamma t)$ como una perturbación, y para sincronización a menudo se busca la respuesta en el marco de referencia giratorio, pero para una comparación directa en el marco fijo, podemos aproximar el término de forzamiento como constante para una solución particular de la forma $\phi = C$. Para el término $g \sin(\phi)$, usamos la aproximación de ángulo pequeño. Para el término de forzamiento, si el movimiento del péndulo es principalmente en torno a la vertical ($\phi \approx 0$), entonces $\cos(\phi - \gamma t) \approx \cos(-\gamma t) = \cos(\gamma t)$.

Sin embargo, la ecuación dada es:
$l\ddot{\phi} + g \sin(\phi) - \sigma\gamma^2 \cos(\phi - \gamma t) = 0$

Una linealización común para un péndulo impulsado sinusoidalmente (como si $\cos(\phi - \gamma t)$ fuera una señal de entrada) sería mantener el término de forzamiento con dependencia temporal y linealizar solo el término gravitacional:

$l\ddot{\phi} + g\phi = \sigma\gamma^2 \cos(\phi - \gamma t)$

Pero, si la pregunta es sobre la sincronización, estamos interesados en la respuesta del péndulo en un sistema de referencia que rota a la misma velocidad que el soporte. Si definimos un nuevo ángulo $\alpha = \phi - \gamma t$, entonces $\phi = \alpha + \gamma t$. Sustituyendo esto en la ecuación original:

$l(\ddot{\alpha} + 0) + g \sin(\alpha + \gamma t) - \sigma\gamma^2 \cos(\alpha) = 0$

Esta transformación no linealiza el seno, pero simplifica el coseno. Si volvemos a la ecuación original y usamos la aproximación para pequeñas oscilaciones alrededor de $\phi = 0$ y una fuerza impulsora simplificada a $\cos(-\gamma t) = \cos(\gamma t)$ (si $\phi$ es pequeño), la ecuación se convierte en:

$l\ddot{\phi} + g\phi - \sigma\gamma^2 \cos(\gamma t) = 0$

$\ddot{\phi} + \frac{g}{l}\phi = \frac{\sigma\gamma^2}{l} \cos(\gamma t)$

Esta es la ecuación de un oscilador armónico amortiguado forzado (en este caso sin amortiguamiento). La solución general es la suma de la solución homogénea y una solución particular.

Frecuencia natural: $\omega_0 = \sqrt{\frac{g}{l}}$

La solución homogénea es: $\phi_h(t) = A \cos(\omega_0 t) + B \sin(\omega_0 t)$

La solución particular para un forzamiento $\cos(\gamma t)$ es (si $\gamma \neq \omega_0$):
$\phi_p(t) = \frac{(\sigma\gamma^2/l)}{\omega_0^2 - \gamma^2} \cos(\gamma t)$

Entonces, la solución analítica linealizada es:

$\phi(t) = A \cos(\omega_0 t) + B \sin(\omega_0 t) + \frac{(\sigma\gamma^2/l)}{(g/l) - \gamma^2} \cos(\gamma t)$

Donde $A$ y $B$ se determinan a partir de las condiciones iniciales $\phi(0)=\phi_0$ y $\dot{\phi}(0)=\dot{\phi}_0$:

De $\phi(0) = \phi_0$:
$\phi_0 = A + \frac{(\sigma\gamma^2/l)}{(g/l) - \gamma^2}$
$A = \phi_0 - \frac{(\sigma\gamma^2/l)}{(g/l) - \gamma^2}$

De $\dot{\phi}(0) = \dot{\phi}_0$:
$\dot{\phi}(t) = -A\omega_0 \sin(\omega_0 t) + B\omega_0 \cos(\omega_0 t) - \frac{(\sigma\gamma^2/l)\gamma}{(g/l) - \gamma^2} \sin(\gamma t)$
$\dot{\phi}_0 = B\omega_0$
$B = \frac{\dot{\phi}_0}{\omega_0}$

Consideraremos el caso de resonancia $\gamma = \omega_0$ por separado, pero para este problema, usualmente $\gamma \neq \omega_0$ para observar la sincronización.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters (from M8_MHHAhYF3F)
l_val = 1.0     # Length of the pendulum (m)
g_val = 9.81    # Acceleration due to gravity (m/s^2)
sigma_val = 0.5 # Amplitude factor for the support's rotation (dimensionless)
gamma_val = 2.0 # Angular frequency of the support's rotation (rad/s)

# Initial Conditions (from M8_MHHAhYF3F)
phi0 = np.deg2rad(90) # Initial angle (10 degrees)
phi_dot0 = 0.0        # Initial angular velocity

# Time points for evaluation (from M8_MHHAhYF3F)
t_eval_lin = t_eval # Use the same time array as the non-linear simulation

# --- Calculate Linear Analytical Solution ---
omega0 = np.sqrt(g_val / l_val)

# Check for resonance condition (gamma == omega0)
if np.isclose(gamma_val, omega0):
    print("Warning: Resonance condition (gamma == omega0) detected. The current analytical solution formula is not valid for resonance. The solution will grow linearly with time.")
    # For resonance, the particular solution takes the form t*sin(omega0*t)
    # We'll use a simplified version for now, acknowledging its limitations.
    # For exact resonance, a specific solution form is needed (e.g., A*t*sin(omega0*t)).
    # For demonstration, we'll proceed with a very small perturbation to avoid division by zero.
    denominator = (g_val / l_val) - gamma_val**2 + 1e-9 # Add a small epsilon to avoid division by zero
else:
    denominator = (g_val / l_val) - gamma_val**2

# Calculate constants A and B
term_force = (sigma_val * gamma_val**2 / l_val)

A_const = phi0 - (term_force / denominator)
B_const = phi_dot0 / omega0

# Linear analytical solution for phi(t)
phi_analytical = A_const * np.cos(omega0 * t_eval_lin) + \
                 B_const * np.sin(omega0 * t_eval_lin) + \
                 (term_force / denominator) * np.cos(gamma_val * t_eval_lin)

print("Linear analytical solution calculated.")

### 3.6 Comparación de la Solución Numérica (No Lineal) y la Solución Analítica (Linealizada)

Aquí, visualizamos cómo se compara el comportamiento del péndulo obtenido de la simulación numérica no lineal con la solución analítica aproximada del sistema linealizado. Esto nos ayuda a entender el rango de validez de la aproximación lineal y a observar fenómenos como la sincronización cuando las condiciones se acercan al régimen lineal.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming phi_sol (non-linear numerical solution) and phi_analytical (linear analytical solution)
# are available from previous cells, along with t_eval.

plt.figure(figsize=(12, 6))
plt.plot(t_eval, phi_sol, label='Numerical Solution (Non-Linear)', color='blue', linewidth=1.5, alpha=0.8)
plt.plot(t_eval_lin, phi_analytical, label='Analytical Solution (Linearized)', color='red', linestyle='--', linewidth=1.5)
plt.title(r'Comparison: Rotating Support Pendulum (Non-Linear vs. Linearized)\nInitial $\phi_0$: ' + f'{np.rad2deg(phi0):.1f}' + ' degrees')
plt.xlabel('Time (s)')
plt.ylabel(r'Angle $\phi$ (rad)')
plt.grid(True)
plt.legend()
plt.xlim(0, 50) # Focus on initial phase
plt.show()

print("Comparative plot of numerical and analytical solutions generated.")

### 3.7 Analysis of Sigma's Impact on Synchronization

This section provides an interactive tool to explore how the amplitude factor of the support's rotation, `sigma`, influences the pendulum's synchronization. By adjusting the `sigma` slider, you can observe changes in the pendulum's angular position ($\phi$) over time, revealing conditions for synchronization or more complex behaviors.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from IPython.display import display

# Default gravity constant
DEFAULT_G = 9.81

# Default length for standard case frequency calculation
DEFAULT_L_STANDARD = 1.0

# Reuse system_dynamics_rotating_support_nonlinear from M8_MHHAhYF3F
# and parameters like l_val, g_val, gamma_val from M8_MHHAhYF3F

def interactive_sigma_impact(l_val, g_val, sigma_val, gamma_val, phi0_deg, phi_dot0):
    # Convert initial angle from degrees to radians
    phi0 = np.deg2rad(phi0_deg)
    y0 = [phi0, phi_dot0]

    # Simulation Time (can be adjusted for finer detail)
    T_sim = 100
    t_eval = np.linspace(0, T_sim, 5000)

    # Solve the non-linear equations of motion
    sol_nonlinear_rotating_sigma = solve_ivp(system_dynamics_rotating_support_nonlinear, (0, T_sim), y0,
                                              method='DOP853', t_eval=t_eval, atol=1e-9, rtol=1e-9,
                                              args=(l_val, g_val, sigma_val, gamma_val))

    phi_sol_sigma = sol_nonlinear_rotating_sigma.y[0]

    plt.figure(figsize=(12, 6))

    # Plotting angular position (phi) as amplitude
    plt.plot(t_eval, phi_sol_sigma, label=r'$\phi(t)$', color='blue', linewidth=1.5, alpha=0.8)
    plt.title(rf'Amplitude (Angular Position) vs. Time (Non-Linear, $\sigma$={sigma_val:.2f}, $\gamma$={gamma_val:.2f})')
    plt.xlabel('Time (s)')
    plt.ylabel(r'Amplitude (Angle, rad)')

    plt.grid(True)
    plt.legend()
    plt.xlim(0, 50) # Focus on initial phase

    plt.tight_layout()
    plt.show()

# Create interactive sliders for parameters relevant to this analysis
# Reusing existing parameters where appropriate, but making sigma interactive
widgets.interact(
    interactive_sigma_impact,
    l_val=widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=l_val, description='Longitud (l):', continuous_update=False),
    g_val=widgets.FloatSlider(min=0.1, max=20.0, step=0.1, value=g_val, description='Gravedad (g):', continuous_update=False),
    sigma_val=widgets.FloatSlider(min=0.0, max=2.0, step=0.01, value=sigma_val, description=r'Sigma ($\sigma$):', continuous_update=True),
    gamma_val=widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=gamma_val, description=r'Gamma ($\gamma$):', continuous_update=True), # Make gamma_val interactive
    phi0_deg=widgets.FloatSlider(min=-180.0, max=180.0, step=5.0, value=np.rad2deg(phi0), description='Phi0 (deg):', continuous_update=False),
    phi_dot0=widgets.FloatSlider(min=-5.0, max=5.0, step=0.1, value=phi_dot0, description='Phi_dot0:', continuous_update=False)
);

### 3.8 Time Evolution of Relative Angle to Rotating Support (Non-Linear System)

This plot visualizes the relative angle of the pendulum with respect to the rotating support frame ($\phi - \gamma t$). This is particularly useful for observing synchronization phenomena, where the pendulum's motion might lock into a stable phase relationship with the rotating support. When synchronized, this relative angle tends to a constant value or oscillates around a constant value.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming phi_sol, gamma_val, and t_eval are available from previous simulation cells

# Calculate the relative angle
relative_angle = phi_sol - gamma_val * t_eval

plt.figure(figsize=(12, 6))
plt.plot(t_eval, relative_angle, label=r'$\phi - \gamma t$', color='green', linewidth=1.5, alpha=0.8)
plt.title(r'Relative Angle to Rotating Support vs. Time (Non-Linear)')
plt.xlabel('Time (s)')
plt.ylabel(r'Relative Angle (rad)')
plt.grid(True)
plt.legend()
plt.xlim(0, 50) # Focus on initial phase for better visibility
plt.show()

print("Time evolution plot for relative angle generated.")

3.b) horizontal rotating suport

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from IPython.display import display

# --- 3.1 Non-Linear System Dynamics for Pendulum with Horizontal Rotating Support ---
# Equation of motion: ml^2 ϕ̈ + mlg sin(ϕ) - mσlγ^2 cos(γt) cos(ϕ) = 0
# Dividing by ml^2: ϕ̈ = -(g/l) sin(ϕ) + (σγ^2/l) cos(γt) cos(ϕ)
# Let state = [phi, phi_dot]

def system_dynamics_horizontal_rotating_nonlinear(t, state, l, g, sigma, gamma):
    phi, phi_dot = state
    phi_ddot = -(g / l) * np.sin(phi) + (sigma * gamma**2 / l) * np.cos(gamma * t) * np.cos(phi)
    return [phi_dot, phi_ddot]

# --- 3.2 Linearized System Dynamics for Pendulum with Horizontal Rotating Support ---
# Equation of motion: ϕ̈ + ω₀² ϕ = (σγ²)/l cos(γt), where ω₀² = g/l
# Let state = [phi, phi_dot]

def system_dynamics_horizontal_rotating_linearized(t, state, l, g, sigma, gamma):
    phi, phi_dot = state
    omega0_sq = g / l
    phi_ddot = -omega0_sq * phi + (sigma * gamma**2 / l) * np.cos(gamma * t)
    return [phi_dot, phi_ddot]

print("System dynamics for pendulum with horizontal rotating support (non-linear and linearized) defined.")

# --- 3.3 Interactive Time Evolution and Phase Space Plots ---

def interactive_horizontal_pendulum_plots(l_val, g_val, sigma_val, gamma_val, phi0_deg, phi_dot0, T_sim, use_linearized):
    phi0 = np.deg2rad(phi0_deg)
    y0 = [phi0, phi_dot0]
    t_eval = np.linspace(0, T_sim, 5000) # Time points for evaluation

    if use_linearized:
        sol = solve_ivp(system_dynamics_horizontal_rotating_linearized, (0, T_sim), y0,
                        method='DOP853', t_eval=t_eval, atol=1e-9, rtol=1e-9,
                        args=(l_val, g_val, sigma_val, gamma_val))
        title_prefix = "Linearized"
    else:
        sol = solve_ivp(system_dynamics_horizontal_rotating_nonlinear, (0, T_sim), y0,
                        method='DOP853', t_eval=t_eval, atol=1e-9, rtol=1e-9,
                        args=(l_val, g_val, sigma_val, gamma_val))
        title_prefix = "Non-Linear"

    phi_sol = sol.y[0]
    phi_dot_sol = sol.y[1]

    fig, axs = plt.subplots(1, 2, figsize=(16, 6))

    # Plot 1: Amplitude vs Time
    axs[0].plot(t_eval, phi_sol, label=r'$\phi(t)$', color='blue', linewidth=1.5, alpha=0.8)
    axs[0].set_title(f'{title_prefix}: Angular Position vs. Time')
    axs[0].set_xlabel('Time (s)')
    axs[0].set_ylabel(r'Angle $\phi$ (rad)')
    axs[0].grid(True)
    axs[0].legend()

    # Plot 2: Phase Space (phi vs phi_dot)
    axs[1].plot(phi_sol, phi_dot_sol, label=r'$(\phi, \dot{\phi})$', color='red', linewidth=1.0, alpha=0.8)
    axs[1].set_title(f'{title_prefix}: Phase Space ({r"$\phi$"} vs {r"$\dot{\phi}$"})')
    axs[1].set_xlabel(r'Angle $\phi$ (rad)')
    axs[1].set_ylabel(r'Angular Velocity $\dot{\phi}$ (rad/s)')
    axs[1].grid(True)
    axs[1].legend()

    plt.tight_layout()
    plt.show()

# --- Interactive Widgets Setup ---
l_slider = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Length (l):', continuous_update=False)
g_slider = widgets.FloatSlider(value=9.81, min=0.1, max=20.0, step=0.1, description='Gravity (g):', continuous_update=False)
sigma_slider = widgets.FloatSlider(value=0.5, min=0.0, max=2.0, step=0.01, description=r'Sigma ($\sigma$):', continuous_update=False)
gamma_slider = widgets.FloatSlider(value=2.0, min=0.1, max=5.0, step=0.1, description=r'Gamma ($\gamma$):', continuous_update=False)
phi0_slider = widgets.FloatSlider(value=10.0, min=-180.0, max=180.0, step=5.0, description='Phi0 (deg):', continuous_update=False)
phi_dot0_slider = widgets.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1, description='Phi_dot0:', continuous_update=False)
T_sim_slider = widgets.FloatSlider(value=50.0, min=10.0, max=200.0, step=10.0, description='Sim Time (s):', continuous_update=False)
use_linearized_checkbox = widgets.Checkbox(value=False, description='Use Linearized Model', indent=False)

interactive_plot = widgets.interactive(
    interactive_horizontal_pendulum_plots,
    l_val=l_slider,
    g_val=g_slider,
    sigma_val=sigma_slider,
    gamma_val=gamma_slider,
    phi0_deg=phi0_slider,
    phi_dot0=phi_dot0_slider,
    T_sim=T_sim_slider,
    use_linearized=use_linearized_checkbox
)

display(interactive_plot)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from IPython.display import display, HTML
import matplotlib.animation as animation

# Re-using the non-linear system dynamics from cell 75WCGcYCmj6R
def system_dynamics_horizontal_rotating_nonlinear(t, state, l, g, sigma, gamma):
    phi, phi_dot = state
    phi_ddot = -(g / l) * np.sin(phi) + (sigma * gamma**2 / l) * np.cos(gamma * t) * np.cos(phi)
    return [phi_dot, phi_ddot]

# --- Lyapunov Dynamics for Horizontal Rotating Support Pendulum ---
# This function combines the original system dynamics with the tangent space dynamics.
# State: [phi, phi_dot, delta_phi, delta_phi_dot]
# where delta_phi and delta_phi_dot represent the perturbation vector.
def lyapunov_dynamics_horizontal_rotating(t, combined_state, l, g, sigma, gamma):
    phi, phi_dot, delta_phi, delta_phi_dot = combined_state

    # System dynamics (f) for the main trajectory (phi, phi_dot)
    f_phi_dot, f_phi_ddot = system_dynamics_horizontal_rotating_nonlinear(t, [phi, phi_dot], l, g, sigma, gamma)

    # Jacobian matrix elements J = d(f_i)/d(state_j)
    # f1 = phi_dot
    # f2 = -(g/l) * sin(phi) + (sigma * gamma**2 / l) * cos(gamma * t) * cos(phi)

    J11 = 0 # d(f1)/d(phi)
    J12 = 1 # d(f1)/d(phi_dot)
    J21 = -(g / l) * np.cos(phi) - (sigma * gamma**2 / l) * np.cos(gamma * t) * np.sin(phi) # d(f2)/d(phi)
    J22 = 0 # d(f2)/d(phi_dot)

    # Tangent space dynamics: d(delta_state)/dt = J * delta_state
    # [d(delta_phi)/dt, d(delta_phi_dot)/dt] = [[J11, J12], [J21, J22]] * [delta_phi, delta_phi_dot]
    d_delta_phi_dt = J11 * delta_phi + J12 * delta_phi_dot
    d_delta_phi_dot_dt = J21 * delta_phi + J22 * delta_phi_dot

    return [f_phi_dot, f_phi_ddot, d_delta_phi_dt, d_delta_phi_dot_dt]

# --- Function to Calculate Lyapunov Exponent ---
def calculate_lyapunov_horizontal_rotating(y0, l, g, sigma, gamma, T_max=500, dt=0.1, initial_perturbation_magnitude=1e-8):
    t_steps = np.arange(0, T_max, dt)
    x = np.array(y0) # Main trajectory state [phi, phi_dot]

    # Start with a small perturbation vector in a random direction
    delta_x = np.random.rand(len(y0)) # Perturbation vector [delta_phi, delta_phi_dot]
    delta_x /= np.linalg.norm(delta_x) # Normalize to unit length

    lyap_history = []
    running_sum = 0.0
    num_renormalizations = 0

    for i, t in enumerate(t_steps):
        # Combine main state and scaled perturbation for the augmented system
        combined_state = np.concatenate([x, delta_x * initial_perturbation_magnitude])

        sol = solve_ivp(lyapunov_dynamics_horizontal_rotating, [t, t + dt], combined_state,
                        method='DOP853', atol=1e-10, rtol=1e-10, args=(l, g, sigma, gamma), max_step=dt)

        if sol.status != 0 or np.any(np.isnan(sol.y[:, -1])) or np.any(np.isinf(sol.y[:, -1])):
            lyap_history.extend([np.nan] * (len(t_steps) - i))
            break

        res = sol.y[:, -1]
        x_next = res[:len(y0)]
        new_delta_x_scaled = res[len(y0):] # This is delta_x scaled by current divergence

        current_perturbation_magnitude = np.linalg.norm(new_delta_x_scaled)

        if current_perturbation_magnitude < 1e-20 or np.any(np.isnan(current_perturbation_magnitude)): # Avoid log of zero or extremely small/invalid numbers
            lyap_history.extend([np.nan] * (len(t_steps) - i))
            break

        running_sum += np.log(current_perturbation_magnitude / initial_perturbation_magnitude)
        num_renormalizations += 1

        # Renormalize the perturbation vector for the next step
        delta_x = new_delta_x_scaled / current_perturbation_magnitude # Normalize new_delta_x_scaled to unit vector
        x = x_next

        # Calculate and store the instantaneous Lyapunov exponent
        if num_renormalizations * dt > 0:
            lyap_history.append(running_sum / (num_renormalizations * dt))
        else:
            lyap_history.append(np.nan)

    # Fill remaining with NaN if simulation broke early
    while len(lyap_history) < len(t_steps):
        lyap_history.append(np.nan)

    return t_steps, np.array(lyap_history)

# --- Interactive Plot for Lyapunov Exponent ---
def interactive_lyapunov_plot_horizontal_rotating(l_val, g_val, sigma_val, gamma_val, phi0_deg, phi_dot0):
    phi0 = np.deg2rad(phi0_deg) # Convert initial angle to radians
    y0_actual = [phi0, phi_dot0]

    print(f"Calculating Lyapunov exponent for phi0={phi0_deg} deg, sigma={sigma_val}, gamma={gamma_val}...")
    t_mle, mle_val = calculate_lyapunov_horizontal_rotating(y0_actual, l_val, g_val, sigma_val, gamma_val, T_max=1000, dt=0.5)

    plt.figure(figsize=(10, 6))
    plt.plot(t_mle, mle_val, color='red', label='Maximal Lyapunov Exponent')
    plt.axhline(y=0, color='black', linestyle='--')
    plt.title(f"Lyapunov Exponent Convergence ($\phi_0={phi0_deg}^\circ, \sigma={sigma_val:.2f}, \gamma={gamma_val:.2f}$)")
    plt.xlabel("Time (s)")
    plt.ylabel(r"$\lambda$ (bits/s)")
    plt.grid(True)
    plt.legend()
    plt.ylim(auto=True)
    plt.show()

    # Print final MLE, handling cases where it might be NaN
    if len(mle_val) > 0 and not np.isnan(mle_val[-1]):
        print(f"Final MLE: {mle_val[-1]:.5f}")
    else:
        print("Final MLE: Diverged (NaN) or not computed.")

print("--- Interactive Lyapunov Exponent Plot ---")
widgets.interact(
    interactive_lyapunov_plot_horizontal_rotating,
    l_val=widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='l'),
    g_val=widgets.FloatSlider(min=0.1, max=20.0, step=0.1, value=9.81, description='g'),
    sigma_val=widgets.FloatSlider(min=0.0, max=2.0, step=0.01, value=0.5, description='sigma'),
    gamma_val=widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=2.0, description='gamma'),
    phi0_deg=widgets.FloatSlider(min=0.0, max=360.0, step=1.0, value=180.0, description='phi0 (deg)'),
    phi_dot0=widgets.FloatSlider(min=-5.0, max=5.0, step=0.1, value=0.0, description='phi_dot0')
);

# --- Divergence Animation for Horizontal Rotating Support Pendulum ---
def run_divergence_animation_horizontal_rotating(l_val=1.0, g_val=9.81, sigma_val=0.5, gamma_val=2.0, phi0_deg=180.0, phi_dot0=0.0, T_anim=50.0):

    # Convert initial angle to radians. Use a slight perturbation for the second trajectory.
    phi0_A = np.deg2rad(phi0_deg) # Main trajectory
    perturbation_deg = 0.001 # Small angular perturbation in degrees
    phi0_B = np.deg2rad(phi0_deg + perturbation_deg) # Perturbed trajectory

    y0_A = [phi0_A, phi_dot0]
    y0_B = [phi0_B, phi_dot0]

    dt_frames = 0.02 # Time step for animation frames
    t_eval_anim = np.linspace(0, T_anim, int(T_anim / dt_frames))

    print(f"Simulating two pendulums with initial angles {phi0_deg} and {phi0_deg + perturbation_deg} degrees...")
    sol_A = solve_ivp(system_dynamics_horizontal_rotating_nonlinear, (0, T_anim), y0_A,
                        method='DOP853', t_eval=t_eval_anim, atol=1e-10, rtol=1e-10,
                        args=(l_val, g_val, sigma_val, gamma_val))
    sol_B = solve_ivp(system_dynamics_horizontal_rotating_nonlinear, (0, T_anim), y0_B,
                        method='DOP853', t_eval=t_eval_anim, atol=1e-10, rtol=1e-10,
                        args=(l_val, g_val, sigma_val, gamma_val))

    # Extract positions and velocities
    phi_A = sol_A.y[0]
    phi_dot_A = sol_A.y[1]
    phi_B = sol_B.y[0]
    phi_dot_B = sol_B.y[1]

    # Calculate physical (x,y) coordinates for animation
    # Support's y-position is fixed for plotting. Let's say at y_support = 0
    y_support = 0.0 # Fixed vertical position of the support
    x_support_base = sigma_val * np.cos(gamma_val * t_eval_anim)

    x_bob_A = x_support_base + l_val * np.sin(phi_A)
    y_bob_A = y_support - l_val * np.cos(phi_A)

    x_bob_B = x_support_base + l_val * np.sin(phi_B)
    y_bob_B = y_support - l_val * np.cos(phi_B)

    # Calculate divergence (Euclidean distance in state space: phi, phi_dot)
    divergence = np.linalg.norm(sol_A.y - sol_B.y, axis=0)
    # Initial perturbation magnitude is between phi0_A and phi0_B in state space
    # Assuming phi_dot0 is the same, initial perturbation is perturbation_deg in phi
    initial_perturbation_state_space = np.linalg.norm(np.array([phi0_A, phi_dot0]) - np.array([phi0_B, phi_dot0]))

    # Avoid log of zero or very small numbers by replacing them with a small epsilon
    log_divergence = np.log(np.maximum(divergence / initial_perturbation_state_space, 1e-10))

    # Calculate Total Mechanical Energy for each pendulum
    # Note: The energy is not conserved due to the external forcing from the support's oscillation.
    # However, we can plot it to observe its time evolution.
    energy_A = np.array([calculate_total_mechanical_energy_pendulum(l_val, g_val, sigma_val, gamma_val, phi_A[i], phi_dot_A[i], t_eval_anim[i]) for i in range(len(t_eval_anim))])
    energy_B = np.array([calculate_total_mechanical_energy_pendulum(l_val, g_val, sigma_val, gamma_val, phi_B[i], phi_dot_B[i], t_eval_anim[i]) for i in range(len(t_eval_anim))])

    # --- Animation Setup ---
    fig = plt.figure(figsize=(15, 10))
    gs = fig.add_gridspec(2, 2, height_ratios=[2, 1], width_ratios=[1, 1])

    ax_pendulum = fig.add_subplot(gs[0, 0]) # Top-Left for pendulum motion (both A and B)
    ax_divergence = fig.add_subplot(gs[0, 1]) # Top-Right for log divergence
    ax_energy = fig.add_subplot(gs[1, :])     # Bottom row, spanning both columns, for energy plot

    fig.suptitle(f'Horizontal Rotating Support Pendulum Divergence ($\phi_0={phi0_deg}^\circ, \sigma={sigma_val:.2f}, \gamma={gamma_val:.2f}$)', fontsize=14)

    # Set limits for pendulum plots
    max_x_range = l_val + sigma_val + 0.2 # Max x extent from center

    ax_pendulum.set_xlim(-max_x_range, max_x_range)
    ax_pendulum.set_ylim(-l_val - 0.2, l_val + 0.2) # Centered around y_support
    ax_pendulum.set_aspect('equal', 'box')
    ax_pendulum.set_title('Pendulum Motion')
    ax_pendulum.grid(True)

    line_A, = ax_pendulum.plot([], [], 'o-', lw=2, color='blue', label='Pendulum A')
    # Represent support as a large circle ('o') for a 'wheel'
    support_A, = ax_pendulum.plot([], [], 'o', markersize=15, color='gray', label='Support')
    line_B, = ax_pendulum.plot([], [], 'o-', lw=2, color='red', label='Pendulum B')
    support_B, = ax_pendulum.plot([], [], 'o', markersize=15, color='darkgray', label='Support B')
    ax_pendulum.legend(loc='upper right')

    # Add text for sigma_val to the pendulum plot
    text_sigma = ax_pendulum.text(0.02, 0.95, f'$\sigma$ = {sigma_val:.2f}', transform=ax_pendulum.transAxes, fontsize=10, verticalalignment='top')


    # Log Divergence Plot
    divergence_line, = ax_divergence.plot([], [], lw=2, color='green')
    ax_divergence.set_title('Log-Separation')
    ax_divergence.set_xlabel('Time (s)')
    ax_divergence.set_ylabel(r'$\ln(|\delta \vec{x}| / |\delta \vec{x}_0|)$') # Fixed LaTeX here
    ax_divergence.grid(True)
    divergence_y_min = np.nanmin(log_divergence[~np.isnan(log_divergence)]) if np.any(~np.isnan(log_divergence)) else -10
    divergence_y_max = np.nanmax(log_divergence[~np.isnan(log_divergence)]) if np.any(~np.isnan(log_divergence)) else 10
    ax_divergence.set_xlim(0, T_anim)
    ax_divergence.set_ylim(divergence_y_min - 1, divergence_y_max + 1)

    # Energy Plot
    energy_line_A, = ax_energy.plot([], [], lw=2, color='blue', label='Energy A')
    energy_line_B, = ax_energy.plot([], [], lw=2, color='red', label='Energy B')
    ax_energy.set_title('Total Mechanical Energy')
    ax_energy.set_xlabel('Time (s)')
    ax_energy.set_ylabel('Energy')
    ax_energy.grid(True)
    ax_energy.legend(loc='upper right')
    # Dynamically set y-limits for energy plot
    if not np.all(np.isnan(energy_A)) and not np.all(np.isnan(energy_B)):
        min_e = np.nanmin([np.nanmin(energy_A), np.nanmin(energy_B)])
        max_e = np.nanmax([np.nanmax(energy_A), np.nanmax(energy_B)])
        ax_energy.set_ylim(min_e * 0.9, max_e * 1.1)
    ax_energy.set_xlim(0, T_anim)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent suptitle overlap

    # Initialization function for animation
    def init():
        line_A.set_data([], [])
        support_A.set_data([], [])
        line_B.set_data([], [])
        support_B.set_data([], [])
        divergence_line.set_data([], [])
        energy_line_A.set_data([], [])
        energy_line_B.set_data([], [])
        return line_A, support_A, line_B, support_B, divergence_line, energy_line_A, energy_line_B

    # Animation update function
    def animate(i):
        # Pendulum A
        line_A.set_data([x_support_base[i], x_bob_A[i]], [y_support, y_bob_A[i]])
        support_A.set_data([x_support_base[i]], [y_support])

        # Pendulum B
        line_B.set_data([x_support_base[i], x_bob_B[i]], [y_support, y_bob_B[i]])
        support_B.set_data([x_support_base[i]], [y_support])

        # Divergence
        divergence_line.set_data(t_eval_anim[:i+1], log_divergence[:i+1])

        # Energy
        energy_line_A.set_data(t_eval_anim[:i+1], energy_A[:i+1])
        energy_line_B.set_data(t_eval_anim[:i+1], energy_B[:i+1])

        return line_A, support_A, line_B, support_B, divergence_line, energy_line_A, energy_line_B, text_sigma

    print("Generating divergence animation...")
    an = animation.FuncAnimation(fig, animate, frames=len(t_eval_anim), interval=dt_frames*1000, blit=True)
    plt.close(fig)

    # Save the animation
    animation_filename = 'horizontal_rotating_pendulum_divergence.mp4'
    print(f"Saving animation as '{animation_filename}'...")
    an.save(animation_filename, writer='ffmpeg', fps=30)
    print("Animation saved.")

    # Display the animation in the notebook
    display(HTML(an.to_jshtml()))
    print("Divergence animation generated and displayed.")

# --- Placeholder for calculate_total_mechanical_energy_pendulum function (if not already defined) ---
# This function should be defined elsewhere in the notebook or added if missing.
# For the purpose of fixing the AttributeError, we assume it's available.
# It calculates the 'naive' mechanical energy of the pendulum bob, not the conserved Hamiltonian.
# Its definition is critical for the energy plots.

def calculate_total_mechanical_energy_pendulum(l, g, sigma, gamma, phi, phi_dot, t):
    # Pendulum kinetic energy and potential energy
    m = 1.0 # Assuming unit mass for simplicity as it's often absorbed or relative

    # Support velocity (dx_support/dt)
    vx_s = -sigma * gamma * np.sin(gamma * t)

    # Pendulum bob velocity components
    vx_bob = vx_s + l * phi_dot * np.cos(phi)
    vy_bob = l * phi_dot * np.sin(phi)

    KE_bob = 0.5 * m * (vx_bob**2 + vy_bob**2)

    # Potential energy relative to the lowest point of the bob if support is fixed at y=0
    PE_bob = m * g * l * (1 - np.cos(phi))

    return KE_bob + PE_bob


print("\n--- Divergence Animation (Fixed Parameters) ---")
run_divergence_animation_horizontal_rotating(
    l_val=1.0, g_val=9.81, sigma_val=0.5, gamma_val=2.0,
    phi0_deg=180.0, phi_dot0=0.0, T_anim=50.0
);

Problem 3.c pendulum kapitza

### 3.c.1 Effective Potential Analysis for Kapitza Pendulum

This section visualizes the instantaneous effective potential ($U_{eff}$) for the Kapitza pendulum as defined by the user: $U_{eff}=m\sigma l\gamma^2 \cos(\gamma t) \cos(\phi)$.

**Note on Kapitza Stability:** It's important to understand that the well-known Kapitza stability (stable inverted pendulum) is typically explained using an *averaged effective potential*. This averaged potential is derived by considering the rapid oscillations of the support and their effect on the pendulum's motion over time. The instantaneous potential plotted below fluctuates with time and doesn't directly show the time-averaged stability at the inverted position (e.g., $\phi = \pi$) in a static graph. For this reason, the next section includes a dynamic animation to illustrate the effect more clearly.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display

# --- Effective Potential Definition for Kapitza Pendulum (User-provided) ---
def effective_potential_kapitza(phi, t, m, l, sigma, gamma):
    # U_eff = m * sigma * l * gamma**2 * cos(gamma * t) * cos(phi)
    return m * sigma * l * gamma**2 * np.cos(gamma * t) * np.cos(phi)

# --- Averaged Effective Potential for Kapitza Pendulum ---
def averaged_effective_potential_kapitza(phi, m, l, g, sigma, gamma):
    # This is the standard averaged Kapitza potential that explains inverted stability.
    # It combines gravitational potential with a term from averaged rapid oscillations.
    # For vertical support oscillation y_s = sigma * cos(gamma*t)
    gravitational_potential = m * g * l * (1 - np.cos(phi))
    oscillation_induced_potential = 0.25 * m * (sigma * gamma)**2 * np.cos(phi)**2
    return gravitational_potential + oscillation_induced_potential

# Parameters (from Problem 3.a context, adjusted for Kapitza effect)
l_val_kap = 1.0     # Length of the pendulum (m)
g_val_kap = 9.81    # Acceleration due to gravity (m/s^2)
sigma_val_kap = .5 # Amplitude factor for the support's vertical oscillation (dimensionless)
gamma_val_kap = 20.0 # Angular frequency of the support's oscillation (rad/s) - HIGH FREQUENCY
m_val_kap = 1.0     # Mass of the pendulum bob (kg)

def interactive_kapitza_potential(t_plot_kap, m_val_kap_interactive):
    # Define a range for phi (angular position)
    phi_range_kap = np.linspace(-2 * np.pi, 2 * np.pi, 500) # From -360 to 360 degrees

    # Calculate the instantaneous effective potential over the phi range for the selected t_plot_kap
    instantaneous_potential_values_kap = effective_potential_kapitza(phi_range_kap, t_plot_kap, m_val_kap_interactive, l_val_kap, sigma_val_kap, gamma_val_kap)

    # Calculate the averaged effective potential
    averaged_potential_values_kap = averaged_effective_potential_kapitza(phi_range_kap, m_val_kap_interactive, l_val_kap, g_val_kap, sigma_val_kap, gamma_val_kap)

    plt.figure(figsize=(10, 6))
    plt.plot(phi_range_kap, instantaneous_potential_values_kap, label=f'Instantaneous $U_{{eff}}(\phi, t={t_plot_kap:.2f})$', color='darkgreen')
    plt.plot(phi_range_kap, averaged_potential_values_kap, label='Averaged $U_{{eff}}(\phi)$', color='blue', linestyle='--')
    plt.title('Kapitza Pendulum: Effective Potential Energy')
    plt.xlabel(r'$\phi$ (pendulum angle, rad)')
    plt.ylabel(r'$U_{{eff}}$ (Effective Potential Energy)')
    plt.grid(True)
    plt.axhline(0, color='gray', linestyle='--')
    plt.axvline(np.pi, color='red', linestyle=':', label='$\phi = \pi$ (Inverted)')
    plt.axvline(0, color='blue', linestyle=':', label='$\phi = 0$ (Downward)')
    plt.legend()
    plt.show()

print("Interactive effective potential plot for Kapitza pendulum generated.")

# Create an interactive widget for t_plot_kap and m_val_kap
widgets.interact(
    interactive_kapitza_potential,
    t_plot_kap=widgets.FloatSlider(min=0.0, max=10.0, step=0.1, value=0.0, description='Time (t):'),
    m_val_kap_interactive=widgets.FloatSlider(min=0.1, max=10.0, step=0.1, value=m_val_kap, description='Mass (m):')
);

### 3.c.2 Kapitza Pendulum Animation: Visualizing the 'Shadow' Effect

This animation visualizes the dynamics of the Kapitza pendulum, specifically demonstrating the 'shadow' effect. This effect refers to the phenomenon where a pendulum with a rapidly oscillating support can be stabilized in its inverted position (i.e., pointing upwards), even though it would normally be unstable under static conditions. The pendulum bob will appear to rapidly vibrate around an average stable position, creating a 'shadow' of its rapid motion while its overall average trajectory remains fixed.

We use the non-linear equation provided by the user: $ml^2 \ddot{\phi} + mlg \sin(\phi) + m\sigma l\gamma^2 \cos(\gamma t) \sin(\phi) = 0$.

To observe the inverted stability, the frequency of the support's oscillation ($\gamma$) must be high, and the amplitude factor ($\sigma$) must be sufficiently large. The initial conditions are set near the inverted equilibrium ($\phi = \pi$) with a small perturbation to show its stability.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import matplotlib.animation as animation
from IPython.display import HTML, display, FileLink

# Adjust animation embedding limit to avoid warnings for larger animations
plt.rcParams['animation.embed_limit'] = 50.0  # Set to 50 MB

# --- System Dynamics for Kapitza Pendulum (User-provided equation) ---
def system_dynamics_kapitza_nonlinear(t, state, l, g, sigma, gamma):
    phi, phi_dot = state
    # ml^2 phi'' + mlg sin(phi) + m sigma l gamma^2 cos(gamma t) sin(phi) = 0
    # phi'' = -(g/l) sin(phi) - (sigma * gamma**2 / l) cos(gamma t) sin(phi)
    phi_ddot = -(g / l) * np.sin(phi) - (sigma * gamma**2 / l) * np.cos(gamma * t) * np.sin(phi)
    return [phi_dot, phi_ddot]

# --- Simple Pendulum Dynamics (Comparison Model) ---
# This is a simple, undriven pendulum, which will fall from the inverted position.
# phi'' + (g/l)sin(phi) = 0
def system_dynamics_kapitza_simple_pendulum_comparison(t, state, l, g, sigma, gamma):
    phi, phi_dot = state
    phi_ddot = -(g / l) * np.sin(phi)
    return [phi_dot, phi_ddot]

# --- Simulation Parameters ---
l_val_kap = 1.0     # Length of the pendulum (m)
g_val_kap = 9.81    # Acceleration due to gravity (m/s^2)
sigma_val_kap = .5 # Amplitude factor for the support's oscillation (dimensionless)
gamma_val_kap = 20.0 # Angular frequency of the support's oscillation (rad/s) - HIGH FREQUENCY
m_val_kap = 1.0     # Mass (not explicitly used in ODE, but conceptual)

T_anim_kap = 60.0 # Animation duration (s) - Increased to 1 minute
dt_anim_frames_kap = 0.02 # Time step for frames
t_eval_anim_kap = np.linspace(0, T_anim_kap, int(T_anim_kap / dt_anim_frames_kap))

# Initial Conditions for Kapitza Pendulum
# Start near the inverted position to show Kapitza stability
phi0_kap = np.deg2rad(179.9) # Initial angle (179.9 degrees - very close to inverted)
phi_dot0_kap = 0.0
y0_kap_nonlinear = [phi0_kap, phi_dot0_kap] # For non-linear Kapitza simulation

# Initial Conditions for Simple Pendulum Comparison Model
y0_kap_simple = [phi0_kap, phi_dot0_kap] # Same initial conditions

print("Simulating non-linear Kapitza pendulum...")
sol_kapitza_nonlinear = solve_ivp(system_dynamics_kapitza_nonlinear, (0, T_anim_kap), y0_kap_nonlinear,
                         method='DOP853', t_eval=t_eval_anim_kap, atol=1e-9, rtol=1e-9,
                         args=(l_val_kap, g_val_kap, sigma_val_kap, gamma_val_kap))

print("Simulating simple pendulum for comparison...")
sol_kapitza_simple = solve_ivp(system_dynamics_kapitza_simple_pendulum_comparison, (0, T_anim_kap), y0_kap_simple,
                         method='DOP853', t_eval=t_eval_anim_kap, atol=1e-9, rtol=1e-9,
                         args=(l_val_kap, g_val_kap, sigma_val_kap, gamma_val_kap))

phi_sol_nonlinear = sol_kapitza_nonlinear.y[0]
phi_sol_simple = sol_kapitza_simple.y[0]

# --- Calculate Support Position and Pendulum Bob Position ---
# The user's equation implies a vertical oscillation of the support.
# The forcing term is m sigma l gamma^2 cos(gamma t) sin(phi),
# which comes from a vertically oscillating support y_s(t) = sigma * cos(gamma * t).

y_support_kap_nonlinear = sigma_val_kap * np.cos(gamma_val_kap * t_eval_anim_kap)

# Positions for non-linear pendulum
x_bob_kap_nonlinear = l_val_kap * np.sin(phi_sol_nonlinear)
y_bob_kap_nonlinear = y_support_kap_nonlinear - l_val_kap * np.cos(phi_sol_nonlinear)

# Positions for simple pendulum (uses the same support oscillation for visual comparison)
y_support_kap_simple = sigma_val_kap * np.cos(gamma_val_kap * t_eval_anim_kap)
x_bob_kap_simple = l_val_kap * np.sin(phi_sol_simple)
y_bob_kap_simple = y_support_kap_simple - l_val_kap * np.cos(phi_sol_simple)

# Calculate temporal average for non-linear phi (unwrap to handle angles crossing 2pi)
sim_duration_for_average = min(100.0, T_anim_kap)
idx_for_average = np.where(t_eval_anim_kap >= sim_duration_for_average / 2.0)[0]

if len(idx_for_average) > 0:
    phi_data_for_average = np.unwrap(phi_sol_nonlinear[idx_for_average])
    temporal_average_phi_raw = np.mean(phi_data_for_average)
    # Normalize the average angle to be within [-pi, pi) for better readability
    temporal_average_phi = (temporal_average_phi_raw + np.pi) % (2 * np.pi) - np.pi
else:
    temporal_average_phi = np.nan # Not enough data for average


# --- Animation Setup ---
# Create a figure with 1 row and 2 columns: left for animation, right for phi vs t plot
fig_kap, (ax_kap_anim, ax_kap_plot) = plt.subplots(1, 2, figsize=(16, 8))
fig_kap.suptitle(rf'Kapitza Pendulum (Non-Linear vs. Simple Pendulum) ($\gamma={gamma_val_kap:.1f}, \sigma={sigma_val_kap:.1f}$)', fontsize=14)

# Configure Animation Subplot (Left)
ax_kap_anim.set_aspect('equal') # Removed autoscale_on=False
# Adjust limits to accommodate motion around inverted position
ax_kap_anim.set_xlim(-1.5*l_val_kap, 1.5*l_val_kap)
ax_kap_anim.set_ylim(-l_val_kap - sigma_val_kap - 0.2, l_val_kap + sigma_val_kap + 0.2)
ax_kap_anim.grid(True)
ax_kap_anim.set_title('Kapitza Pendulum Animation')

line_kap_nonlinear, = ax_kap_anim.plot([], [], 'o-', lw=2, color='blue', label='Non-Linear Kapitza Bob')
pivot_kap_nonlinear, = ax_kap_anim.plot([], [], 's', markersize=10, color='red', label='Kapitza Pivot') # Square for pivot
trajectory_kap_nonlinear, = ax_kap_anim.plot([], [], '-', lw=0.5, color='gray', alpha=0.5, label='Kapitza Trajectory') # For 'shadow'

line_kap_simple, = ax_kap_anim.plot([], [], 'o--', lw=1, color='green', label='Simple Bob')
pivot_kap_simple, = ax_kap_anim.plot([], [], 'x', markersize=8, color='darkred', label='Simple Pivot') # Cross for pivot

ax_kap_anim.legend(loc='upper right')


# Configure Phi vs T Plot (Right)
ax_kap_plot.set_title(rf'Angular Position vs. Time (Initial $\phi_0$: {np.rad2deg(phi0_kap):.1f} deg)')
ax_kap_plot.set_xlabel('Time (s)')
ax_kap_plot.set_ylabel(r'Angle $\phi$ (rad)')
ax_kap_plot.grid(True)
ax_kap_plot.set_xlim(0, T_anim_kap)
# Dynamically set y-limits for phi plot based on both solutions
max_phi_plot = max(np.max(phi_sol_nonlinear), np.max(phi_sol_simple))
min_phi_plot = min(np.min(phi_sol_nonlinear), np.min(phi_sol_simple))
# Ensure y-limits cover 0 and pi for better context
ax_kap_plot.set_ylim(min(min_phi_plot, -np.pi) - 0.2, max(max_phi_plot, np.pi) + 0.2)

line_phi_nonlinear, = ax_kap_plot.plot([], [], label=r'Non-Linear Kapitza $\phi(t)$', color='blue', linewidth=1.5, alpha=0.8)
line_phi_simple_plot, = ax_kap_plot.plot([], [], label=r'Simple $\phi(t)$ (Comparison)', color='green', linestyle='--', linewidth=1.5)
line_phi_average, = ax_kap_plot.plot([0, T_anim_kap], [temporal_average_phi, temporal_average_phi],
                                     label=rf'Average Kapitza $\phi$: {temporal_average_phi:.3f} rad ({np.rad2deg(temporal_average_phi):.1f} deg)',
                                     color='red', linestyle=':', linewidth=2)
ax_kap_plot.legend()


def init_kap():
    line_kap_nonlinear.set_data([], [])
    pivot_kap_nonlinear.set_data([], [])
    trajectory_kap_nonlinear.set_data([], [])
    line_kap_simple.set_data([], [])
    pivot_kap_simple.set_data([], [])
    line_phi_nonlinear.set_data([], [])
    line_phi_simple_plot.set_data([], [])
    return line_kap_nonlinear, pivot_kap_nonlinear, trajectory_kap_nonlinear, line_kap_simple, pivot_kap_simple, line_phi_nonlinear, line_phi_simple_plot

def animate_kap(i):
    # Update Animation Subplot - Non-Linear Pendulum
    pivot_x_nonlin = 0 # Pivot oscillates along y-axis at x=0
    pivot_y_nonlin = y_support_kap_nonlinear[i]
    pivot_kap_nonlinear.set_data([pivot_x_nonlin], [pivot_y_nonlin])

    bob_x_nonlin = x_bob_kap_nonlinear[i] # X-coordinate of the bob relative to the pivot's x (which is 0)
    bob_y_nonlin = y_bob_kap_nonlinear[i] # Absolute y-coordinate of the bob
    line_kap_nonlinear.set_data([pivot_x_nonlin, bob_x_nonlin], [pivot_y_nonlin, bob_y_nonlin])

    # Update trajectory 'shadow' for non-linear
    num_shadow_points = 20
    if i > num_shadow_points:
        trajectory_kap_nonlinear.set_data(x_bob_kap_nonlinear[i-num_shadow_points:i],
                                y_bob_kap_nonlinear[i-num_shadow_points:i])
    else:
        trajectory_kap_nonlinear.set_data(x_bob_kap_nonlinear[:i],
                                y_bob_kap_nonlinear[:i])

    # Update Animation Subplot - Simple Pendulum
    pivot_x_simple = 0 # For visual consistency, pivot is shown oscillating even though it's undriven
    pivot_y_simple = y_support_kap_simple[i]
    pivot_kap_simple.set_data([pivot_x_simple], [pivot_y_simple])

    bob_x_simple = x_bob_kap_simple[i] # X-coordinate of the bob relative to the pivot's x (which is 0)
    bob_y_simple = y_bob_kap_simple[i] # Absolute y-coordinate of the bob
    line_kap_simple.set_data([pivot_x_simple, bob_x_simple], [pivot_y_simple, bob_y_simple])

    # Update Phi vs T Plot
    line_phi_nonlinear.set_data(t_eval_anim_kap[:i+1], phi_sol_nonlinear[:i+1])
    line_phi_simple_plot.set_data(t_eval_anim_kap[:i+1], phi_sol_simple[:i+1])

    return line_kap_nonlinear, pivot_kap_nonlinear, trajectory_kap_nonlinear, line_kap_simple, pivot_kap_simple, line_phi_nonlinear, line_phi_simple_plot

print("Generating Kapitza pendulum animation...")
an_kap = animation.FuncAnimation(fig_kap, animate_kap, frames=len(t_eval_anim_kap), interval=dt_anim_frames_kap*1000, blit=True)
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout for suptitle
plt.close(fig_kap)

# Display the animation
display(HTML(an_kap.to_jshtml()))
print("Kapitza pendulum animation generated and displayed. Observe the fast vibrations (shadow) around the inverted position.")

# Save the animation to a file and provide a download link
animation_filename = 'kapitza_pendulum_animation.mp4'
print(f"Saving animation as '{animation_filename}'...")
an_kap.save(animation_filename, writer='ffmpeg', fps=30)
print("Animation saved.")
display(FileLink(animation_filename))


### 3.c.3 Time Evolution of Angular Position and its Temporal Average (Kapitza Pendulum)

This plot displays the pendulum's angular position ($\phi$) over time, along with a horizontal line indicating its temporal average over the specified simulation duration (or up to 100 seconds if the simulation is longer). This helps to visualize the overall stability and oscillatory behavior around a mean position.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Ensure phi_sol_nonlinear and t_eval_anim_kap are available from previous cells
# (specifically from cell 451b9adc where the Kapitza pendulum is simulated)

# Define the time range for calculating the average
# The user requested average up to 100s, but the animation simulation (T_anim_kap) is 10s.
# So, we'll average over the full simulation time available.
sim_duration_for_average = min(100.0, T_anim_kap)

# Find the indices corresponding to the sim_duration_for_average
idx_for_average = np.where(t_eval_anim_kap <= sim_duration_for_average)[0]

# Extract the relevant data for averaging
phi_data_for_average = phi_sol_nonlinear[idx_for_average]
time_data_for_average = t_eval_anim_kap[idx_for_average]

# Calculate the temporal average of phi
temporal_average_phi = np.mean(phi_data_for_average)

plt.figure(figsize=(12, 6))
plt.plot(t_eval_anim_kap, phi_sol_nonlinear, label=r'$\phi(t)$', color='blue', linewidth=1.5, alpha=0.8)
plt.axhline(y=temporal_average_phi, color='red', linestyle='--', label=rf'Temporal Average ($\phi$): {temporal_average_phi:.3f} rad')
plt.title('Kapitza Pendulum: Angular Position vs. Time with Temporal Average')
plt.xlabel('Time (s)')
plt.ylabel(r'Angle $\phi$ (rad)')
plt.grid(True)
plt.legend()
plt.xlim(0, T_anim_kap) # Plot over the full animation duration
plt.show()

print(f"Temporal average of phi over {sim_duration_for_average} seconds: {temporal_average_phi:.3f} rad")


Problem 4: Walt regulator

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from IPython.display import display

# --- 4.1 Parameter Definitions ---
# These parameters are representative for a Walt regulator or a similar system.
# You can adjust them to explore different dynamic regimes.
m1 = 1.0  # Mass of the sleeve/slider (or part of the rotating arm)
m2 = 0.5  # Mass of the flyball/bob (per arm, assuming two flyballs)
a = 1.0   # Length of the arms (distance from pivot to attachment point)
g = 9.81  # Acceleration due to gravity
Omega = 5.0 # Angular velocity of the main shaft (spin rate) - uppercase Omega in formulas

# --- 4.2 Lagrangian Function ---
def lagrangian(theta_dot, theta, m1_val, m2_val, a_val, Omega_val, g_val):
    """Calculates the Lagrangian L(theta_dot, theta) for the Walt Regulator."""
    # L(θ̇,θ) = m_1 a^2 (θ̇^2 + Ω^2 sin^2(θ)) + 2m_2 a^2 θ̇^2 sin^2(θ) + 2(m_1+m_2)ga cos(θ)
    term_m1 = m1_val * a_val**2 * (theta_dot**2 + Omega_val**2 * np.sin(theta)**2)
    term_m2 = 2 * m2_val * a_val**2 * theta_dot**2 * np.sin(theta)**2
    term_gravity = 2 * (m1_val + m2_val) * g_val * a_val * np.cos(theta)
    return term_m1 + term_m2 + term_gravity

# --- 4.3 Effective Potential from Lagrangian ---
def effective_potential_L(theta, m1_val, m2_val, a_val, Omega_val, g_val):
    """Calculates the effective potential V_eff(theta) derived from the Lagrangian."""
    # V_eff(θ) = -m_1 a^2 Ω^2 sin^2(θ) - 2(m_1+m_2)ga cos(θ)
    term_omega = -m1_val * a_val**2 * Omega_val**2 * np.sin(theta)**2
    term_gravity = -2 * (m1_val + m2_val) * g_val * a_val * np.cos(theta)
    return term_omega + term_gravity

# --- 4.4 Hamiltonian Function ---
def hamiltonian(p_theta, theta, m1_val, m2_val, a_val, Omega_val, g_val):
    """Calculates the Hamiltonian H(p_theta, theta) for the Walt Regulator."""
    # H(p_θ,θ) = (p_θ^2) / (4a^2 (m_1 + 2m_2 sin^2(θ))) - m_1 a^2 Ω^2 sin^2(θ) - 2(m_1+m_2)ga cos(θ)
    denominator = 4 * a_val**2 * (m1_val + 2 * m2_val * np.sin(theta)**2)
    if np.abs(denominator) < 1e-12:
        return np.nan # Avoid division by zero at theta=0 or pi
    term_p_theta = p_theta**2 / denominator
    term_omega = -m1_val * a_val**2 * Omega_val**2 * np.sin(theta)**2
    term_gravity = -2 * (m1_val + m2_val) * g_val * a_val * np.cos(theta)
    return term_p_theta + term_omega + term_gravity

# --- 4.5 Hamilton's Equations of Motion ---
def hamilton_equations_walt_regulator(t, state, m1_val, m2_val, a_val, Omega_val, g_val):
    """Implements Hamilton's equations for the Walt Regulator.
    state = [theta, p_theta]
    Returns [dtheta/dt, dp_theta/dt]
    """
    theta, p_theta = state

    # dtheta/dt = p_theta / (4a^2 (m_1 + 2m_2 sin^2(θ)))
    denominator_dtheta_dt = 4 * a_val**2 * (m1_val + 2 * m2_val * np.sin(theta)**2)
    if np.abs(denominator_dtheta_dt) < 1e-12:
        return np.array([np.nan, np.nan]) # Indicate integration failure
    theta_dot = p_theta / denominator_dtheta_dt

    # dp_theta/dt = -(p_theta / (4a^2 (m_1 + 2m_2 sin^2(θ))))^2 * 4a^2 m_2 sin(θ) cos(θ) - 2a sin(θ) [(m_1+m_2)g - m_1 aΩ^2 cos(θ)]
    # Note: the first term is -(theta_dot)^2 * ...
    term1_factor = 4 * a_val**2 * m2_val * np.sin(theta) * np.cos(theta)
    term1 = -(theta_dot)**2 * term1_factor

    term2_factor = (m1_val + m2_val) * g_val - m1_val * a_val * Omega_val**2 * np.cos(theta)
    term2 = -2 * a_val * np.sin(theta) * term2_factor

    p_theta_dot = term1 + term2

    return np.array([theta_dot, p_theta_dot])

# --- 4.6 Routhian Effective Potential ---
def routhian_effective_potential(theta, p_phi, m1_val, m2_val, a_val, g_val):
    """Calculates the Routhian Effective Potential V_Routh(theta).
    Note: p_phi is a conserved momentum if phi is cyclic, or a general constant.
    User formula: V_Routh(θ) = U(θ) + (p_ϕ^2)/(2I(θ)) = -2(m_1+m_2)ga cos(θ) + (p_ϕ^2)/(4m_1 a^2 sin(θ))
    It seems U(theta) here refers to only the gravitational potential.
    """
    # U(theta) is typically the non-generalized potential energy, usually from gravity.
    # From the Lagrangian, the actual potential terms are -2(m1+m2)ga cos(theta) - m1 a^2 Omega^2 sin^2(theta)
    # The provided formula for V_Routh uses U(theta) = -2(m1+m2)ga cos(theta), which is just the gravitational part.
    # And p_phi is probably related to the phi coordinate (azimuthal angle).
    # Let's assume p_phi is a constant value for now.

    # Assuming p_phi represents the conserved angular momentum for the phi coordinate if it's cyclic,
    # and the expression 4m_1 a^2 sin(theta) is the moment of inertia I(theta) related to the phi coordinate.

    gravitational_potential = -2 * (m1_val + m2_val) * g_val * a_val * np.cos(theta)

    # The user-provided formula for the second term: (p_phi^2) / (4 m1 a^2 sin(theta))
    denominator_routh = 4 * m1_val * a_val**2 * np.sin(theta)

    # Use np.where to handle division by zero for array inputs
    denominator_safe = np.where(np.abs(denominator_routh) < 1e-12, np.nan, denominator_routh)

    # p_phi here is a constant related to the angular momentum in phi, not p_theta
    # This p_phi is assumed to be an input constant.
    kinetic_like_term = p_phi**2 / denominator_safe

    return gravitational_potential + kinetic_like_term

print("Walt Regulator system definitions loaded.")

### 4.7 Time Evolution of $\theta(t)$ (Arm Angle) for the Walt Regulator

This plot shows how the angle $\theta$ of the regulator's arms changes over time. $\theta$ represents the angle that the arms make with the vertical axis. Observing its time evolution helps us understand the stability and oscillatory behavior of the regulator under the given initial conditions and system parameters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# --- Simulation of Walt Regulator --- Parameters are already defined in cell GbOiuGLb2LSN

def interactive_walt_regulator_theta0(theta0_deg):
    # Convert initial angle from degrees to radians
    theta0 = np.deg2rad(theta0_deg)

    # Initial Conditions for Walt Regulator
    # [theta_0, p_theta_0]
    # Let's start with the arms slightly open and no initial angular momentum.
    p_theta0 = 0.0          # Initial angular momentum of theta
    y0_walt = [theta0, p_theta0]

    # Simulation Time
    T_sim_walt = 20.0       # Total simulation time (s)
    t_eval_walt = np.linspace(0, T_sim_walt, 1000) # Time points for evaluation

    print("Simulating Walt Regulator...")
    sol_walt = solve_ivp(hamilton_equations_walt_regulator, (0, T_sim_walt), y0_walt,
                               method='DOP853', t_eval=t_eval_walt, atol=1e-10, rtol=1e-10,
                               args=(m1, m2, a, Omega, g))

    theta_sol_walt = sol_walt.y[0]
    p_theta_sol_walt = sol_walt.y[1]

    # --- Analytical Linearized Solutions ---

    # Find the equilibrium theta for the current Omega
    current_Omega_val = Omega # Global Omega from GbOiuGLb2LSN

    # Calculate the cosine value for the non-trivial equilibrium
    cosine_val_eq = ((m1 + m2) * g) / (m1 * a * current_Omega_val**2)

    theta_eq_rad = 0.0
    if np.abs(cosine_val_eq) <= 1:
        theta_eq_rad = np.arccos(cosine_val_eq)
        print(f"Calculated equilibrium angle (theta_eq) for Omega={current_Omega_val}: {np.rad2deg(theta_eq_rad):.2f} degrees")
    else:
        # If no non-trivial equilibrium, use initial theta0 as reference for linearization
        # This might not represent a stable equilibrium, but serves as a reference point for perturbation.
        print(f"No non-trivial equilibrium for Omega={current_Omega_val}. Using initial theta0 as reference for linearization.")
        theta_eq_rad = theta0 # Fallback

    # Calculate omega_0_user: \u03c9\u2080 = \u03a9 sin(\u03b8\u2080) \u221a(m\u2081/(m\u2081 + 2m\u2082 sin\u00b2(\u03b8\u2080)))
    # Ensure sin(theta_eq_rad) is not zero before division and square root
    if np.sin(theta_eq_rad) == 0 or (m1 + 2 * m2 * np.sin(theta_eq_rad)**2) == 0:
        omega_0_user = 0.0
    else:
        omega_0_user = current_Omega_val * np.sin(theta_eq_rad) * np.sqrt(m1 / (m1 + 2 * m2 * np.sin(theta_eq_rad)**2))

    # Calculate omega_Routh_user: \u03c9_Routh = \u03a9 sin(\u03b8\u2080) \u221a((4m\u2081 cos\u00b2(\u03b8\u2080))/(m\u2081 + 2m\u2082 sin\u00b2(\u03b8\u2080)))
    # Note: (1+3) is simplified to 4
    if np.sin(theta_eq_rad) == 0 or (m1 + 2 * m2 * np.sin(theta_eq_rad)**2) == 0:
        omega_Routh_user = 0.0
    else:
        omega_Routh_user = current_Omega_val * np.sin(theta_eq_rad) * np.sqrt((4 * m1 * np.cos(theta_eq_rad)**2) / (m1 + 2 * m2 * np.sin(theta_eq_rad)**2))

    # Initial conditions for perturbation \u03be(t) = theta(t) - theta_eq_rad
    # \u03be(0) = theta0 - theta_eq_rad
    # \u03be_dot(0) = dtheta/dt (0)
    # dtheta/dt = p_theta / (4a^2 (m1 + 2m2 sin^2(theta)))

    denominator_theta_dot0 = (4 * a**2 * (m1 + 2 * m2 * np.sin(theta0)**2))
    theta_dot0 = p_theta0 / denominator_theta_dot0 if denominator_theta_dot0 != 0 else 0.0

    A_initial_perturbation = theta0 - theta_eq_rad

    # For omega_0_user
    B_omega_0_user = theta_dot0 / omega_0_user if omega_0_user != 0 else 0.0
    xi_0_user_t = A_initial_perturbation * np.cos(omega_0_user * t_eval_walt) + B_omega_0_user * np.sin(omega_0_user * t_eval_walt)
    theta_analytical_0_user = theta_eq_rad + xi_0_user_t

    # For omega_Routh_user
    B_omega_Routh_user = theta_dot0 / omega_Routh_user if omega_Routh_user != 0 else 0.0
    xi_Routh_user_t = A_initial_perturbation * np.cos(omega_Routh_user * t_eval_walt) + B_omega_Routh_user * np.sin(omega_Routh_user * t_eval_walt)
    theta_analytical_Routh_user = theta_eq_rad + xi_Routh_user_t

    # Plotting the time evolution of theta with analytical solutions
    plt.figure(figsize=(12, 7))
    plt.plot(t_eval_walt, np.rad2deg(theta_sol_walt), label=r'Numerical $\theta(t)$ (Hamiltonian)', color='blue', linewidth=2)
    plt.plot(t_eval_walt, np.rad2deg(theta_analytical_0_user), label=r'Analytical $\theta(t)$ ($\omega_0$ from user)', color='red', linestyle='--', linewidth=1.5)
    plt.plot(t_eval_walt, np.rad2deg(theta_analytical_Routh_user), label=r'Analytical $\theta(t)$ ($\omega_{\text{Routh}}$ from user)', color='green', linestyle=':', linewidth=1.5)
    plt.axhline(np.rad2deg(theta_eq_rad), color='gray', linestyle='-.', label=r'Equilibrium $\theta_{\text{eq}}$', alpha=0.7)
    plt.title('Walt Regulator: Arm Angle ($\theta$) vs. Time (Numerical vs. Linearized Analytical)')
    plt.xlabel('Time (s)')
    plt.xlim(0,10)
    plt.ylabel('Arm Angle $\theta$ (degrees)')
    plt.grid(True)
    plt.legend()
    plt.show()

    print("Walt Regulator simulation and time evolution plot generated with linearized analytical comparisons.")

# Create an interactive widget for theta0_deg
widgets.interact(
    interactive_walt_regulator_theta0,
    theta0_deg=widgets.FloatSlider(min=0.1, max=90.0, step=0.1, value=45.0, description='Theta0 (deg):')
);

In [ ]:
plt.figure(figsize=(12, 7))
plt.plot(t_eval_walt, np.rad2deg(xi_0_user_t), label=r'Perturbation $\xi(t)$ ($\omega_0$ from user)', color='red', linestyle='--', linewidth=1.5)
plt.plot(t_eval_walt, np.rad2deg(xi_Routh_user_t), label=r'Perturbation $\xi(t)$ ($\omega_{\text{Routh}}$ from user)', color='green', linestyle=':', linewidth=1.5)
plt.axhline(0, color='gray', linestyle='-.', label=r'Equilibrium ($\xi=0$)', alpha=0.7)
plt.title('Walt Regulator: Perturbation from Equilibrium (Numerical vs. Linearized Analytical)')
plt.xlabel('Time (s)')
plt.xlim(0, 10)
plt.ylabel(r'Perturbation $\xi$ (degrees)')
plt.grid(True)
plt.legend()
plt.show()

print(r"Perturbation \xi(t) plot generated comparing both analytical models.")

### Explanation of \(\xi(t)\) (Perturbation from Equilibrium) for Walt Regulator

The \(\xi(t)\) represents the deviation of the arm angle \(\theta(t)\) from its equilibrium position \(\theta_{eq}\). It quantifies the small oscillations around a stable operating point, which is crucial for analyzing the regulator's stability and response to disturbances.

$$\xi(t) = \theta(t) - \theta_{eq}$$

When the system is linearized around an equilibrium point, the equation of motion for \(\xi(t)\) becomes that of a simple harmonic oscillator: \(\ddot{\xi} + \omega^2 \xi = 0\), where \(\omega\) is the oscillation frequency of the small perturbations.

This table summarizes the two analytical expressions for \(\omega\) provided by the user and their significance in the context of the Walt Regulator:

| Term               | Formula                                                                      | Description                                                                                                                                              |
| :----------------- | :--------------------------------------------------------------------------- | :------------------------------------------------------------------------------------------------------------------------------------------------------- |
| \(\omega_0\) (User)      | \(\Omega \sin(\theta_{eq}) \sqrt{\frac{m_1}{m_1 + 2m_2 \sin^2(\theta_{eq})}}\) | This frequency represents the oscillation of the perturbation \(\xi(t)\) when the system is linearized around \(\theta_{eq}\). It describes how quickly the arm angle returns to equilibrium after a small displacement. |
| \(\omega_{\text{Routh}}\) (User) | \(\Omega \sin(\theta_{eq}) \sqrt{\frac{4m_1 \cos^2(\theta_{eq})}{m_1 + 2m_2 \sin^2(\theta_{eq})}}\) | This frequency is also derived from a linearized analysis, potentially through a Routhian approach. It provides an alternative perspective on the oscillation frequency of the arms around equilibrium, often leading to slightly different values depending on the linearization assumptions.          |

### 4.8 Equilibrium Angle $\theta$ vs. Shaft Angular Velocity $\Omega$

This plot illustrates the stable (and unstable) equilibrium positions of the Walt Regulator's arms as a function of the main shaft's angular velocity, $\Omega$. Equilibrium occurs when $\dot{\theta}=0$ and $\dot{p}_{\theta}=0$. From Hamilton's equations, this means $p_{\theta}=0$ and the condition:

$\sin(\theta) \left[ (m_1+m_2)g - m_1 a \Omega^2 \cos(\theta) \right] = 0$

This equation yields two types of equilibrium points:
1.  **$\theta = 0$ or $\theta = \pi$ (downward and upward vertical positions):** These are always equilibrium positions, though their stability depends on $\Omega$.
2.  **$\cos(\theta) = \frac{(m_1+m_2)g}{m_1 a \Omega^2}$:** These are the non-trivial equilibrium angles where the arms are spread outwards. These solutions only exist when $$\left| \frac{(m_1+m_2)g}{m_1 a \Omega^2} \right| \le 1$$. We are primarily interested in $\theta \in [0, \pi/2]$ for the physically relevant range of operation.

The plot will show how the arm angle $\theta$ changes from $0$ (closed) to higher values as $\Omega$ increases.

In [ ]:
# --- Plotting Equilibrium Theta vs. Omega ---
# Parameters are already defined in cell GbOiuGLb2LSN

Omega_values = np.linspace(0.1, 10.0, 500) # Range of Omega values
equilibrium_theta_values = []

for current_Omega in Omega_values:
    # Check for trivial equilibria (theta = 0 and theta = pi)
    # We'll plot these as separate lines if they are stable/relevant.

    # Non-trivial equilibrium: cos(theta) = ((m1+m2)g) / (m1 a Omega^2)
    cosine_val = ((m1 + m2) * g) / (m1 * a * current_Omega**2)

    if np.abs(cosine_val) <= 1:
        theta_eq = np.arccos(cosine_val) # This gives theta in [0, pi]
        # Consider both theta and pi-theta if needed, but for regulators, [0, pi/2] is typical.
        equilibrium_theta_values.append(np.rad2deg(theta_eq))
    else:
        equilibrium_theta_values.append(np.nan) # No real solution for theta at this Omega

plt.figure(figsize=(10, 6))
plt.plot(Omega_values, equilibrium_theta_values, label=r'Non-trivial $\theta_{eq}$', color='purple')

# Add the trivial equilibrium at theta=0 (downward position)
plt.axhline(y=0, color='gray', linestyle='--', label=r'Trivial $\theta_{eq} = 0$')

# Calculate the critical Omega for the existence of non-trivial equilibrium
Omega_crit_sq = ((m1 + m2) * g) / (m1 * a)
Omega_crit = np.sqrt(Omega_crit_sq)

# Add a vertical line at the critical Omega
plt.axvline(x=Omega_crit, color='blue', linestyle='-.', label=r'Critical $\Omega$ (Start of non-trivial $\theta_{eq}$)', alpha=0.7)

plt.title(r'Walt Regulator: Equilibrium Arm Angle $\theta$ vs. Shaft Angular Velocity $\Omega$')
plt.xlabel(r'Shaft Angular Velocity $\Omega$ (rad/s)')
plt.ylabel(r'Equilibrium Arm Angle $\theta$ (degrees)')
plt.grid(True)
plt.legend()
plt.ylim(-5, 95) # Focus on the relevant range for theta
plt.show()

print("Equilibrium plot generated.")

### 4.9 Comparison of Effective Potentials

Here we compare two different effective potential functions for the Walt Regulator: the `effective_potential_L` derived from the Lagrangian and the `routhian_effective_potential`. These potentials help us understand the stability of equilibrium points and the system's energy landscape.

*   **Lagrangian Effective Potential ($V_{eff}$ from L):** This potential includes terms related to both gravity and the centrifugal force due to the shaft's rotation. Its minima correspond to stable equilibrium points.
*   **Routhian Effective Potential ($V_{Routh}$):** The Routhian is a transformation of the Lagrangian that is useful when dealing with cyclic coordinates (like the azimuthal angle $\phi$ in this system, which is implicitly conserved due to symmetry). The Routhian potential effectively 'absorbs' the kinetic energy associated with the cyclic coordinate into the potential, making it a function only of the non-cyclic coordinates and the conserved momentum $p_\phi$.
    *   For this plot, `p_phi` (the conserved angular momentum associated with the rotation around the main shaft) is set to a constant value (e.g., `p_phi = 1.0`). Changing this value will alter the shape of the Routhian potential, highlighting how conserved quantities influence stability.

Both potentials are plotted against $\theta$ to show their shapes and how they relate to each other, revealing where the system prefers to settle.

In [ ]:
# --- Comparison of Effective Potentials --- Parameters are already defined in cell GbOiuGLb2LSN

theta_range = np.linspace(0.01, np.pi - 0.01, 500) # Avoid 0 and pi for Routhian denominator

# Calculate Lagrangian effective potential values
L_eff_potential_values = effective_potential_L(theta_range, m1, m2, a, Omega, g)

# Calculate Routhian effective potential values
p_phi_constant = 1.0 # Arbitrary constant value for p_phi for illustration
Routh_eff_potential_values = routhian_effective_potential(theta_range, p_phi_constant, m1, m2, a, g)

plt.figure(figsize=(10, 6))
plt.plot(np.rad2deg(theta_range), L_eff_potential_values, label='Lagrangian Effective Potential', color='darkgreen')
plt.plot(np.rad2deg(theta_range), Routh_eff_potential_values, label=f'Routhian Effective Potential (p_phi={p_phi_constant:.1f})', color='darkred', linestyle='--')

plt.title('Walt Regulator: Comparison of Effective Potentials')
plt.xlabel('Arm Angle $\theta$ (degrees)')
plt.ylabel('Effective Potential Energy')
plt.grid(True)
plt.legend()
plt.ylim(bottom=-100) # Adjust y-limit for better visualization if needed
plt.show()

print("Effective potential comparison plot generated.")

### 4.10 Animation of the Walt Regulator

This animation visualizes the dynamic behavior of the Walt Regulator. As the central shaft rotates with angular velocity $\Omega$, the flyballs (mass $m_2$) move outwards or inwards based on the equilibrium condition. The animation shows the movement of the arms (length $a$) and flyballs as the angle $\theta$ changes over time according to the Hamiltonian dynamics.

**Components in the Animation:**
*   **Central Axis:** The vertical line around which the system rotates.
*   **Arms:** Lines connecting the central axis to the flyballs, forming angle $\theta$ with the vertical.
*   **Flyballs:** Spheres representing the masses $m_2$ at the end of the arms.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from matplotlib.animation import FuncAnimation

# --- Parámetros PhD ---
m1, m2, a, g = 2.0, 1.5, 1.0, 9.81
Omega_ref = 7.0

# Equilibrio
cos_theta0 = ((m1 + m2) * g) / (m1 * a * Omega_ref**2)
theta0 = np.arccos(cos_theta0)
p_phi = (2 * m1 * a**2 * np.sin(theta0)**2) * Omega_ref

# --- Frecuencias Teóricas (Hessianas) ---
# Forced: ω = sqrt(V_eff'' / M)
M0 = 2 * a**2 * (m1 + 2 * m2 * np.sin(theta0)**2)
V_f_pp = 2 * m1 * a**2 * Omega_ref**2 * np.sin(theta0)**2
omega_f = np.sqrt(V_f_pp / M0)

# Routh: ω_r = ω_f * sqrt(1 + 3cos²θ)
omega_r = omega_f * np.sqrt(1 + 3 * np.cos(theta0)**2)

def dynamics(t, y, mode='forced'):
    th, p_th = y
    M = 2 * a**2 * (m1 + 2 * m2 * np.sin(th)**2)
    if mode == 'forced':
        dV = 2*m1*a**2*Omega_ref**2*np.sin(th)*np.cos(th) + 2*(m1+m2)*g*a*np.sin(th)
        dp = (p_th**2 * 8*a**2*m2*np.sin(th)*np.cos(th))/(2*M**2) + 2*m1*a**2*Omega_ref**2*np.sin(th)*np.cos(th) - 2*(m1+m2)*g*a*np.sin(th)
    else:
        I = 2*m1*a**2*np.sin(th)**2
        dV = 2*(m1+m2)*g*a*np.sin(th) - (p_phi**2 * 4*m1*a**2*np.sin(th)*np.cos(th))/(I**2)
        dp = (p_th**2 * 8*a**2*m2*np.sin(th)*np.cos(th))/(2*M**2) - dV
    return [p_th/M, dp]

t_eval = np.linspace(0, 12, 400)
y0 = [theta0 + 0.25, 0.0]
sol_f = solve_ivp(dynamics, (0, 12), y0, args=('forced',), t_eval=t_eval, rtol=1e-9)
sol_r = solve_ivp(dynamics, (0, 12), y0, args=('routh',), t_eval=t_eval, rtol=1e-9)

# --- Animación Multivista con Frecuencias ---
fig = plt.figure(figsize=(22, 12))
ax1 = fig.add_subplot(221, projection='3d') # Cámara Rotatoria
ax2 = fig.add_subplot(222, projection='3d') # Cámara Fija
ax3 = fig.add_subplot(223)                 # Potencial
ax4 = fig.add_subplot(224)                 # Evolución Theta (Frecuencias)

th_range = np.linspace(0.15, 1.4, 200)
V_f_plot = -m1*a**2*Omega_ref**2*np.sin(th_range)**2 - 2*(m1+m2)*g*a*np.cos(th_range)
V_r_plot = -2*(m1+m2)*g*a*np.cos(th_range) + p_phi**2 / (4*m1*a**2*np.sin(th_range)**2)

def update(i):
    ax1.clear(); ax2.clear(); ax3.clear(); ax4.clear()
    phi_f = Omega_ref * t_eval[i]

    # 1. Dibujo en ambas cámaras
    for ax, azim_mode in zip([ax1, ax2], [np.degrees(phi_f)-90, 30]):
        ax.view_init(elev=20, azim=azim_mode)
        for sol, col, lbl in zip([sol_f, sol_r], ['#1f77b4', '#d62728'], ['Forced', 'Routh']):
            th = sol.y[0, i]
            # Phi para Routh (conservación de p_phi)
            I_h = 2*m1*a**2*np.sin(sol.y[0, :i+1])**2
            phi = np.trapz(p_phi / I_h, t_eval[:i+1]) if (lbl=='Routh' and i>0) else phi_f

            x, y, z = a*np.sin(th)*np.cos(phi), a*np.sin(th)*np.sin(phi), -a*np.cos(th)
            z_r = -2*a*np.cos(th)
            ax.plot([0, x], [0, y], [0, z], color=col, lw=2)
            ax.plot([0, -x], [0, -y], [0, z], color=col, lw=2)
            ax.plot([x, 0], [y, 0], [z, z_r], color=col, lw=2)
            ax.plot([-x, 0], [-y, 0], [z, z_r], color=col, lw=2)
            ax.scatter([x, -x], [y, -y], [z, z], color='black', s=50)
            ax.scatter(0, 0, z_r, color='gray', s=100, marker='s')
        ax.set_zlim(-2.2, 0); ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2)

    # 2. Potenciales
    ax3.plot(th_range, V_f_plot, 'b--', alpha=0.4, label=f'V_f (ω={omega_f:.2f})')
    ax3.plot(th_range, V_r_plot, 'r-', label=f'V_r (ω={omega_r:.2f})')
    ax3.scatter(sol_f.y[0, i], V_f_plot[np.abs(th_range-sol_f.y[0,i]).argmin()], color='blue')
    ax3.scatter(sol_r.y[0, i], V_r_plot[np.abs(th_range-sol_r.y[0,i]).argmin()], color='red')
    ax3.legend(); ax3.set_title("Effective Potentials & Theoretical Frequencies")

    # 3. Comparativa de Oscilación (Frecuencias en acción)
    ax4.plot(t_eval[:i], np.degrees(sol_f.y[0, :i]), 'b', label='Forced Oscillation')
    ax4.plot(t_eval[:i], np.degrees(sol_r.y[0, :i]), 'r', label='Routh Oscillation')
    ax4.set_title("Dynamic Frequency Comparison"); ax4.legend(); ax4.grid(True)
    ax4.set_ylabel("Theta (deg)"); ax4.set_xlabel("Time (s)")

ani = FuncAnimation(fig, update, frames=len(t_eval), interval=35)
ani.save('walt_phd_final_comparison.mp4', writer='ffmpeg')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from matplotlib.animation import FuncAnimation

# --- Parámetros PhD ---
m1, m2, a, g = 2.0, 1.5, 1.0, 9.81
Omega_ref = 7.0

# Equilibrio
cos_theta0 = ((m1 + m2) * g) / (m1 * a * Omega_ref**2)
theta0 = np.arccos(cos_theta0)
p_phi = (2 * m1 * a**2 * np.sin(theta0)**2) * Omega_ref

# --- Función para Frecuencia Instantánea ---
def get_omega_inst(th, mode='forced'):
    M = 2 * a**2 * (m1 + 2 * m2 * np.sin(th)**2)
    if mode == 'forced':
        V_pp = 2 * m1 * a**2 * Omega_ref**2 * (np.cos(th)**2 - np.sin(th)**2) + 2*(m1+m2)*g*a*np.cos(th)
    else:
        I = 2*m1*a**2*np.sin(th)**2
        V_pp = 2*(m1+m2)*g*a*np.cos(th) + (p_phi**2 * (4*m1*a**2*(1 + 3*np.cos(th)**2))) / (I**2 * np.sin(th)**2)
    return np.sqrt(np.abs(V_pp) / M)

def dynamics(t, y, mode='forced'):
    th, p_th = y
    M = 2 * a**2 * (m1 + 2 * m2 * np.sin(th)**2)
    if mode == 'forced':
        dV = 2*m1*a**2*Omega_ref**2*np.sin(th)*np.cos(th) + 2*(m1+m2)*g*a*np.sin(th)
        dp = (p_th**2 * 8*a**2*m2*np.sin(th)*np.cos(th))/(2*M**2) + 2*m1*a**2*Omega_ref**2*np.sin(th)*np.cos(th) - 2*(m1+m2)*g*a*np.sin(th)
    else:
        I = 2*m1*a**2*np.sin(th)**2
        dV = 2*(m1+m2)*g*a*np.sin(th) - (p_phi**2 * 4*m1*a**2*np.sin(th)*np.cos(th))/(I**2)
        dp = (p_th**2 * 8*a**2*m2*np.sin(th)*np.cos(th))/(2*M**2) - dV
    return [p_th/M, dp]

t_eval = np.linspace(0, 12, 400)
y0 = [theta0 + 0.25, 0.0]
sol_f = solve_ivp(dynamics, (0, 12), y0, args=('forced',), t_eval=t_eval, rtol=1e-9)
sol_r = solve_ivp(dynamics, (0, 12), y0, args=('routh',), t_eval=t_eval, rtol=1e-9)

# --- Animación ---
fig = plt.figure(figsize=(22, 12))
ax1 = fig.add_subplot(221, projection='3d')
ax2 = fig.add_subplot(222, projection='3d')
ax3 = fig.add_subplot(223)
ax4 = fig.add_subplot(224)

th_range = np.linspace(0.15, 1.4, 200)
V_f_plot = -m1*a**2*Omega_ref**2*np.sin(th_range)**2 - 2*(m1+m2)*g*a*np.cos(th_range)
V_r_plot = -2*(m1+m2)*g*a*np.cos(th_range) + p_phi**2 / (4*m1*a**2*np.sin(th_range)**2)

def update(i):
    ax1.clear(); ax2.clear(); ax3.clear(); ax4.clear()
    phi_f = Omega_ref * t_eval[i]

    for ax, azim_mode in zip([ax1, ax2], [np.degrees(phi_f)-90, 30]):
        ax.view_init(elev=20, azim=azim_mode)
        for sol, col, lbl in zip([sol_f, sol_r], ['#1f77b4', '#d62728'], ['Forced', 'Routh']):
            th = sol.y[0, i]

            # Dinámica de rotación diferenciada
            if lbl == 'Routh':
                omega_phi = p_phi / (2*m1*a**2*np.sin(th)**2)
                phi = np.trapz(p_phi/(2*m1*a**2*np.sin(sol.y[0,:i+1])**2), t_eval[:i+1]) if i>0 else 0
            else:
                omega_phi = Omega_ref
                phi = phi_f

            x, y, z = a*np.sin(th)*np.cos(phi), a*np.sin(th)*np.sin(phi), -a*np.cos(th)
            z_r = -2*a*np.cos(th)

            # Dibujo Estructura
            ax.plot([0, x], [0, y], [0, z], color=col, lw=2, alpha=0.5)
            ax.plot([x, 0], [y, 0], [z, z_r], color=col, lw=2, alpha=0.5)
            ax.scatter([x], [y], [z], color='black', s=50)
            ax.scatter(0, 0, z_r, color='gray', s=80, marker='s')

            # --- VECTORES DE FUERZA (Cámara Síncrona) ---
            if ax == ax1:
                scale = 0.03
                # Fuerza Centrífuga
                f_c = m1 * (omega_phi**2) * (a * np.sin(th))
                ax.quiver(x, y, z, f_c*scale*np.cos(phi), f_c*scale*np.sin(phi), 0, color='blue', arrow_length_ratio=0.3)
                # Fuerza Gravedad
                f_g = m1 * g
                ax.quiver(x, y, z, 0, 0, -f_g*scale, color='green', arrow_length_ratio=0.3)

                # Etiqueta de Frecuencia Instantánea
                w_inst = get_omega_inst(th, lbl.lower())
                ax.text(x, y, z + 0.2, f"ω={w_inst:.2f}", color=col, fontsize=10, fontweight='bold')

        ax.set_zlim(-2.2, 0); ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2)
        ax.set_title("Synchronous View + Force Vectors" if ax==ax1 else "Fixed Inertial Frame")

    # Potenciales y Gráfica Theta (se mantienen igual que tu esquema base)
    ax3.plot(th_range, V_f_plot, 'b--', alpha=0.4); ax3.plot(th_range, V_r_plot, 'r-', alpha=0.6)
    ax3.scatter(sol_f.y[0, i], V_f_plot[np.abs(th_range-sol_f.y[0,i]).argmin()], color='blue')
    ax3.scatter(sol_r.y[0, i], V_r_plot[np.abs(th_range-sol_r.y[0,i]).argmin()], color='red')
    ax3.set_title("Effective Potentials")

    ax4.plot(t_eval[:i], np.degrees(sol_f.y[0, :i]), 'b', label='Forced'); ax4.plot(t_eval[:i], np.degrees(sol_r.y[0, :i]), 'r', label='Routh')
    ax4.legend(loc='upper right'); ax4.grid(True); ax4.set_ylabel("Theta (deg)")

ani = FuncAnimation(fig, update, frames=len(t_eval), interval=35)
ani.save('walt_vectors_comparison.mp4', writer='ffmpeg')
